# L1+L6: Perfect Metal + Surface Kinetics — Analysis Notebook

**Physics:**
- Surface dissociation/recombination at the gas–metal interface (L6)
- Bulk lattice diffusion of atomic H through a perfect metal (L1)
- No oxide layer

**Two competing resistances:**
- $R_{surface} \sim 1 / (k_{diss} \cdot P_{up} \cdot (1-\theta)^2)$
- $R_{metal} = L_m / (D_m \cdot K_{s,m})$

**Key physics question:** At what pressure and temperature does the rate-limiting step shift from surface dissociation to bulk diffusion?


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import os
import sys

# Add parent directory to path
parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
# This notebook must run with its OWN folder as the working directory: that is how
# calculations.config.model_config finds this folder's study_config.py, and how
# RESULTS_DIR below resolves. Fails loudly rather than silently using another study.
from calculations.config import assert_study_is_local
print(assert_study_is_local())

# Figures from this notebook go in one folder beside it.
from calculations.figures import results_dir, save_figure
FIGS = results_dir("Surface_proposal")

study: incoloy802_cr2o3  (incoloy802_cr2o3/study_config.py, via folder)


study: incoloy802_cr2o3  (incoloy802_cr2o3/study_config.py, via folder)


In [2]:
# =============================================================================
# CELL 1 — SETUP
# Run once. All imports and helper functions.
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import brentq
import pandas as pd
from itertools import groupby
from operator import itemgetter
from IPython.display import display

from calculations.surface_kinetics import solve_steady_state_flux_L1L6
from calculations.config.model_config import (
    R, F,
    METALS, CONDITIONS,
    PLOT_STYLE as STYLE,
    CURVE_STYLES,
    apply_style,
    build_simulation_config,
)

# ── Simulation config ────────────────────────────────────────────────────────
SIM       = build_simulation_config(
)

METAL_KEY = SIM['metal_name']
METAL     = METALS[METAL_KEY]
SK        = METAL['surface_kinetics']     # surface kinetics sub-dict

# ── Pressure sweep (matches widget range) ────────────────────────────────────
P_RANGE_ARR = np.logspace(
    np.log10(CONDITIONS['P_range'][0]),
    np.log10(CONDITIONS['P_range'][1]),
    CONDITIONS['n_points']
)   #np.logspace(-3, 12, 100)
P_DOWN      = SIM['P_downstream']         # Pa
T_RANGE     = CONDITIONS['T_range']       # K
N_T         = CONDITIONS['n_points']

# ── Arrhenius helper functions ───────────────────────────────────────────────
def get_k_diss_at_T(T_K):
    """k_diss at temperature T_K via Arrhenius from config."""
    return SK['k_diss_metal_ref'] * np.exp(
        (-SK['E_diss_metal'] / R) * (1/T_K - 1/SK['T_ref'])
    )

def get_K_eq_at_T(T_K):
    """K_eq at temperature T_K via van't Hoff from config."""
    return SK['K_eq_metal_ref'] * np.exp(
        (-SK['H_eq_metal'] / R) * (1/T_K - 1/SK['T_ref'])
    )

def get_D_m_at_T(T_K):
    """D_m at temperature T_K via Arrhenius from config."""
    return METAL['D_ref'] * np.exp(
        (-METAL['E_D'] / R) * (1/T_K - 1/METAL['T_ref'])
    )

def get_K_s_m_at_T(T_K):
    """K_s_m at temperature T_K via Arrhenius from config."""
    return METAL['K_s_ref'] * np.exp(
        (-METAL['H_s'] / R) * (1/T_K - 1/METAL['T_ref'])
    )

# ── Known reference activation energies (for validation) ────────────────────
E_REF_SURFACE = SK['E_diss_metal']              # J/mol — surface dissociation
E_REF_METAL   = METAL['E_D'] + METAL['H_s']    # J/mol — bulk permeation

print("=" * 60)
print("SETUP COMPLETE")
print("=" * 60)
print(f"  Metal:           {METAL_KEY}")
print(f"  E_ref surface:   {E_REF_SURFACE/1000:.3f} kJ/mol  (E_diss)")
print(f"  E_ref metal:     {E_REF_METAL/1000:.3f} kJ/mol  (E_D + H_s)")
print(f"  P range:         {P_RANGE_ARR[0]:.0e} – {P_RANGE_ARR[-1]:.0e} Pa")
print(f"  T range:         {T_RANGE[0]} – {T_RANGE[1]} K")

SETUP COMPLETE
  Metal:           metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985
  E_ref surface:   81.560 kJ/mol  (E_diss)
  E_ref metal:     79.400 kJ/mol  (E_D + H_s)
  P range:         1e-07 – 1e+07 Pa
  T range:         573 – 1273 K


In [3]:
P_RANGE_ARR

array([1.00000000e-07, 2.28546386e-07, 5.22334507e-07, 1.19377664e-06,
       2.72833338e-06, 6.23550734e-06, 1.42510267e-05, 3.25702066e-05,
       7.44380301e-05, 1.70125428e-04, 3.88815518e-04, 8.88623816e-04,
       2.03091762e-03, 4.64158883e-03, 1.06081836e-02, 2.42446202e-02,
       5.54102033e-02, 1.26638017e-01, 2.89426612e-01, 6.61474064e-01,
       1.51177507e+00, 3.45510729e+00, 7.89652287e+00, 1.80472177e+01,
       4.12462638e+01, 9.42668455e+01, 2.15443469e+02, 4.92388263e+02,
       1.12533558e+03, 2.57191381e+03, 5.87801607e+03, 1.34339933e+04,
       3.07029063e+04, 7.01703829e+04, 1.60371874e+05, 3.66524124e+05,
       8.37677640e+05, 1.91448198e+06, 4.37547938e+06, 1.00000000e+07])

In [4]:
# =============================================================================
# CELL 2 — PARAMETERS
# All physics values at operating temperature, pulled from config.
# Change placeholder values in config and rerun from this cell downward.
# =============================================================================

# ── Operating conditions ─────────────────────────────────────────────────────
T_op  = SIM['T_operating']    # K
L_m   = SIM['L_metal']        # m
P_down = SIM['P_downstream']  # Pa

# ── Surface kinetics at T_op ─────────────────────────────────────────────────
k_diss = get_k_diss_at_T(T_op)
K_eq   = get_K_eq_at_T(T_op)
k_recomb = k_diss / K_eq      # derived — not stored separately

# ── Metal transport at T_op ──────────────────────────────────────────────────
D_m   = get_D_m_at_T(T_op)
K_s_m = get_K_s_m_at_T(T_op)

# ── Derived ──────────────────────────────────────────────────────────────────
beta = D_m * K_s_m / L_m      # metal permeance [mol/m²/s/Pa^0.5]

# ── Print verification ───────────────────────────────────────────────────────
print("=" * 60)
print(f"L1+L6 PARAMETERS AT T = {T_op-273.15:.3f}°C  ({T_op} K)")
print("=" * 60)

print(f"\n  Geometry")
print(f"    L_m      = {L_m:.3e}  m")
print(f"    P_down   = {P_down:.1e}  Pa")

print(f"\n  Surface kinetics  [config: surface_kinetics sub-dict]")
print(f"    k_diss   = {k_diss:.3e}  mol/m²/s/Pa")
print(f"    K_eq     = {K_eq:.3e}  Pa⁻¹")
print(f"    k_recomb = {k_recomb:.3e}  mol/m²/s  (= k_diss / K_eq)")

print(f"\n  Metal transport  [config: Arrhenius]")
print(f"    D_m      = {D_m:.3e}  m²/s")
print(f"    K_s_m    = {K_s_m:.3e}  mol/m³/Pa^0.5")

print(f"\n  Derived")
print(f"    β        = {beta:.3e}  mol/m²/s/Pa^0.5  (= D_m × K_s_m / L_m)")

print(f"\n  Reference activation energies  [config]")
print(f"    E_diss   = {E_REF_SURFACE/1000:.3f} kJ/mol   (surface dissociation)")
print(f"    E_D+Hs   = {E_REF_METAL/1000:.3f} kJ/mol   (bulk permeation)")

# ── Placeholder warning ──────────────────────────────────────────────────────
if SK['k_diss_metal_ref'] <= 1e-12:
    print(f"\n  ⚠  WARNING: k_diss_metal_ref = {SK['k_diss_metal_ref']:.0e} looks like")
    print(f"     a placeholder. Update config surface_kinetics sub-dict")
    print(f"     with literature values before interpreting results.")

print("\n" + "=" * 60)

L1+L6 PARAMETERS AT T = 599.850°C  (873 K)

  Geometry
    L_m      = 1.000e-03  m
    P_down   = 0.0e+00  Pa

  Surface kinetics  [config: surface_kinetics sub-dict]
    k_diss   = 4.611e-07  mol/m²/s/Pa
    K_eq     = 8.212e-04  Pa⁻¹
    k_recomb = 5.615e-04  mol/m²/s  (= k_diss / K_eq)

  Metal transport  [config: Arrhenius]
    D_m      = 2.487e-10  m²/s
    K_s_m    = 3.834e-02  mol/m³/Pa^0.5

  Derived
    β        = 9.534e-09  mol/m²/s/Pa^0.5  (= D_m × K_s_m / L_m)

  Reference activation energies  [config]
    E_diss   = 81.560 kJ/mol   (surface dissociation)
    E_D+Hs   = 79.400 kJ/mol   (bulk permeation)



In [5]:
# =============================================================================
# CELL 3 — COMPUTE
# Single-pass loop over full pressure range.
# Builds all arrays, regime summary, and Arrhenius pressure selection.
# =============================================================================

# ── Single-pass loop ─────────────────────────────────────────────────────────
rows             = []
J_arr            = []
theta_arr        = []
P_int_arr        = []
frac_surface_arr = []
frac_metal_arr   = []
R_surface_arr    = []
R_metal_arr      = []
R_total_arr      = []
rate_lim_arr     = []

for P_up in P_RANGE_ARR:
    try:
        r = solve_steady_state_flux_L1L6(
            P_up, P_down, L_m,
            k_diss, K_eq, D_m, K_s_m
        )
        J_arr.append(r['J_ss'])
        theta_arr.append(r['theta'])
        P_int_arr.append(r['P_int'])
        frac_surface_arr.append(r['resistances']['fraction_surface'])
        frac_metal_arr.append(r['resistances']['fraction_metal'])
        R_surface_arr.append(r['resistances']['R_surface'])
        R_metal_arr.append(r['resistances']['R_metal'])
        R_total_arr.append(r['resistances']['R_total'])
        rate_lim_arr.append(r['rate_limiting'])

        rows.append({
            'P_up (Pa)':            P_up,
            'P_int (Pa)':           r['P_int'],
            'J_ss (mol/m²/s)':      r['J_ss'],
            'θ_surface':            r['theta'],
            'β_metal':              r['beta'],
            'R_surface':            r['resistances']['R_surface'],
            'R_metal':              r['resistances']['R_metal'],
            'R_total':              r['resistances']['R_total'],
            'fraction_surface (%)': r['resistances']['fraction_surface'] * 100,
            'fraction_metal (%)':   r['resistances']['fraction_metal']   * 100,
            'Rate-Limiting':        r['rate_limiting'].upper(),
        })

    except Exception as e:
        J_arr.append(np.nan);            theta_arr.append(np.nan)
        P_int_arr.append(np.nan);        frac_surface_arr.append(np.nan)
        frac_metal_arr.append(np.nan);   R_surface_arr.append(np.nan)
        R_metal_arr.append(np.nan);      R_total_arr.append(np.nan)
        rate_lim_arr.append('error')
        rows.append({'P_up (Pa)': P_up, 'Rate-Limiting': 'ERROR', 'Error': str(e)})

# ── Convert to arrays ────────────────────────────────────────────────────────
J_arr            = np.array(J_arr)
theta_arr        = np.array(theta_arr)
P_int_arr        = np.array(P_int_arr)
frac_surface_arr = np.array(frac_surface_arr)
frac_metal_arr   = np.array(frac_metal_arr)
R_surface_arr    = np.array(R_surface_arr)
R_metal_arr      = np.array(R_metal_arr)
R_total_arr      = np.array(R_total_arr)
rate_lim_arr     = np.array(rate_lim_arr)
valid            = ~np.isnan(J_arr)

# ── Rate-limiting classification array (for plot shading) ────────────────────
rl_arr = np.where(frac_surface_arr > 0.5, 'surface',
         np.where(frac_metal_arr   > 0.5, 'metal', 'mixed'))

# ── Regime summary ───────────────────────────────────────────────────────────
# Count points in each regime
n_surface = np.sum((frac_surface_arr > 0.5)  & valid)
n_metal   = np.sum((frac_metal_arr   > 0.5)  & valid)
n_mixed   = np.sum(
    (frac_surface_arr <= 0.5) & (frac_metal_arr <= 0.5) & valid
)

# Pressure ranges for each dominant regime (>90% — clean, not transitional)
surf_dom_mask  = (frac_surface_arr > 0.90) & valid
metal_dom_mask = (frac_metal_arr   > 0.90) & valid

P_surf_dom  = P_RANGE_ARR[surf_dom_mask]
P_metal_dom = P_RANGE_ARR[metal_dom_mask]

# ── Arrhenius pressure selection ─────────────────────────────────────────────
# Use median of each dominant region — no dependence on crossover
P_for_surface_Ea = np.median(P_surf_dom)  if len(P_surf_dom)  > 3 else None
P_for_metal_Ea   = np.median(P_metal_dom) if len(P_metal_dom) > 3 else None

# ── Crossover pressure (purely diagnostic, not used for Ea selection) ────────
P_crossover = None
valid_cross = valid & ~np.isnan(frac_surface_arr)
if np.sum(valid_cross) > 2:
    fs_v   = frac_surface_arr[valid_cross]
    P_v    = P_RANGE_ARR[valid_cross]
    sc     = np.where(np.diff(np.sign(fs_v - 0.5)))[0]
    if len(sc) > 0:
        idx = sc[0]
        # Direction check: surface fraction must drop (not rise) at crossover
        if idx + 1 < len(fs_v) and fs_v[idx] > 0.5 and fs_v[idx + 1] < 0.5:
            P_crossover = P_v[idx]

# ── Print regime summary ─────────────────────────────────────────────────────
print("=" * 60)
print(f"L1+L6 REGIME SUMMARY  (T = {T_op-273.15:.3f}°C, {np.sum(valid)} valid points)")
print("=" * 60)

print(f"\n  Regime breakdown (fraction > 50%)")
print(f"    Surface-limited:  {n_surface:3d} points")
print(f"    Metal-limited:    {n_metal:3d} points")
print(f"    Mixed:            {n_mixed:3d} points")

print(f"\n  Dominant regions (fraction > 90%) — used for Arrhenius")
if len(P_surf_dom) > 3:
    print(f"    Surface-dominant: {len(P_surf_dom):3d} points  "
          f"P = {P_surf_dom.min():.1e} – {P_surf_dom.max():.1e} Pa")
    print(f"    → P_for_surface_Ea = {P_for_surface_Ea:.2e} Pa")
else:
    print(f"    Surface-dominant: {len(P_surf_dom):3d} points  "
          f"(insufficient — surface Ea not extractable)")

if len(P_metal_dom) > 3:
    print(f"    Metal-dominant:   {len(P_metal_dom):3d} points  "
          f"P = {P_metal_dom.min():.1e} – {P_metal_dom.max():.1e} Pa")
    print(f"    → P_for_metal_Ea  = {P_for_metal_Ea:.2e} Pa")
else:
    print(f"    Metal-dominant:   {len(P_metal_dom):3d} points  "
          f"(insufficient — metal Ea not extractable)")

print(f"\n  Crossover pressure (diagnostic only)")
if P_crossover is not None:
    print(f"    P_crossover = {P_crossover:.2e} Pa")
else:
    print(f"    No crossover detected in physical pressure range")
    dominant = 'surface-limited' if n_surface > n_metal else 'metal-limited'
    print(f"    System is predominantly {dominant} across full range")

print(f"\n  θ range:  {np.nanmin(theta_arr):.4f} → {np.nanmax(theta_arr):.4f}")
print(f"  J range:  {np.nanmin(J_arr):.2e} → {np.nanmax(J_arr):.2e} mol/m²/s")
print("=" * 60)

L1+L6 REGIME SUMMARY  (T = 599.850°C, 40 valid points)

  Regime breakdown (fraction > 50%)
    Surface-limited:   15 points
    Metal-limited:     25 points
    Mixed:              0 points

  Dominant regions (fraction > 90%) — used for Arrhenius
    Surface-dominant:  13 points  P = 1.0e-07 – 2.0e-03 Pa
    → P_for_surface_Ea = 1.43e-05 Pa
    Metal-dominant:    22 points  P = 2.9e-01 – 1.0e+07 Pa
    → P_for_metal_Ea  = 1.85e+03 Pa

  Crossover pressure (diagnostic only)
    P_crossover = 1.06e-02 Pa

  θ range:  0.0000 → 0.9888
  J range:  4.61e-14 → 2.94e-05 mol/m²/s


In [6]:
# =============================================================================
# CELL 4 — FIGURE 1: Core Validation (2×2)
# (A) Flux vs Pressure
# (B) Surface Coverage θ vs Pressure
# (C) Resistance Fractions vs Pressure
# (D) Limit Check: Sieverts Recovery
# =============================================================================


"""
Figure 1 validates the L1+L6 model from four angles at the operating temperature.

Panel (A) — Flux vs Pressure
    Question: Does the model produce the correct pressure-dependent behaviour?
    The hero plot. Shows J_ss vs P_up on a log-log scale. The curve should
    transition from slope=1 (surface-limited, low P) to slope=0.5
    (metal-limited, high P). Style follows the widget — no fill_between.
    Instead the curve itself is recolored by regime using thick colored
    segments, with a slope annotation box sitting directly on each segment.
    The black model curve sits underneath, the colored overlay sits on top.
    Reference lines for slope=1 and slope=0.5 are drawn as dashed lines
    anchored to the start and end of the curve respectively.
    Net slope annotation box in the bottom right corner.

Panel (B) — Surface Coverage θ vs Pressure
    Question: How does surface occupancy evolve with pressure, and does it
    match the Langmuir equilibrium isotherm?
    Plots the steady-state θ from the compute loop alongside the analytical
    Langmuir isotherm θ_eq = sqrt(K_eq * P) / (1 + sqrt(K_eq * P)) as a
    gray dash-dot line. The gap between θ_ss and θ_eq is physically
    meaningful — when they coincide the surface is in local equilibrium
    (metal-limited). When θ_ss < θ_eq the surface is being depleted by fast
    diffusion into the metal (surface-limited). Crossover pressure marked
    if it exists.

Panel (C) — Resistance Fractions vs Pressure
    Question: Which step controls the rate, and how does the balance shift
    with pressure?
    Plots R_surface/R_total and R_metal/R_total as percentages vs pressure
    on a semilog scale. The 50% threshold line marks the crossover. If no
    crossover exists in the physical range a text annotation states which
    regime dominates throughout. Crossover pressure annotated if present.

Panel (D) — Limit Check: Sieverts Recovery
    Question: As k_diss → ∞, does L1+L6 recover the pure Sieverts result?
    Runs the solver at k_diss = 1e-3 (fast kinetics limit) and compares
    J_L1+L6 against the analytical Sieverts flux J = beta*(sqrt(P_up) -
    sqrt(P_down)). A parity plot shows J_L1+L6 vs J_Sieverts — perfect
    agreement is a diagonal line. Max and mean relative errors are shown
    in an annotation box. Green box if max error < 1%, yellow otherwise.
    This validates the model correctly recovers L1 in the fast-kinetics limit.
"""

props = dict(boxstyle='round', facecolor='wheat', alpha=0.9)

fig, axes = plt.subplots(2, 2, figsize=STYLE['figsize'])
fig.suptitle(
    f'L1+L6: Perfect Metal + Surface Kinetics — Core Validation\n'
    f'{METAL_KEY}  |  T = {T_op-273.15:.3f}°C  |  L = {L_m*1e3:.3f} mm',
    fontsize=STYLE['fontsize_suptitle'], fontweight='bold', y=0.98
)

# ─────────────────────────────────────────────────────────────────────────────
# (A) FLUX vs PRESSURE — widget style: colored segments + slope annotations
# ─────────────────────────────────────────────────────────────────────────────
ax1 = axes[0, 0]

# Base black curve
ax1.loglog(
    P_RANGE_ARR[valid], J_arr[valid],
    color='black', lw=STYLE['linewidth'],
    label='L1+L6 Model'
)

# Slope reference lines
if np.any(valid):
    P_ref1  = P_RANGE_ARR[valid][0]
    J_ref1  = J_arr[valid][0]
    ax1.loglog(
        P_RANGE_ARR[valid],
        J_ref1 * (P_RANGE_ARR[valid] / P_ref1) ** 1.0,
        color=CURVE_STYLES['slope_1']['color'],
        ls=CURVE_STYLES['slope_1']['ls'],
        lw=CURVE_STYLES['slope_1']['lw'],
        alpha=CURVE_STYLES['slope_1']['alpha'],
        label='Slope = 1 (surface)'
    )
    P_ref05 = P_RANGE_ARR[valid][-1]
    J_ref05 = J_arr[valid][-1]
    ax1.loglog(
        P_RANGE_ARR[valid],
        J_ref05 * (P_RANGE_ARR[valid] / P_ref05) ** 0.5,
        color=CURVE_STYLES['slope_05']['color'],
        ls=CURVE_STYLES['slope_05']['ls'],
        lw=CURVE_STYLES['slope_05']['lw'],
        alpha=CURVE_STYLES['slope_05']['alpha'],
        label='Slope = 0.5 (metal)'
    )

# Colored regime segments + slope annotation boxes (widget style)
region_styles = [
    ('surface', CURVE_STYLES['surface_region']['color'], 'Surface-limited'),
    ('metal',   CURVE_STYLES['metal_region']['color'],   'Metal-limited'),
    ('mixed',   CURVE_STYLES['mixed_region']['color'],   'Mixed'),
]

for regime, color, label in region_styles:
    mask = (rl_arr == regime) & valid
    if not np.any(mask):
        continue
    idxs = np.where(mask)[0]
    for _, g in groupby(enumerate(idxs), lambda x: x[0] - x[1]):
        group = list(map(itemgetter(1), g))
        if len(group) < 3:
            continue
        P_seg = P_RANGE_ARR[group]
        J_seg = J_arr[group]
        ax1.loglog(P_seg, J_seg, color=color, lw=4, alpha=0.7)
        slope_seg, *_ = stats.linregress(
            np.log10(P_seg), np.log10(np.abs(J_seg))
        )
        mid = len(group) // 2
        ax1.text(
            P_seg[mid], J_seg[mid],
            f'{label}\nSlope={slope_seg:.2f}',
            color=color,
            fontsize=STYLE['fontsize_annotation'],
            fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
        )

# Net slope annotation
log_P_v = np.log10(P_RANGE_ARR[valid])
log_J_v = np.log10(np.abs(J_arr[valid]))
slope_net, *_ = stats.linregress(log_P_v, log_J_v)
ax1.text(
    0.98, 0.02, f'Net slope = {slope_net:.2f}',
    transform=ax1.transAxes, ha='right', va='bottom',
    fontsize=STYLE['fontsize_annotation'], fontweight='bold',
    bbox=dict(boxstyle='square', fc='wheat', ec='gray', alpha=1)
)
ax1.text(
    0.05, 0.95,
    r'$J = \beta\left(g(\theta) - \sqrt{P_{down}}\right)$',
    transform=ax1.transAxes, fontsize=10, va='top',
    bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8)
)

ax1.set_xlabel('Upstream Pressure $P_{up}$ (Pa)', fontsize=STYLE['fontsize_axis'])
ax1.set_ylabel('Flux $J_{ss}$ (mol/m²/s)',         fontsize=STYLE['fontsize_axis'])
ax1.set_title('(A) Flux vs Pressure',               fontsize=STYLE['fontsize_title'])
ax1.legend(fontsize=STYLE['fontsize_legend']-1, loc='upper left')
ax1.grid(True, which='both', alpha=STYLE['grid_alpha'])
ax1.tick_params(labelsize=STYLE['fontsize_tick'])

# ─────────────────────────────────────────────────────────────────────────────
# (B) SURFACE COVERAGE θ vs PRESSURE
# ─────────────────────────────────────────────────────────────────────────────
ax2 = axes[0, 1]

# Steady-state θ from compute loop
ax2.semilogx(
    P_RANGE_ARR[valid], theta_arr[valid],
    color=CURVE_STYLES['L1_L6']['color'],
    ls=CURVE_STYLES['L1_L6']['ls'],
    lw=STYLE['linewidth'],
    marker=CURVE_STYLES['L1_L6']['marker'],
    ms=CURVE_STYLES['L1_L6']['ms'],
    markevery=8,
    label='θ steady-state'
)

# Equilibrium Langmuir isotherm overlay
theta_eq = (np.sqrt(K_eq * P_RANGE_ARR) /
            (1 + np.sqrt(K_eq * P_RANGE_ARR)))
ax2.semilogx(
    P_RANGE_ARR, theta_eq,
    color='gray', ls='-.', lw=1.5, alpha=0.7,
    label=r'θ$_{eq}$ (Langmuir isotherm)'
)

# Reference lines
ax2.axhline(
    1.0, color='red', ls='--', lw=1.5, alpha=0.5,
    label='θ = 1 (saturation)'
)
ax2.axhline(
    0.5, color='orange', ls='--', lw=1.5, alpha=0.5,
    label='θ = 0.5'
)

# Crossover line
if P_crossover is not None:
    ax2.axvline(
        P_crossover, color='gray', ls=':', lw=1.5, alpha=0.7,
        label=f'Crossover  P={P_crossover:.1e} Pa'
    )

ax2.text(
    0.05, 0.95,
    r'$\theta_{eq} = \frac{\sqrt{K_{eq} P}}{1 + \sqrt{K_{eq} P}}$',
    transform=ax2.transAxes, fontsize=10, va='top',
    bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8)
)
ax2.text(
    0.95, 0.30,
    'θ_ss < θ_eq → surface depleted\n   by fast metal diffusion\n'
    'θ_ss ≈ θ_eq → local equilibrium\n   (metal-limited)',
    transform=ax2.transAxes, fontsize=STYLE['fontsize_annotation']-1,
    va='top', ha='right',
    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8)
)

ax2.set_xlabel('Upstream Pressure $P_{up}$ (Pa)',  fontsize=STYLE['fontsize_axis'])
ax2.set_ylabel('Surface Coverage $\\theta$',        fontsize=STYLE['fontsize_axis'])
ax2.set_title('(B) Surface Coverage vs Pressure',   fontsize=STYLE['fontsize_title'])
ax2.set_ylim(-0.05, 1.1)
ax2.legend(fontsize=STYLE['fontsize_legend']-1, loc='lower right')
ax2.grid(True, which='both', alpha=STYLE['grid_alpha'])
ax2.tick_params(labelsize=STYLE['fontsize_tick'])

# ─────────────────────────────────────────────────────────────────────────────
# (C) RESISTANCE FRACTIONS vs PRESSURE
# ─────────────────────────────────────────────────────────────────────────────
ax3 = axes[1, 0]

ax3.semilogx(
    P_RANGE_ARR[valid], frac_surface_arr[valid] * 100,
    color=CURVE_STYLES['surface_region']['color'],
    lw=STYLE['linewidth'], label='Surface (dissociation)'
)
ax3.semilogx(
    P_RANGE_ARR[valid], frac_metal_arr[valid] * 100,
    color=CURVE_STYLES['metal_region']['color'],
    lw=STYLE['linewidth'], label='Metal (diffusion)'
)

# 50% threshold
ax3.axhline(
    50,
    color=CURVE_STYLES['threshold']['color'],
    ls=CURVE_STYLES['threshold']['ls'],
    lw=CURVE_STYLES['threshold']['lw'],
    alpha=CURVE_STYLES['threshold']['alpha'],
    label='50% threshold'
)

# Crossover annotation
if P_crossover is not None:
    ax3.axvline(P_crossover, color='gray', ls=':', lw=1.5, alpha=0.7)
    ax3.text(
        P_crossover * 1.5, 53,
        f'P = {P_crossover:.1e} Pa',
        fontsize=STYLE['fontsize_annotation']-1, color='gray'
    )
else:
    dominant = 'Surface-limited' if n_surface > n_metal else 'Metal-limited'
    ax3.text(
        0.05, 0.50,
        f'No crossover in range\n{dominant} throughout',
        transform=ax3.transAxes,
        fontsize=STYLE['fontsize_annotation'],
        va='center',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9)
    )

ax3.set_xlabel('Upstream Pressure $P_{up}$ (Pa)', fontsize=STYLE['fontsize_axis'])
ax3.set_ylabel('Resistance Fraction (%)',          fontsize=STYLE['fontsize_axis'])
ax3.set_title('(C) Rate-Limiting Step Analysis',   fontsize=STYLE['fontsize_title'])
ax3.set_ylim(0, 105)
ax3.legend(fontsize=STYLE['fontsize_legend'], loc='center right')
ax3.grid(True, alpha=STYLE['grid_alpha'])
ax3.tick_params(labelsize=STYLE['fontsize_tick'])

# ─────────────────────────────────────────────────────────────────────────────
# (D) LIMIT CHECK — Sieverts Recovery as k_diss → ∞
# ─────────────────────────────────────────────────────────────────────────────
ax4 = axes[1, 1]

# Sieverts reference (L1, no surface kinetics)
J_sieverts = beta * (np.sqrt(P_RANGE_ARR) - np.sqrt(P_down))

# Sweep k_diss over many orders of magnitude
k_diss_sweep = np.logspace(
    np.log10(CONDITIONS['k_diss_sweep_range'][0]),
    np.log10(CONDITIONS['k_diss_sweep_range'][1]),
    CONDITIONS['n_points']
)

# Compute L1+L6 at the highest k_diss only — for parity plot
J_high_k = []
for P_up in P_RANGE_ARR:
    try:
        r = solve_steady_state_flux_L1L6(
            P_up, P_down, L_m,
            k_diss_sweep[-1], K_eq, D_m, K_s_m
        )
        J_high_k.append(r['J_ss'])
    except Exception:
        J_high_k.append(np.nan)
J_high_k = np.array(J_high_k)

valid_parity = ~np.isnan(J_high_k) & (J_sieverts > 0) & (J_high_k > 0)

# Parity plot
ax4.loglog(
    J_sieverts[valid_parity], J_high_k[valid_parity],
    'o',
    color=CURVE_STYLES['L1_L6']['color'],
    markersize=6, alpha=0.8,
    label=f'L1+L6  (k_diss = {k_diss_sweep[-1]:.0e})'
)

# Perfect agreement line
J_min = min(J_sieverts[valid_parity].min(), J_high_k[valid_parity].min()) * 0.5
J_max = max(J_sieverts[valid_parity].max(), J_high_k[valid_parity].max()) * 2.0
ax4.loglog(
    [J_min, J_max], [J_min, J_max],
    color=CURVE_STYLES['parity']['color'],
    ls=CURVE_STYLES['parity']['ls'],
    lw=CURVE_STYLES['parity']['lw'],
    alpha=CURVE_STYLES['parity']['alpha'],
    label='Perfect agreement'
)

# Max relative error
rel_errors = (np.abs(J_high_k[valid_parity] - J_sieverts[valid_parity])
              / J_sieverts[valid_parity] * 100)
max_err    = rel_errors.max()
mean_err   = rel_errors.mean()
box_color  = 'lightgreen' if max_err < 1.0 else 'lightyellow'

ax4.text(
    0.05, 0.95,
    f'k_diss = {k_diss_sweep[-1]:.0e}\n'
    f'Max error  = {max_err:.2e}%\n'
    f'Mean error = {mean_err:.2e}%\n'
    f'{"✓ VALIDATED" if max_err < 1.0 else "⚠ CHECK k_diss"}',
    transform=ax4.transAxes,
    fontsize=STYLE['fontsize_annotation'],
    va='top',
    bbox=dict(boxstyle='round', facecolor=box_color, alpha=0.9)
)

ax4.set_xlabel('Sieverts Flux $J_{L1}$ (mol/m²/s)',         fontsize=STYLE['fontsize_axis'])
ax4.set_ylabel('L1+L6 Flux $J_{L1+L6}$ (mol/m²/s)',         fontsize=STYLE['fontsize_axis'])
ax4.set_title(r'(D) Limit Check: $k_{diss} \to \infty$ → Sieverts', fontsize=STYLE['fontsize_title'])
ax4.set_aspect('equal', adjustable='box')
ax4.legend(fontsize=STYLE['fontsize_legend'], loc='lower right')
ax4.grid(True, which='both', alpha=STYLE['grid_alpha'])
ax4.tick_params(labelsize=STYLE['fontsize_tick'])

# ─────────────────────────────────────────────────────────────────────────────
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
save_figure(fig, "L1L6_core_validation", FIGS)

# ── Validation summary ───────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("L1+L6 FIGURE 1 — VALIDATION SUMMARY")
print("=" * 60)
print(f"  Net log-log slope:       {slope_net:.4f}")
print(f"  θ range:                 "
      f"{np.nanmin(theta_arr):.4f} → {np.nanmax(theta_arr):.4f}")
print(f"  Sieverts recovery error: {max_err:.2e}%  "
      f"({'✓' if max_err < 1.0 else '⚠'})")
if P_crossover is not None:
    print(f"  Crossover pressure:      {P_crossover:.2e} Pa")
else:
    dominant = 'surface-limited' if n_surface > n_metal else 'metal-limited'
    print(f"  No crossover — system is {dominant} across full range")

findfont: Failed to find font weight bold for STIXSizeFiveSym, now using 400.


findfont: Failed to find font weight bold for cmmi10, now using 400.


findfont: Failed to find font weight bold for cmsy10, now using 400.


findfont: Failed to find font weight bold for cmr10, now using 400.


findfont: Failed to find font weight bold for cmtt10, now using 400.


findfont: Failed to find font weight bold for cmti10, now using 400.


... (2 outputs left out)

findfont: Failed to find font weight bold for cmex10, now using 400.


findfont: Failed to find font weight bold for DejaVu Sans Display, now using 400.


/var/folders/d4/mt_nqf_x42vgxs9t9mw1l59m0000gn/T/ipykernel_20394/2271907852.py:355: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



L1+L6 FIGURE 1 — VALIDATION SUMMARY
  Net log-log slope:       0.5904
  θ range:                 0.0000 → 0.9888
  Sieverts recovery error: 7.12e+01%  (⚠)
  Crossover pressure:      1.06e-02 Pa


In [8]:
# =============================================================================
# CELL 6 — DATAFRAME OUTPUT
# =============================================================================
df = pd.DataFrame(rows)

# ── Formatting ────────────────────────────────────────────────────────────────
df_display = df.copy()

for col in df_display.columns:
    if col == 'Rate-Limiting' or col == 'Error':
        continue
    if df_display[col].dtype == float:
        if 'fraction' in col.lower():
            df_display[col] = df_display[col].round(2)
        elif 'theta' in col.lower():
            df_display[col] = df_display[col].round(6)
        else:
            df_display[col] = df_display[col].apply(
                lambda x: f'{x:.4e}' if pd.notna(x) else 'NaN'
            )

# ── Summary line ──────────────────────────────────────────────────────────────
n_surface_df = (df['Rate-Limiting'] == 'SURFACE').sum()
n_metal_df   = (df['Rate-Limiting'] == 'METAL').sum()
n_mixed_df   = (df['Rate-Limiting'] == 'MIXED').sum()
n_error_df   = (df['Rate-Limiting'] == 'ERROR').sum()

print("=" * 60)
print("L1+L6 RESULTS DATAFRAME")
print("=" * 60)
print(f"  Metal:       {METAL_KEY}")
print(f"  T:           {T_op-273.15:.3f}°C  ({T_op} K)")
print(f"  L_m:         {L_m*1e3:.3f} mm")
print(f"  k_diss:      {k_diss:.3e}  mol/m²/s/Pa")
print(f"  K_eq:        {K_eq:.3e}  Pa⁻¹")
print(f"  D_m:         {D_m:.3e}  m²/s")
print(f"  K_s_m:       {K_s_m:.3e}  mol/m³/Pa^0.5")
print(f"\n  Total rows:        {len(df)}")
print(f"  Surface-limited:   {n_surface_df}")
print(f"  Metal-limited:     {n_metal_df}")
print(f"  Mixed:             {n_mixed_df}")
if n_error_df > 0:
    print(f"  Errors:            {n_error_df}  ⚠")
print("=" * 60)

display(df_display)

L1+L6 RESULTS DATAFRAME
  Metal:       metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985
  T:           599.850°C  (873 K)
  L_m:         1.000 mm
  k_diss:      4.611e-07  mol/m²/s/Pa
  K_eq:        8.212e-04  Pa⁻¹
  D_m:         2.487e-10  m²/s
  K_s_m:       3.834e-02  mol/m³/Pa^0.5

  Total rows:        40
  Surface-limited:   15
  Metal-limited:     25
  Mixed:             0


,P_up (Pa),P_int (Pa),J_ss (mol/m²/s),θ_surface,β_metal,R_surface,R_metal,R_total,fraction_surface (%),fraction_metal (%),Rate-Limiting
0,1.0000e-07,2.3377e-11,4.6098e-14,1.3855e-07,9.5343e-09,2.1688e+13,1.0488e+08,2.1688e+13,100.00,0.00,SURFACE
1,2.2855e-07,1.2203e-10,1.0532e-13,3.1656e-07,9.5343e-09,9.4895e+12,1.0488e+08,9.4896e+12,100.00,0.00,SURFACE
2,5.2233e-07,6.3654e-10,2.4055e-13,7.2298e-07,9.5343e-09,4.1521e+12,1.0488e+08,4.1522e+12,100.00,0.00,SURFACE
3,1.1938e-06,3.3145e-09,5.4890e-13,1.6498e-06,9.5343e-09,1.8168e+12,1.0488e+08,1.8169e+12,99.99,0.01,SURFACE
4,2.7283e-06,1.7190e-08,1.2501e-12,3.7571e-06,9.5343e-09,7.9492e+11,1.0488e+08,7.9502e+11,99.99,0.01,SURFACE
5,6.2355e-06,8.8372e-08,2.8343e-12,8.5186e-06,9.5343e-09,3.4782e+11,1.0488e+08,3.4792e+11,99.97,0.03,SURFACE
6,1.4251e-05,4.4570e-07,6.3652e-12,1.9131e-05,9.5343e-09,1.5219e+11,1.0488e+08,1.5230e+11,99.93,0.07,SURFACE
7,3.2570e-05,2.1622e-06,1.4020e-11,4.2135e-05,9.5343e-09,6.6594e+10,1.0488e+08,6.6699e+10,99.84,0.16,SURFACE
8,7.4438e-05,9.7754e-06,2.9810e-11,8.9587e-05,9.5343e-09,2.9141e+10,1.0488e+08,2.9246e+10,99.64,0.36,SURFACE
9,1.7013e-04,3.9734e-05,6.0100e-11,1.8060e-04,9.5343e-09,1.2753e+10,1.0488e+08,1.2858e+10,99.18,0.82,SURFACE


### L2a+L6: Perfect Oxide + Surface Kinetics 

In [9]:
# =============================================================================
# CELL 1 — SETUP
# Run once. All imports and helper functions.
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import brentq
import pandas as pd
from itertools import groupby
from operator import itemgetter
from IPython.display import display

from calculations.surface_kinetics import solve_steady_state_flux_L2aL6
from calculations.config.model_config import (
    R, F,
    METALS, OXIDES, CONDITIONS, SUGGESTED_RANGES_LEVEL5,
    PLOT_STYLE as STYLE,
    CURVE_STYLES,
    apply_style,
    build_simulation_config,
)

# ── Simulation config ─────────────────────────────────────────────────────────
SIM      = build_simulation_config()

OXIDE_KEY = SIM['oxide_name']
OXIDE     = OXIDES[OXIDE_KEY]
SK        = OXIDE['surface_kinetics']     # oxide surface kinetics sub-dict

# ── Pressure sweep (matches widget range) ─────────────────────────────────────
P_RANGE_ARR = np.logspace(np.log10(CONDITIONS['P_range'][0]), np.log10(CONDITIONS['P_range'][1]), CONDITIONS['n_points'])
P_DOWN      = SIM['P_downstream']
T_RANGE     = CONDITIONS['T_range']
N_T         = CONDITIONS['n_points']

# ── Arrhenius helper functions ────────────────────────────────────────────────
def get_k_diss_at_T(T_K):
    """k_diss at temperature T_K via Arrhenius from config."""
    return SK['k_diss_ref'] * np.exp(
        (-SK['E_diss'] / R) * (1/T_K - 1/SK['T_ref'])
    )

def get_K_eq_at_T(T_K):
    """K_eq at temperature T_K via van't Hoff from config."""
    return SK['K_eq_ref'] * np.exp(
        (-SK['H_eq'] / R) * (1/T_K - 1/SK['T_ref'])
    )

def get_D_ox_at_T(T_K):
    """D_ox at temperature T_K via Arrhenius from config."""
    return OXIDE['D_ox_ref'] * np.exp(
        (-OXIDE['E_D_ox'] / R) * (1/T_K - 1/OXIDE['T_ref'])
    )

def get_K_ox_at_T(T_K):
    """K_ox at temperature T_K via Arrhenius from config."""
    return OXIDE['K_ox_ref'] * np.exp(
        (-OXIDE['H_sol_ox'] / R) * (1/T_K - 1/OXIDE['T_ref'])
    )

# ── Known reference activation energies ──────────────────────────────────────
E_REF_SURFACE = SK['E_diss']                          # J/mol — surface dissociation
E_REF_OXIDE   = OXIDE['E_D_ox'] + OXIDE['H_sol_ox']  # J/mol — oxide permeation

print("=" * 60)
print("SETUP COMPLETE")
print("=" * 60)
print(f"  Oxide:           {OXIDE_KEY}")
print(f"  E_ref surface:   {E_REF_SURFACE/1000:.3f} kJ/mol  (E_diss)")
print(f"  E_ref oxide:     {E_REF_OXIDE/1000:.3f} kJ/mol  (E_D_ox + H_sol_ox)")
print(f"  P range:         {P_RANGE_ARR[0]:.0e} – {P_RANGE_ARR[-1]:.0e} Pa")
print(f"  T range:         {T_RANGE[0]} – {T_RANGE[1]} K")

SETUP COMPLETE
  Oxide:           Cr2O3_sample4
  E_ref surface:   57.950 kJ/mol  (E_diss)
  E_ref oxide:     234.000 kJ/mol  (E_D_ox + H_sol_ox)
  P range:         1e-07 – 1e+07 Pa
  T range:         573 – 1273 K


In [10]:
# =============================================================================
# CELL 2 — PARAMETERS
# All physics values at operating temperature, pulled from config.
# Change placeholder values in config and rerun from this cell downward.
# =============================================================================

# ── Operating conditions ──────────────────────────────────────────────────────
T_op   = SIM['T_operating']
L_ox   = SIM['L_oxide']
P_down = SIM['P_downstream']

# ── Surface kinetics at T_op ──────────────────────────────────────────────────
k_diss   = get_k_diss_at_T(T_op)
K_eq     = get_K_eq_at_T(T_op)
k_recomb = k_diss / K_eq

# ── Oxide transport at T_op ───────────────────────────────────────────────────
D_ox = get_D_ox_at_T(T_op)
K_ox = get_K_ox_at_T(T_op)

# ── Derived ───────────────────────────────────────────────────────────────────
alpha = D_ox * K_ox / L_ox    # oxide permeance [mol/m²/s/Pa^0.5]

# ── Print verification ────────────────────────────────────────────────────────
print("=" * 60)
print(f"L2a+L6 PARAMETERS AT T = {T_op-273.15:.3f}°C  ({T_op} K)")
print("=" * 60)

print(f"\n  Geometry")
print(f"    L_ox     = {L_ox:.3e}  m")
print(f"    P_down   = {P_down:.1e}  Pa")

print(f"\n  Surface kinetics  [config: oxide surface_kinetics sub-dict]")
print(f"    k_diss   = {k_diss:.3e}  mol/m²/s/Pa")
print(f"    K_eq     = {K_eq:.3e}  Pa⁻¹")
print(f"    k_recomb = {k_recomb:.3e}  mol/m²/s  (= k_diss / K_eq)")

print(f"\n  Oxide transport  [config: Arrhenius]")
print(f"    D_ox     = {D_ox:.3e}  m²/s")
print(f"    K_ox     = {K_ox:.3e}  mol/m³/Pa^0.5")

print(f"\n  Derived")
print(f"    α        = {alpha:.3e}  mol/m²/s/Pa^0.5  (= D_ox × K_ox / L_ox)")

print(f"\n  Reference activation energies  [config]")
print(f"    E_diss        = {E_REF_SURFACE/1000:.3f} kJ/mol   (surface dissociation)")
print(f"    E_D_ox+H_sol  = {E_REF_OXIDE/1000:.3f} kJ/mol   (oxide permeation)")

if SK['k_diss_ref'] <= 1e-12:
    print(f"\n  ⚠  WARNING: k_diss_ref = {SK['k_diss_ref']:.0e} looks like a placeholder.")
    print(f"     Update config oxide surface_kinetics sub-dict with literature values.")

print("\n" + "=" * 60)

L2a+L6 PARAMETERS AT T = 599.850°C  (873 K)

  Geometry
    L_ox     = 1.000e-06  m
    P_down   = 0.0e+00  Pa

  Surface kinetics  [config: oxide surface_kinetics sub-dict]
    k_diss   = 2.370e-09  mol/m²/s/Pa
    K_eq     = 2.799e-05  Pa⁻¹
    k_recomb = 8.468e-05  mol/m²/s  (= k_diss / K_eq)

  Oxide transport  [config: Arrhenius]
    D_ox     = 1.395e-17  m²/s
    K_ox     = 2.869e+02  mol/m³/Pa^0.5

  Derived
    α        = 4.002e-09  mol/m²/s/Pa^0.5  (= D_ox × K_ox / L_ox)

  Reference activation energies  [config]
    E_diss        = 57.950 kJ/mol   (surface dissociation)
    E_D_ox+H_sol  = 234.000 kJ/mol   (oxide permeation)



In [11]:
# =============================================================================
# CELL 3 — COMPUTE
# Single-pass loop over full pressure range.
# Builds all arrays, regime summary, and Arrhenius pressure selection.
# =============================================================================

# ── Single-pass loop ──────────────────────────────────────────────────────────
rows             = []
J_arr            = []
theta_arr        = []
P_surf_arr       = []
frac_surface_arr = []
frac_oxide_arr   = []
R_surface_arr    = []
R_oxide_arr      = []
R_total_arr      = []
rate_lim_arr     = []

for P_up in P_RANGE_ARR:
    try:
        r = solve_steady_state_flux_L2aL6(
            P_up, P_down,
            k_diss, K_eq, D_ox, K_ox, L_ox
        )
        J_arr.append(r['J_ss'])
        theta_arr.append(r['theta'])
        P_surf_arr.append(r['P_surf'])
        frac_surface_arr.append(r['resistances']['fraction_surface'])
        frac_oxide_arr.append(r['resistances']['fraction_oxide'])
        R_surface_arr.append(r['resistances']['R_surface'])
        R_oxide_arr.append(r['resistances']['R_oxide'])
        R_total_arr.append(r['resistances']['R_total'])
        rate_lim_arr.append(r['rate_limiting'])

        rows.append({
            'P_up (Pa)':            P_up,
            'P_surf (Pa)':          r['P_surf'],
            'J_ss (mol/m²/s)':      r['J_ss'],
            'θ_surface':            r['theta'],
            'α_oxide':              r['alpha'],
            'R_surface':            r['resistances']['R_surface'],
            'R_oxide':              r['resistances']['R_oxide'],
            'R_total':              r['resistances']['R_total'],
            'fraction_surface (%)': r['resistances']['fraction_surface'] * 100,
            'fraction_oxide (%)':   r['resistances']['fraction_oxide']   * 100,
            'Rate-Limiting':        r['rate_limiting'].upper(),
        })

    except Exception as e:
        J_arr.append(np.nan);            theta_arr.append(np.nan)
        P_surf_arr.append(np.nan);       frac_surface_arr.append(np.nan)
        frac_oxide_arr.append(np.nan);   R_surface_arr.append(np.nan)
        R_oxide_arr.append(np.nan);      R_total_arr.append(np.nan)
        rate_lim_arr.append('error')
        rows.append({'P_up (Pa)': P_up, 'Rate-Limiting': 'ERROR', 'Error': str(e)})

# ── Convert to arrays ─────────────────────────────────────────────────────────
J_arr            = np.array(J_arr)
theta_arr        = np.array(theta_arr)
P_surf_arr       = np.array(P_surf_arr)
frac_surface_arr = np.array(frac_surface_arr)
frac_oxide_arr   = np.array(frac_oxide_arr)
R_surface_arr    = np.array(R_surface_arr)
R_oxide_arr      = np.array(R_oxide_arr)
R_total_arr      = np.array(R_total_arr)
rate_lim_arr     = np.array(rate_lim_arr)
valid            = ~np.isnan(J_arr)

# ── Rate-limiting classification array ───────────────────────────────────────
rl_arr = np.where(frac_surface_arr > 0.5, 'surface',
         np.where(frac_oxide_arr   > 0.5, 'oxide', 'mixed'))

# ── Regime summary ────────────────────────────────────────────────────────────
n_surface = np.sum((frac_surface_arr > 0.5) & valid)
n_oxide   = np.sum((frac_oxide_arr   > 0.5) & valid)
n_mixed   = np.sum(
    (frac_surface_arr <= 0.5) & (frac_oxide_arr <= 0.5) & valid
)

surf_dom_mask  = (frac_surface_arr > 0.90) & valid
oxide_dom_mask = (frac_oxide_arr   > 0.90) & valid
P_surf_dom     = P_RANGE_ARR[surf_dom_mask]
P_oxide_dom    = P_RANGE_ARR[oxide_dom_mask]

# ── Arrhenius pressure selection (no crossover dependence) ───────────────────
P_for_surface_Ea = np.median(P_surf_dom)  if len(P_surf_dom)  > 3 else None
P_for_oxide_Ea   = np.median(P_oxide_dom) if len(P_oxide_dom) > 3 else None

# ── Crossover pressure (diagnostic only) ─────────────────────────────────────
P_crossover = None
valid_cross = valid & ~np.isnan(frac_surface_arr)
if np.sum(valid_cross) > 2:
    fs_v = frac_surface_arr[valid_cross]
    P_v  = P_RANGE_ARR[valid_cross]
    sc   = np.where(np.diff(np.sign(fs_v - 0.5)))[0]
    if len(sc) > 0:
        idx = sc[0]
        if idx + 1 < len(fs_v) and fs_v[idx] > 0.5 and fs_v[idx+1] < 0.5:
            P_crossover = P_v[idx]

# ── Print regime summary ──────────────────────────────────────────────────────
print("=" * 60)
print(f"L2a+L6 REGIME SUMMARY  (T = {T_op-273.15:.3f}°C, {np.sum(valid)} valid points)")
print("=" * 60)

print(f"\n  Regime breakdown (fraction > 50%)")
print(f"    Surface-limited:  {n_surface:3d} points")
print(f"    Oxide-limited:    {n_oxide:3d} points")
print(f"    Mixed:            {n_mixed:3d} points")

print(f"\n  Dominant regions (fraction > 90%) — used for Arrhenius")
if len(P_surf_dom) > 3:
    print(f"    Surface-dominant: {len(P_surf_dom):3d} points  "
          f"P = {P_surf_dom.min():.1e} – {P_surf_dom.max():.1e} Pa")
    print(f"    → P_for_surface_Ea = {P_for_surface_Ea:.2e} Pa")
else:
    print(f"    Surface-dominant: {len(P_surf_dom):3d} points  "
          f"(insufficient — surface Ea not extractable)")

if len(P_oxide_dom) > 3:
    print(f"    Oxide-dominant:   {len(P_oxide_dom):3d} points  "
          f"P = {P_oxide_dom.min():.1e} – {P_oxide_dom.max():.1e} Pa")
    print(f"    → P_for_oxide_Ea  = {P_for_oxide_Ea:.2e} Pa")
else:
    print(f"    Oxide-dominant:   {len(P_oxide_dom):3d} points  "
          f"(insufficient — oxide Ea not extractable)")

print(f"\n  Crossover pressure (diagnostic only)")
if P_crossover is not None:
    print(f"    P_crossover = {P_crossover:.2e} Pa")
else:
    print(f"    No crossover detected in physical pressure range")
    dominant = 'surface-limited' if n_surface > n_oxide else 'oxide-limited'
    print(f"    System is predominantly {dominant} across full range")

print(f"\n  θ range:  {np.nanmin(theta_arr):.4f} → {np.nanmax(theta_arr):.4f}")
print(f"  J range:  {np.nanmin(J_arr):.2e} → {np.nanmax(J_arr):.2e} mol/m²/s")
print("=" * 60)

L2a+L6 REGIME SUMMARY  (T = 599.850°C, 40 valid points)

  Regime breakdown (fraction > 50%)
    Surface-limited:   21 points
    Oxide-limited:     19 points
    Mixed:              0 points

  Dominant regions (fraction > 90%) — used for Arrhenius
    Surface-dominant:  18 points  P = 1.0e-07 – 1.3e-01 Pa
    → P_for_surface_Ea = 1.22e-04 Pa
    Oxide-dominant:    17 points  P = 1.8e+01 – 1.0e+07 Pa
    → P_for_oxide_Ea  = 1.34e+04 Pa

  Crossover pressure (diagnostic only)
    P_crossover = 1.51e+00 Pa

  θ range:  0.0000 → 0.9396
  J range:  2.37e-16 → 1.18e-05 mol/m²/s


In [12]:
# =============================================================================
# CELL 4 — FIGURE 1: Core Validation (2×2)
# =============================================================================
"""
Figure 1 validates the L2a+L6 model from four angles at operating temperature.

Panel (A) — Flux vs Pressure
    Question: Does the model produce the correct pressure-dependent behaviour?
    The hero plot. Shows J_ss vs P_up on a log-log scale. The curve should
    transition from slope=1 (surface-limited, low P) to slope=0.5
    (oxide-limited, high P). Note: in the pure L2a limit (no surface
    kinetics), the oxide follows Sieverts-like solubility so the slope is
    always 0.5. The addition of L6 introduces the slope=1 surface-limited
    regime at low P. Style follows the widget — no fill_between. The curve
    is recolored by regime using thick colored segments with slope annotation
    boxes sitting directly on each segment. Reference lines for slope=1 and
    slope=0.5 are drawn as dashed lines. Net slope annotation box in the
    bottom right corner.

Panel (B) — Surface Coverage θ vs Pressure
    Question: How does surface occupancy evolve with pressure, and does it
    match the Langmuir equilibrium isotherm?
    Plots steady-state θ from the compute loop alongside the equilibrium
    Langmuir isotherm θ_eq = sqrt(K_eq*P)/(1+sqrt(K_eq*P)) as a gray
    dash-dot line. In the oxide-limited regime the surface is in local
    equilibrium (θ_ss ≈ θ_eq). In the surface-limited regime the surface
    is being depleted faster than it can be replenished (θ_ss < θ_eq).
    Crossover pressure marked if it exists.

Panel (C) — Resistance Fractions vs Pressure
    Question: Which step controls the rate, and how does the balance shift
    with pressure?
    Plots R_surface/R_total and R_oxide/R_total as percentages vs pressure
    on a semilog scale. Note: there is no metal resistance term here — the
    two fractions always sum to 100%. The 50% threshold line marks the
    crossover. If no crossover exists a text annotation states which regime
    dominates throughout.

Panel (D) — Limit Check: Fast-Kinetics Recovery (L2a)
    Question: As k_diss → ∞, does L2a+L6 recover the pure L2a result?
    Runs the solver at k_diss = 1e-3 (fast kinetics limit) and compares
    J_L2a+L6 against the analytical L2a flux J = alpha*(sqrt(P_up) -
    sqrt(P_down)). A parity plot shows J_L2a+L6 vs J_L2a — perfect
    agreement is a diagonal line. This validates the model correctly
    recovers oxide-only Sieverts diffusion when surface kinetics is fast.
    Green box if max error < 1%, yellow otherwise.
"""

props = dict(boxstyle='round', facecolor='wheat', alpha=0.9)

fig, axes = plt.subplots(2, 2, figsize=STYLE['figsize'])
fig.suptitle(
    f'L2a+L6: Perfect Oxide + Surface Kinetics — Core Validation\n'
    f'{OXIDE_KEY}  |  T = {T_op-273.15:.3f}°C  |  L_ox = {L_ox*1e6:.3f} μm',
    fontsize=STYLE['fontsize_suptitle'], fontweight='bold', y=0.98
)

# ─────────────────────────────────────────────────────────────────────────────
# (A) FLUX vs PRESSURE
# ─────────────────────────────────────────────────────────────────────────────
ax1 = axes[0, 0]

ax1.loglog(
    P_RANGE_ARR[valid], J_arr[valid],
    color='black', lw=STYLE['linewidth'], label='L2a+L6 Model'
)

if np.any(valid):
    P_ref1 = P_RANGE_ARR[valid][0]
    J_ref1 = J_arr[valid][0]
    ax1.loglog(
        P_RANGE_ARR[valid],
        J_ref1 * (P_RANGE_ARR[valid] / P_ref1) ** 1.0,
        color=CURVE_STYLES['slope_1']['color'],
        ls=CURVE_STYLES['slope_1']['ls'],
        lw=CURVE_STYLES['slope_1']['lw'],
        alpha=CURVE_STYLES['slope_1']['alpha'],
        label='Slope = 1 (surface)'
    )
    P_ref05 = P_RANGE_ARR[valid][-1]
    J_ref05 = J_arr[valid][-1]
    ax1.loglog(
        P_RANGE_ARR[valid],
        J_ref05 * (P_RANGE_ARR[valid] / P_ref05) ** 0.5,
        color=CURVE_STYLES['slope_05']['color'],
        ls=CURVE_STYLES['slope_05']['ls'],
        lw=CURVE_STYLES['slope_05']['lw'],
        alpha=CURVE_STYLES['slope_05']['alpha'],
        label='Slope = 0.5 (oxide)'
    )

# Colored regime segments + slope annotation boxes
region_styles = [
    ('surface', CURVE_STYLES['surface_region']['color'], 'Surface-limited'),
    ('oxide',   CURVE_STYLES['oxide_region']['color'],   'Oxide-limited'),
    ('mixed',   CURVE_STYLES['mixed_region']['color'],   'Mixed'),
]

for regime, color, label in region_styles:
    mask = (rl_arr == regime) & valid
    if not np.any(mask):
        continue
    idxs = np.where(mask)[0]
    for _, g in groupby(enumerate(idxs), lambda x: x[0] - x[1]):
        group = list(map(itemgetter(1), g))
        if len(group) < 3:
            continue
        P_seg = P_RANGE_ARR[group]
        J_seg = J_arr[group]
        ax1.loglog(P_seg, J_seg, color=color, lw=4, alpha=0.7)
        slope_seg, *_ = stats.linregress(
            np.log10(P_seg), np.log10(np.abs(J_seg))
        )
        mid = len(group) // 2
        ax1.text(
            P_seg[mid], J_seg[mid],
            f'{label}\nSlope={slope_seg:.2f}',
            color=color,
            fontsize=STYLE['fontsize_annotation'],
            fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
        )

log_P_v   = np.log10(P_RANGE_ARR[valid])
log_J_v   = np.log10(np.abs(J_arr[valid]))
slope_net, *_ = stats.linregress(log_P_v, log_J_v)

ax1.text(
    0.98, 0.02, f'Net slope = {slope_net:.2f}',
    transform=ax1.transAxes, ha='right', va='bottom',
    fontsize=STYLE['fontsize_annotation'], fontweight='bold',
    bbox=dict(boxstyle='square', fc='wheat', ec='gray', alpha=1)
)
ax1.text(
    0.05, 0.95,
    r'$J = \alpha\left(g(\theta) - \sqrt{P_{down}}\right)$',
    transform=ax1.transAxes, fontsize=10, va='top',
    bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8)
)

ax1.set_xlabel('Upstream Pressure $P_{up}$ (Pa)', fontsize=STYLE['fontsize_axis'])
ax1.set_ylabel('Flux $J_{ss}$ (mol/m²/s)',         fontsize=STYLE['fontsize_axis'])
ax1.set_title('(A) Flux vs Pressure',               fontsize=STYLE['fontsize_title'])
ax1.legend(fontsize=STYLE['fontsize_legend']-1, loc='upper left')
ax1.grid(True, which='both', alpha=STYLE['grid_alpha'])
ax1.tick_params(labelsize=STYLE['fontsize_tick'])

# ─────────────────────────────────────────────────────────────────────────────
# (B) SURFACE COVERAGE θ vs PRESSURE
# ─────────────────────────────────────────────────────────────────────────────
ax2 = axes[0, 1]

ax2.semilogx(
    P_RANGE_ARR[valid], theta_arr[valid],
    color=CURVE_STYLES['L2a_L6']['color'],
    ls=CURVE_STYLES['L2a_L6']['ls'],
    lw=STYLE['linewidth'],
    marker=CURVE_STYLES['L2a_L6']['marker'],
    ms=CURVE_STYLES['L2a_L6']['ms'],
    markevery=8,
    label='θ steady-state'
)

theta_eq = (np.sqrt(K_eq * P_RANGE_ARR) /
            (1 + np.sqrt(K_eq * P_RANGE_ARR)))
ax2.semilogx(
    P_RANGE_ARR, theta_eq,
    color='gray', ls='-.', lw=1.5, alpha=0.7,
    label=r'θ$_{eq}$ (Langmuir isotherm)'
)
ax2.axhline(1.0, color='red',    ls='--', lw=1.5, alpha=0.5, label='θ = 1 (saturation)')
ax2.axhline(0.5, color='orange', ls='--', lw=1.5, alpha=0.5, label='θ = 0.5')

if P_crossover is not None:
    ax2.axvline(P_crossover, color='gray', ls=':', lw=1.5, alpha=0.7,
                label=f'Crossover  P={P_crossover:.1e} Pa')

ax2.text(
    0.05, 0.95,
    r'$\theta_{eq} = \frac{\sqrt{K_{eq} P}}{1 + \sqrt{K_{eq} P}}$',
    transform=ax2.transAxes, fontsize=10, va='top',
    bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8)
)
ax2.text(
    0.95, 0.30,
    'θ_ss < θ_eq → surface depleted\n   by fast oxide diffusion\n'
    'θ_ss ≈ θ_eq → local equilibrium\n   (oxide-limited)',
    transform=ax2.transAxes, fontsize=STYLE['fontsize_annotation']-1,
    va='top', ha='right',
    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8)
)

ax2.set_xlabel('Upstream Pressure $P_{up}$ (Pa)',  fontsize=STYLE['fontsize_axis'])
ax2.set_ylabel('Surface Coverage $\\theta$',        fontsize=STYLE['fontsize_axis'])
ax2.set_title('(B) Surface Coverage vs Pressure',   fontsize=STYLE['fontsize_title'])
ax2.set_ylim(-0.05, 1.1)
ax2.legend(fontsize=STYLE['fontsize_legend']-1, loc='lower right')
ax2.grid(True, which='both', alpha=STYLE['grid_alpha'])
ax2.tick_params(labelsize=STYLE['fontsize_tick'])

# ─────────────────────────────────────────────────────────────────────────────
# (C) RESISTANCE FRACTIONS vs PRESSURE
# ─────────────────────────────────────────────────────────────────────────────
ax3 = axes[1, 0]

ax3.semilogx(
    P_RANGE_ARR[valid], frac_surface_arr[valid] * 100,
    color=CURVE_STYLES['surface_region']['color'],
    lw=STYLE['linewidth'], label='Surface (dissociation)'
)
ax3.semilogx(
    P_RANGE_ARR[valid], frac_oxide_arr[valid] * 100,
    color=CURVE_STYLES['oxide_region']['color'],
    lw=STYLE['linewidth'], label='Oxide (diffusion)'
)
ax3.axhline(
    50,
    color=CURVE_STYLES['threshold']['color'],
    ls=CURVE_STYLES['threshold']['ls'],
    lw=CURVE_STYLES['threshold']['lw'],
    alpha=CURVE_STYLES['threshold']['alpha'],
    label='50% threshold'
)

if P_crossover is not None:
    ax3.axvline(P_crossover, color='gray', ls=':', lw=1.5, alpha=0.7)
    ax3.text(
        P_crossover * 1.5, 53,
        f'P = {P_crossover:.1e} Pa',
        fontsize=STYLE['fontsize_annotation']-1, color='gray'
    )
else:
    dominant = 'Surface-limited' if n_surface > n_oxide else 'Oxide-limited'
    ax3.text(
        0.05, 0.50,
        f'No crossover in range\n{dominant} throughout',
        transform=ax3.transAxes,
        fontsize=STYLE['fontsize_annotation'],
        va='center',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9)
    )

ax3.set_xlabel('Upstream Pressure $P_{up}$ (Pa)', fontsize=STYLE['fontsize_axis'])
ax3.set_ylabel('Resistance Fraction (%)',          fontsize=STYLE['fontsize_axis'])
ax3.set_title('(C) Rate-Limiting Step Analysis',   fontsize=STYLE['fontsize_title'])
ax3.set_ylim(0, 105)
ax3.legend(fontsize=STYLE['fontsize_legend'], loc='center right')
ax3.grid(True, alpha=STYLE['grid_alpha'])
ax3.tick_params(labelsize=STYLE['fontsize_tick'])

# ─────────────────────────────────────────────────────────────────────────────
# (D) LIMIT CHECK — Fast-Kinetics Recovery (L2a)
# ─────────────────────────────────────────────────────────────────────────────
ax4 = axes[1, 1]

# L2a reference flux (no surface kinetics)
J_L2a = alpha * (np.sqrt(P_RANGE_ARR) - np.sqrt(P_down))

# L2a+L6 at high k_diss (fast kinetics limit)
J_high_k = []
for P_up in P_RANGE_ARR:
    try:
        r = solve_steady_state_flux_L2aL6(
            P_up, P_down,
            1e-3, K_eq, D_ox, K_ox, L_ox    # k_diss → large
        )
        J_high_k.append(r['J_ss'])
    except Exception:
        J_high_k.append(np.nan)
J_high_k = np.array(J_high_k)

valid_parity = ~np.isnan(J_high_k) & (J_L2a > 0) & (J_high_k > 0)

ax4.loglog(
    J_L2a[valid_parity], J_high_k[valid_parity],
    'o', color=CURVE_STYLES['L2a_L6']['color'],
    markersize=6, alpha=0.8,
    label='L2a+L6  (k_diss = 1e-3)'
)

J_min = min(J_L2a[valid_parity].min(), J_high_k[valid_parity].min()) * 0.5
J_max = max(J_L2a[valid_parity].max(), J_high_k[valid_parity].max()) * 2.0
ax4.loglog(
    [J_min, J_max], [J_min, J_max],
    color=CURVE_STYLES['parity']['color'],
    ls=CURVE_STYLES['parity']['ls'],
    lw=CURVE_STYLES['parity']['lw'],
    alpha=CURVE_STYLES['parity']['alpha'],
    label='Perfect agreement'
)

rel_errors = (np.abs(J_high_k[valid_parity] - J_L2a[valid_parity])
              / J_L2a[valid_parity] * 100)
max_err   = rel_errors.max()
mean_err  = rel_errors.mean()
box_color = 'lightgreen' if max_err < 1.0 else 'lightyellow'

ax4.text(
    0.05, 0.95,
    f'k_diss = 1e-3\n'
    f'Max error  = {max_err:.2e}%\n'
    f'Mean error = {mean_err:.2e}%\n'
    f'{"✓ VALIDATED" if max_err < 1.0 else "⚠ CHECK k_diss"}',
    transform=ax4.transAxes, fontsize=STYLE['fontsize_annotation'],
    va='top',
    bbox=dict(boxstyle='round', facecolor=box_color, alpha=0.9)
)

ax4.set_xlabel('L2a Flux $J_{L2a}$ (mol/m²/s)',              fontsize=STYLE['fontsize_axis'])
ax4.set_ylabel('L2a+L6 Flux $J_{L2a+L6}$ (mol/m²/s)',        fontsize=STYLE['fontsize_axis'])
ax4.set_title(r'(D) Limit Check: $k_{diss} \to \infty$ → L2a', fontsize=STYLE['fontsize_title'])
ax4.set_aspect('equal', adjustable='box')
ax4.legend(fontsize=STYLE['fontsize_legend'], loc='lower right')
ax4.grid(True, which='both', alpha=STYLE['grid_alpha'])
ax4.tick_params(labelsize=STYLE['fontsize_tick'])

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
save_figure(fig, "L2aL6_core_validation", FIGS)

print("\n" + "=" * 60)
print("L2a+L6 FIGURE 1 — VALIDATION SUMMARY")
print("=" * 60)
print(f"  Net log-log slope:         {slope_net:.4f}")
print(f"  θ range:                   "
      f"{np.nanmin(theta_arr):.4f} → {np.nanmax(theta_arr):.4f}")
print(f"  L2a recovery error:        {max_err:.2e}%  "
      f"({'✓' if max_err < 1.0 else '⚠'})")
if P_crossover is not None:
    print(f"  Crossover pressure:        {P_crossover:.2e} Pa")
else:
    dominant = 'surface-limited' if n_surface > n_oxide else 'oxide-limited'
    print(f"  No crossover — system is {dominant} across full range")
print("=" * 60)

/var/folders/d4/mt_nqf_x42vgxs9t9mw1l59m0000gn/T/ipykernel_20394/1271253695.py:318: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



L2a+L6 FIGURE 1 — VALIDATION SUMMARY
  Net log-log slope:         0.7716
  θ range:                   0.0000 → 0.9396
  L2a recovery error:        6.31e-01%  (✓)
  Crossover pressure:        1.51e+00 Pa


In [14]:
# =============================================================================
# CELL 6 — DATAFRAME OUTPUT
# =============================================================================
df = pd.DataFrame(rows)

df_display = df.copy()
for col in df_display.columns:
    if col in ('Rate-Limiting', 'Error'):
        continue
    if df_display[col].dtype == float:
        if 'fraction' in col.lower():
            df_display[col] = df_display[col].round(2)
        elif 'theta' in col.lower():
            df_display[col] = df_display[col].round(6)
        else:
            df_display[col] = df_display[col].apply(
                lambda x: f'{x:.4e}' if pd.notna(x) else 'NaN'
            )

n_surface_df = (df['Rate-Limiting'] == 'SURFACE').sum()
n_oxide_df   = (df['Rate-Limiting'] == 'OXIDE').sum()
n_mixed_df   = (df['Rate-Limiting'] == 'MIXED').sum()
n_error_df   = (df['Rate-Limiting'] == 'ERROR').sum()

print("=" * 60)
print("L2a+L6 RESULTS DATAFRAME")
print("=" * 60)
print(f"  Oxide:       {OXIDE_KEY}")
print(f"  T:           {T_op-273.15:.3f}°C  ({T_op} K)")
print(f"  L_ox:        {L_ox*1e6:.3f} μm")
print(f"  k_diss:      {k_diss:.3e}  mol/m²/s/Pa")
print(f"  K_eq:        {K_eq:.3e}  Pa⁻¹")
print(f"  D_ox:        {D_ox:.3e}  m²/s")
print(f"  K_ox:        {K_ox:.3e}  mol/m³/Pa^0.5")
print(f"\n  Total rows:        {len(df)}")
print(f"  Surface-limited:   {n_surface_df}")
print(f"  Oxide-limited:     {n_oxide_df}")
print(f"  Mixed:             {n_mixed_df}")
if n_error_df > 0:
    print(f"  Errors:            {n_error_df}  ⚠")
print("=" * 60)

display(df_display)

L2a+L6 RESULTS DATAFRAME
  Oxide:       Cr2O3_sample4
  T:           599.850°C  (873 K)
  L_ox:        1.000 μm
  k_diss:      2.370e-09  mol/m²/s/Pa
  K_eq:        2.799e-05  Pa⁻¹
  D_ox:        1.395e-17  m²/s
  K_ox:        2.869e+02  mol/m³/Pa^0.5

  Total rows:        40
  Surface-limited:   21
  Oxide-limited:     19
  Mixed:             0


,P_up (Pa),P_surf (Pa),J_ss (mol/m²/s),θ_surface,α_oxide,R_surface,R_oxide,R_total,fraction_surface (%),fraction_oxide (%),Rate-Limiting
0,1.0000e-07,3.5082e-15,2.3702e-16,3.1336e-10,4.0017e-09,4.2191e+15,2.4990e+08,4.2191e+15,100.00,0.00,SURFACE
1,2.2855e-07,1.8325e-14,5.4170e-16,7.1616e-10,4.0017e-09,1.8460e+15,2.4990e+08,1.8460e+15,100.00,0.00,SURFACE
2,5.2233e-07,9.5716e-14,1.2380e-15,1.6368e-09,4.0017e-09,8.0773e+14,2.4990e+08,8.0773e+14,100.00,0.00,SURFACE
3,1.1938e-06,4.9996e-13,2.8295e-15,3.7408e-09,4.0017e-09,3.5342e+14,2.4990e+08,3.5342e+14,100.00,0.00,SURFACE
4,2.7283e-06,2.6114e-12,6.4667e-15,8.5494e-09,4.0017e-09,1.5464e+14,2.4990e+08,1.5464e+14,100.00,0.00,SURFACE
5,6.2355e-06,1.3640e-11,1.4779e-14,1.9539e-08,4.0017e-09,6.7662e+13,2.4990e+08,6.7662e+13,100.00,0.00,SURFACE
6,1.4251e-05,7.1248e-11,3.3777e-14,4.4656e-08,4.0017e-09,2.9605e+13,2.4990e+08,2.9606e+13,100.00,0.00,SURFACE
7,3.2570e-05,3.7215e-10,7.7197e-14,1.0206e-07,4.0017e-09,1.2954e+13,2.4990e+08,1.2954e+13,100.00,0.00,SURFACE
8,7.4438e-05,1.9438e-09,1.7643e-13,2.3325e-07,4.0017e-09,5.6679e+12,2.4990e+08,5.6681e+12,100.00,0.00,SURFACE
9,1.7013e-04,1.0152e-08,4.0321e-13,5.3307e-07,4.0017e-09,2.4800e+12,2.4990e+08,2.4802e+12,99.99,0.01,SURFACE


### L2a +L6

In [15]:
# =============================================================================
# CELL 1 — SETUP
# Run once. All imports and helper functions.
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import brentq
import pandas as pd
from itertools import groupby
from operator import itemgetter
from IPython.display import display

from calculations.surface_kinetics import solve_steady_state_flux_direct
from calculations.config.model_config import (
    R, F,
    METALS, OXIDES, CONDITIONS,
    PLOT_STYLE as STYLE,
    CURVE_STYLES,
    apply_style,
    build_simulation_config,
)

# ── Simulation config ─────────────────────────────────────────────────────────
SIM       = build_simulation_config()

METAL_KEY = SIM['metal_name']
OXIDE_KEY = SIM['oxide_name']
METAL     = METALS[METAL_KEY]
OXIDE     = OXIDES[OXIDE_KEY]
SK        = OXIDE['surface_kinetics']    # oxide surface kinetics sub-dict

# ── Pressure sweep ────────────────────────────────────────────────────────────
P_RANGE_ARR = np.logspace(
    np.log10(CONDITIONS['P_range'][0]),
    np.log10(CONDITIONS['P_range'][1]),
    CONDITIONS['n_points']
)
P_DOWN      = SIM['P_downstream']
T_RANGE     = CONDITIONS['T_range']

In [16]:
# =============================================================================
# CELL 2 — PARAMETERS
# All physics values at operating temperature, pulled from config.
# Change placeholder values in config and rerun from this cell downward.
# =============================================================================

# ── Operating conditions ──────────────────────────────────────────────────────
T_op   = SIM['T_operating']
L_ox   = SIM['L_oxide']
L_m    = SIM['L_metal']
P_down = SIM['P_downstream']

# ── Surface kinetics at T_op ──────────────────────────────────────────────────
k_diss   = get_k_diss_at_T(T_op)
K_eq     = get_K_eq_at_T(T_op)
k_recomb = k_diss / K_eq

# ── Oxide transport at T_op ───────────────────────────────────────────────────
D_ox = get_D_ox_at_T(T_op)
K_ox = get_K_ox_at_T(T_op)

# ── Metal transport at T_op ───────────────────────────────────────────────────
D_m   = get_D_m_at_T(T_op)
K_s_m = get_K_s_m_at_T(T_op)

# ── Derived ───────────────────────────────────────────────────────────────────
alpha = D_ox * K_ox / L_ox
beta  = D_m  * K_s_m / L_m

# ── Print verification ────────────────────────────────────────────────────────
print("=" * 60)
print(f"L2+L6 PARAMETERS AT T = {T_op-273.15:.3f}°C  ({T_op} K)")
print("=" * 60)

print(f"\n  Geometry")
print(f"    L_ox     = {L_ox:.3e}  m")
print(f"    L_m      = {L_m:.3e}  m")
print(f"    P_down   = {P_down:.1e}  Pa")

print(f"\n  Surface kinetics  [config: oxide surface_kinetics sub-dict]")
print(f"    k_diss   = {k_diss:.3e}  mol/m²/s/Pa")
print(f"    K_eq     = {K_eq:.3e}  Pa⁻¹")
print(f"    k_recomb = {k_recomb:.3e}  mol/m²/s  (= k_diss / K_eq)")

print(f"\n  Oxide transport  [config: Arrhenius]")
print(f"    D_ox     = {D_ox:.3e}  m²/s")
print(f"    K_ox     = {K_ox:.3e}  mol/m³/Pa^0.5")

print(f"\n  Metal transport  [config: Arrhenius]")
print(f"    D_m      = {D_m:.3e}  m²/s")
print(f"    K_s_m    = {K_s_m:.3e}  mol/m³/Pa^0.5")

print(f"\n  Derived")
print(f"    α        = {alpha:.3e}  mol/m²/s/Pa^0.5  (= D_ox × K_ox / L_ox)")
print(f"    β        = {beta:.3e}  mol/m²/s/Pa^0.5  (= D_m × K_s_m / L_m)")
print(f"    α/β      = {alpha/beta:.3e}  (oxide/metal permeance ratio)")

print(f"\n  Reference activation energies  [config]")
print(f"    E_diss        = {E_REF_SURFACE/1000:.3f} kJ/mol   (surface)")
print(f"    E_D_ox+H_sol  = {E_REF_OXIDE/1000:.3f} kJ/mol   (oxide permeation)")
print(f"    E_D+H_s       = {E_REF_METAL/1000:.3f} kJ/mol   (metal permeation)")

if SK['k_diss_ref'] <= 1e-12:
    print(f"\n  ⚠  WARNING: k_diss_ref = {SK['k_diss_ref']:.0e} looks like a placeholder.")
    print(f"     Update config oxide surface_kinetics sub-dict with literature values.")

print("\n" + "=" * 60)

L2+L6 PARAMETERS AT T = 599.850°C  (873 K)

  Geometry
    L_ox     = 1.000e-06  m
    L_m      = 1.000e-03  m
    P_down   = 0.0e+00  Pa

  Surface kinetics  [config: oxide surface_kinetics sub-dict]
    k_diss   = 2.370e-09  mol/m²/s/Pa
    K_eq     = 2.799e-05  Pa⁻¹
    k_recomb = 8.468e-05  mol/m²/s  (= k_diss / K_eq)

  Oxide transport  [config: Arrhenius]
    D_ox     = 1.395e-17  m²/s
    K_ox     = 2.869e+02  mol/m³/Pa^0.5

  Metal transport  [config: Arrhenius]
    D_m      = 2.487e-10  m²/s
    K_s_m    = 3.834e-02  mol/m³/Pa^0.5

  Derived
    α        = 4.002e-09  mol/m²/s/Pa^0.5  (= D_ox × K_ox / L_ox)
    β        = 9.534e-09  mol/m²/s/Pa^0.5  (= D_m × K_s_m / L_m)
    α/β      = 4.197e-01  (oxide/metal permeance ratio)

  Reference activation energies  [config]
    E_diss        = 57.950 kJ/mol   (surface)
    E_D_ox+H_sol  = 234.000 kJ/mol   (oxide permeation)
    E_D+H_s       = 79.400 kJ/mol   (metal permeation)



In [17]:
# =============================================================================
# CELL 3 — COMPUTE
# Single-pass loop over full pressure range.
# Builds all arrays, regime summary, and Arrhenius pressure selection.
# =============================================================================

# ── Single-pass loop ──────────────────────────────────────────────────────────
rows             = []
J_arr            = []
theta_arr        = []
P_int_arr        = []
frac_surface_arr = []
frac_oxide_arr   = []
frac_metal_arr   = []
R_surface_arr    = []
R_oxide_arr      = []
R_metal_arr      = []
R_total_arr      = []
rate_lim_arr     = []

for P_up in P_RANGE_ARR:
    try:
        r = solve_steady_state_flux_direct(
            P_up, P_down, L_m,
            k_diss, K_eq,
            D_ox, K_ox, L_ox,
            D_m, K_s_m
        )
        J_arr.append(r['J_ss'])
        theta_arr.append(r['theta'])
        P_int_arr.append(r['P_int'])
        frac_surface_arr.append(r['resistances']['fraction_surface'])
        frac_oxide_arr.append(r['resistances']['fraction_oxide'])
        frac_metal_arr.append(r['resistances']['fraction_metal'])
        R_surface_arr.append(r['resistances']['R_surface'])
        R_oxide_arr.append(r['resistances']['R_oxide'])
        R_metal_arr.append(r['resistances']['R_metal'])
        R_total_arr.append(r['resistances']['R_total'])
        rate_lim_arr.append(r['rate_limiting'])

        rows.append({
            'P_up (Pa)':            P_up,
            'P_int (Pa)':           r['P_int'],
            'J_ss (mol/m²/s)':      r['J_ss'],
            'θ_surface':            r['theta'],
            'α_oxide':              r['alpha'],
            'β_metal':              r['beta'],
            'R_surface':            r['resistances']['R_surface'],
            'R_oxide':              r['resistances']['R_oxide'],
            'R_metal':              r['resistances']['R_metal'],
            'R_total':              r['resistances']['R_total'],
            'fraction_surface (%)': r['resistances']['fraction_surface'] * 100,
            'fraction_oxide (%)':   r['resistances']['fraction_oxide']   * 100,
            'fraction_metal (%)':   r['resistances']['fraction_metal']   * 100,
            'Rate-Limiting':        r['rate_limiting'].upper(),
        })

    except Exception as e:
        for lst in [J_arr, theta_arr, P_int_arr, frac_surface_arr,
                    frac_oxide_arr, frac_metal_arr, R_surface_arr,
                    R_oxide_arr, R_metal_arr, R_total_arr]:
            lst.append(np.nan)
        rate_lim_arr.append('error')
        rows.append({'P_up (Pa)': P_up, 'Rate-Limiting': 'ERROR', 'Error': str(e)})

# ── Convert to arrays ─────────────────────────────────────────────────────────
J_arr            = np.array(J_arr)
theta_arr        = np.array(theta_arr)
P_int_arr        = np.array(P_int_arr)
frac_surface_arr = np.array(frac_surface_arr)
frac_oxide_arr   = np.array(frac_oxide_arr)
frac_metal_arr   = np.array(frac_metal_arr)
R_surface_arr    = np.array(R_surface_arr)
R_oxide_arr      = np.array(R_oxide_arr)
R_metal_arr      = np.array(R_metal_arr)
R_total_arr      = np.array(R_total_arr)
rate_lim_arr     = np.array(rate_lim_arr)
valid            = ~np.isnan(J_arr)

# ── Rate-limiting classification array ───────────────────────────────────────
rl_arr = np.where(frac_surface_arr > 0.5, 'surface',
         np.where(frac_oxide_arr   > 0.5, 'oxide',
         np.where(frac_metal_arr   > 0.5, 'metal', 'mixed')))

# ── Regime summary ────────────────────────────────────────────────────────────
n_surface = np.sum((frac_surface_arr > 0.5) & valid)
n_oxide   = np.sum((frac_oxide_arr   > 0.5) & valid)
n_metal   = np.sum((frac_metal_arr   > 0.5) & valid)
n_mixed   = np.sum(
    (frac_surface_arr <= 0.5) & (frac_oxide_arr <= 0.5) &
    (frac_metal_arr   <= 0.5) & valid
)

surf_dom_mask  = (frac_surface_arr > 0.90) & valid
oxide_dom_mask = (frac_oxide_arr   > 0.90) & valid
metal_dom_mask = (frac_metal_arr   > 0.90) & valid
P_surf_dom     = P_RANGE_ARR[surf_dom_mask]
P_oxide_dom    = P_RANGE_ARR[oxide_dom_mask]
P_metal_dom    = P_RANGE_ARR[metal_dom_mask]

# ── Arrhenius pressure selection (no crossover dependence) ───────────────────
P_for_surface_Ea = np.median(P_surf_dom)  if len(P_surf_dom)  > 3 else None
P_for_oxide_Ea   = np.median(P_oxide_dom) if len(P_oxide_dom) > 3 else None
P_for_metal_Ea   = np.median(P_metal_dom) if len(P_metal_dom) > 3 else None

# ── Crossover pressures (diagnostic only) ────────────────────────────────────
def find_crossover(frac_a, frac_b, valid_mask):
    """Find pressure where frac_a crosses frac_b (from above)."""
    vc = valid_mask & ~np.isnan(frac_a) & ~np.isnan(frac_b)
    if np.sum(vc) < 2:
        return None
    diff = frac_a[vc] - frac_b[vc]
    Pv   = P_RANGE_ARR[vc]
    sc   = np.where(np.diff(np.sign(diff)))[0]
    if len(sc) == 0:
        return None
    idx = sc[0]
    if diff[idx] > 0 and diff[idx+1] < 0:
        return Pv[idx]
    return None

P_cross_surf_oxide = find_crossover(frac_surface_arr, frac_oxide_arr, valid)
P_cross_surf_metal = find_crossover(frac_surface_arr, frac_metal_arr, valid)
P_cross_oxide_metal= find_crossover(frac_oxide_arr,   frac_metal_arr, valid)

# ── Print regime summary ──────────────────────────────────────────────────────
print("=" * 60)
print(f"L2+L6 REGIME SUMMARY  (T = {T_op-273.15:.3f}°C, {np.sum(valid)} valid points)")
print("=" * 60)

print(f"\n  Regime breakdown (dominant fraction > 50%)")
print(f"    Surface-limited:  {n_surface:3d} points")
print(f"    Oxide-limited:    {n_oxide:3d} points")
print(f"    Metal-limited:    {n_metal:3d} points")
print(f"    Mixed:            {n_mixed:3d} points")

print(f"\n  Dominant regions (fraction > 90%) — used for Arrhenius")
for label, dom, P_ea in [
    ('Surface', P_surf_dom,  P_for_surface_Ea),
    ('Oxide',   P_oxide_dom, P_for_oxide_Ea),
    ('Metal',   P_metal_dom, P_for_metal_Ea),
]:
    if len(dom) > 3:
        print(f"    {label}-dominant: {len(dom):3d} points  "
              f"P = {dom.min():.1e} – {dom.max():.1e} Pa  "
              f"→ P_for_{label.lower()}_Ea = {P_ea:.2e} Pa")
    else:
        print(f"    {label}-dominant: {len(dom):3d} points  "
              f"(insufficient — {label.lower()} Ea not extractable)")

print(f"\n  Crossover pressures (diagnostic only)")
for desc, P_co in [
    ('Surface → Oxide',  P_cross_surf_oxide),
    ('Surface → Metal',  P_cross_surf_metal),
    ('Oxide   → Metal',  P_cross_oxide_metal),
]:
    val = f'{P_co:.2e} Pa' if P_co is not None else 'not found'
    print(f"    {desc}:  {val}")

print(f"\n  α/β ratio:  {alpha/beta:.3e}")
print(f"  θ range:    {np.nanmin(theta_arr):.4f} → {np.nanmax(theta_arr):.4f}")
print(f"  J range:    {np.nanmin(J_arr):.2e} → {np.nanmax(J_arr):.2e} mol/m²/s")
print("=" * 60)

L2+L6 REGIME SUMMARY  (T = 599.850°C, 40 valid points)

  Regime breakdown (dominant fraction > 50%)
    Surface-limited:   20 points
    Oxide-limited:     19 points
    Metal-limited:      0 points
    Mixed:              1 points

  Dominant regions (fraction > 90%) — used for Arrhenius
    Surface-dominant:  18 points  P = 1.0e-07 – 1.3e-01 Pa  → P_for_surface_Ea = 1.22e-04 Pa
    Oxide-dominant:   0 points  (insufficient — oxide Ea not extractable)
    Metal-dominant:   0 points  (insufficient — metal Ea not extractable)

  Crossover pressures (diagnostic only)
    Surface → Oxide:  1.51e+00 Pa
    Surface → Metal:  3.46e+00 Pa
    Oxide   → Metal:  not found

  α/β ratio:  4.197e-01
  θ range:    0.0000 → 0.9407
  J range:    2.37e-16 → 8.45e-06 mol/m²/s


In [18]:
# =============================================================================
# CELL 4 — FIGURE 1: Core Validation (2×2)
# =============================================================================
"""
Figure 1 validates the L2+L6 model from four angles at operating temperature.
This is the first level with three competing resistances, making it the
canonical figure for the full series-resistance framework.

Panel (A) — Flux vs Pressure
    Question: Does the model produce the correct three-regime pressure
    behaviour?
    The hero plot. Shows J_ss vs P_up on a log-log scale. The curve should
    show three distinct regimes: slope≈1 (surface-limited, low P), slope≈0.5
    (oxide-limited or metal-limited, intermediate/high P), with possible
    transitions between them. Style follows the widget — no fill_between.
    The curve is recolored by regime using thick colored segments with slope
    annotation boxes sitting directly on each segment. The black model curve
    sits underneath, the colored overlay sits on top. Reference lines for
    slope=1 and slope=0.5 are drawn as dashed lines. Net slope annotation
    box in the bottom right corner.

Panel (B) — Arrhenius: apparent Ea per regime
    Question: Does the model extract the correct activation energy in each
    regime, and does it match the known reference values from config?
    Sweeps temperature from T_RANGE[0] to T_RANGE[1]. At each temperature
    all properties are recomputed via helper functions. The solver is called
    at up to three fixed pressures — one per dominant regime — selected in
    Cell 3 as the median of each dominant region (fraction > 90%). A linear
    fit to ln(J) vs 1000/T gives the apparent Ea for each regime. Extracted
    Ea is compared to config references: E_diss (surface), E_D_ox+H_sol_ox
    (oxide), E_D+H_s (metal). Dual x-axis: 1000/T bottom, °C top.

Panel (C) — Resistance Fractions vs Pressure
    Question: Which step controls the rate, and how does the balance shift
    with pressure?
    Plots R_surface/R_total, R_oxide/R_total, and R_metal/R_total as
    percentages vs pressure on a semilog scale — consistent with L1+L6
    and L2a+L6. With three resistances there can be two crossover
    pressures: surface→oxide (S→O) and oxide→metal (O→M). Both are
    marked with vertical lines if present. The 50% threshold line marks
    where each fraction is dominant. If no crossovers exist a text
    annotation states which regime dominates throughout.

Panel (D) — Limit Check: k_diss → ∞ recovers non-L6 result
    Question: As k_diss → ∞, does L2+L6 recover the oxide+metal result
    without surface kinetics?
    Runs the solver at k_diss = 1e-3 (fast kinetics limit) and compares
    J_L2+L6 against the analytical series-resistance flux:
    J = (sqrt(P_up) - sqrt(P_down)) / (1/alpha + 1/beta).
    A parity plot shows agreement — perfect match is a diagonal line.
    This validates the model correctly recovers the L2+L5 (no L6) limit
    when surface kinetics is fast. Green box if max error < 1%.
"""

props = dict(boxstyle='round', facecolor='wheat', alpha=0.9)

fig, axes = plt.subplots(2, 2, figsize=STYLE['figsize'])
fig.suptitle(
    f'L2+L6: Perfect Oxide + Perfect Metal + Surface Kinetics — Core Validation\n'
    f'{OXIDE_KEY} + {METAL_KEY}  |  T = {T_op-273.15:.3f}°C  |  '
    f'L_ox = {L_ox*1e6:.3f} μm  |  L_m = {L_m*1e3:.3f} mm',
    fontsize=STYLE['fontsize_suptitle']-1, fontweight='bold', y=0.98
)

# ─────────────────────────────────────────────────────────────────────────────
# (A) FLUX vs PRESSURE — three regimes
# ─────────────────────────────────────────────────────────────────────────────
ax1 = axes[0, 0]

ax1.loglog(
    P_RANGE_ARR[valid], J_arr[valid],
    color='black', lw=STYLE['linewidth'], label='L2+L6 Model'
)

if np.any(valid):
    P_ref1  = P_RANGE_ARR[valid][0]
    J_ref1  = J_arr[valid][0]
    ax1.loglog(
        P_RANGE_ARR[valid],
        J_ref1 * (P_RANGE_ARR[valid] / P_ref1) ** 1.0,
        color=CURVE_STYLES['slope_1']['color'],
        ls=CURVE_STYLES['slope_1']['ls'],
        lw=CURVE_STYLES['slope_1']['lw'],
        alpha=CURVE_STYLES['slope_1']['alpha'],
        label='Slope = 1 (surface)'
    )
    P_ref05 = P_RANGE_ARR[valid][-1]
    J_ref05 = J_arr[valid][-1]
    ax1.loglog(
        P_RANGE_ARR[valid],
        J_ref05 * (P_RANGE_ARR[valid] / P_ref05) ** 0.5,
        color=CURVE_STYLES['slope_05']['color'],
        ls=CURVE_STYLES['slope_05']['ls'],
        lw=CURVE_STYLES['slope_05']['lw'],
        alpha=CURVE_STYLES['slope_05']['alpha'],
        label='Slope = 0.5 (diffusion)'
    )

# Colored regime segments + slope annotation boxes
region_styles = [
    ('surface', CURVE_STYLES['surface_region']['color'], 'Surface-limited'),
    ('oxide',   CURVE_STYLES['oxide_region']['color'],   'Oxide-limited'),
    ('metal',   CURVE_STYLES['metal_region']['color'],   'Metal-limited'),
    ('mixed',   CURVE_STYLES['mixed_region']['color'],   'Mixed'),
]

for regime, color, label in region_styles:
    mask = (rl_arr == regime) & valid
    if not np.any(mask):
        continue
    idxs = np.where(mask)[0]
    for _, g in groupby(enumerate(idxs), lambda x: x[0] - x[1]):
        group = list(map(itemgetter(1), g))
        if len(group) < 3:
            continue
        P_seg = P_RANGE_ARR[group]
        J_seg = J_arr[group]
        ax1.loglog(P_seg, J_seg, color=color, lw=4, alpha=0.7)
        slope_seg, *_ = stats.linregress(
            np.log10(P_seg), np.log10(np.abs(J_seg))
        )
        mid = len(group) // 2
        ax1.text(
            P_seg[mid], J_seg[mid],
            f'{label}\nSlope={slope_seg:.2f}',
            color=color,
            fontsize=STYLE['fontsize_annotation'],
            fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
        )

log_P_v   = np.log10(P_RANGE_ARR[valid])
log_J_v   = np.log10(np.abs(J_arr[valid]))
slope_net, *_ = stats.linregress(log_P_v, log_J_v)

ax1.text(
    0.98, 0.02, f'Net slope = {slope_net:.2f}',
    transform=ax1.transAxes, ha='right', va='bottom',
    fontsize=STYLE['fontsize_annotation'], fontweight='bold',
    bbox=dict(boxstyle='square', fc='wheat', ec='gray', alpha=1)
)

ax1.set_xlabel('Upstream Pressure $P_{up}$ (Pa)', fontsize=STYLE['fontsize_axis'])
ax1.set_ylabel('Flux $J_{ss}$ (mol/m²/s)',         fontsize=STYLE['fontsize_axis'])
ax1.set_title('(A) Flux vs Pressure',               fontsize=STYLE['fontsize_title'])
ax1.legend(fontsize=STYLE['fontsize_legend']-1, loc='upper left')
ax1.grid(True, which='both', alpha=STYLE['grid_alpha'])
ax1.tick_params(labelsize=STYLE['fontsize_tick'])

# ─────────────────────────────────────────────────────────────────────────────
# (B) ARRHENIUS — apparent Ea per regime, classified by regime purity
# ─────────────────────────────────────────────────────────────────────────────
ax2 = axes[0, 1]

temperatures_K = np.linspace(T_RANGE[0], T_RANGE[1], N_T)
inv_T          = 1000.0 / temperatures_K
arrhenius_results = {}

# ── Regime classification helper ─────────────────────────────────────────────
# pure_thresh is 0.99, not 0.90: a point that is 90% surface-controlled still
# carries 10% of a path whose activation energy is ~234 kJ/mol against the
# surface's ~58, so admitting it biases the extracted E_diss upward by ~10
# kJ/mol. Measured: at 0.90 the fit returns 67.6, at 0.99 it returns 59.2
# against a configured 57.95, and the oxide fit moves 217 -> 231 against 234.
def classify_regime(fs, fo, fm, pure_thresh=0.99, mixed_thresh=0.50):
    """
    Classify a single T point into one of seven regime categories
    based on resistance fractions at that temperature and pressure.
    pure_thresh  : fraction above which a step is called 'pure'
    mixed_thresh : fraction above which a step is called 'dominant'
    """
    if fs >= pure_thresh:    return 'pure_surface'
    elif fo >= pure_thresh:  return 'pure_oxide'
    elif fm >= pure_thresh:  return 'pure_metal'
    elif fs >= mixed_thresh: return 'mixed_surface'
    elif fo >= mixed_thresh: return 'mixed_oxide'
    elif fm >= mixed_thresh: return 'mixed_metal'
    else:                    return 'mixed'

# ── Per-class plot styles ─────────────────────────────────────────────────────
regime_class_styles = {
    'pure_surface':  {'color': CURVE_STYLES['surface_region']['color'],
                      'marker': 'o', 'ms': 8,  'alpha': 1.0},
    'mixed_surface': {'color': CURVE_STYLES['surface_region']['color'],
                      'marker': 'o', 'ms': 5,  'alpha': 0.35},
    'pure_oxide':    {'color': CURVE_STYLES['oxide_region']['color'],
                      'marker': 's', 'ms': 8,  'alpha': 1.0},
    'mixed_oxide':   {'color': CURVE_STYLES['oxide_region']['color'],
                      'marker': 's', 'ms': 5,  'alpha': 0.35},
    'pure_metal':    {'color': CURVE_STYLES['metal_region']['color'],
                      'marker': '^', 'ms': 8,  'alpha': 1.0},
    'mixed_metal':   {'color': CURVE_STYLES['metal_region']['color'],
                      'marker': '^', 'ms': 5,  'alpha': 0.35},
    'mixed':         {'color': CURVE_STYLES['mixed_region']['color'],
                      'marker': 'x', 'ms': 6,  'alpha': 0.5},
}

regime_Ea_cases = [
    ('surface', P_for_surface_Ea, CURVE_STYLES['surface_region']['color'], 'o'),
    ('oxide',   P_for_oxide_Ea,   CURVE_STYLES['oxide_region']['color'],   's'),
    ('metal',   P_for_metal_Ea,   CURVE_STYLES['metal_region']['color'],   '^'),
]

plotted_class_labels = set()

for label, P_fixed, color, marker in regime_Ea_cases:
    if P_fixed is None:
        arrhenius_results[label] = None
        continue

    J_row     = []
    fs_row    = []
    fo_row    = []
    fm_row    = []
    class_row = []

    for T_K in temperatures_K:
        try:
            r = solve_steady_state_flux_direct(
                P_fixed, P_down, L_m,
                get_k_diss_at_T(T_K), get_K_eq_at_T(T_K),
                get_D_ox_at_T(T_K),   get_K_ox_at_T(T_K), L_ox,
                get_D_m_at_T(T_K),    get_K_s_m_at_T(T_K),
            )
            J_row.append(r['J_ss'])
            fs = r['resistances']['fraction_surface']
            fo = r['resistances']['fraction_oxide']
            fm = r['resistances']['fraction_metal']
            fs_row.append(fs)
            fo_row.append(fo)
            fm_row.append(fm)
            class_row.append(classify_regime(fs, fo, fm))
        except Exception:
            J_row.append(np.nan)
            fs_row.append(np.nan)
            fo_row.append(np.nan)
            fm_row.append(np.nan)
            class_row.append('mixed')

    J_arr_T   = np.array(J_row)
    class_arr = np.array(class_row)

    # Pure mask — only points classified as pure target regime
    pure = (class_arr == f'pure_{label}') & ~np.isnan(J_arr_T) & (J_arr_T > 0)

    # Plot all classified points with distinct size/alpha/marker
    for cls, sty in regime_class_styles.items():
        mask = (class_arr == cls) & ~np.isnan(J_arr_T) & (J_arr_T > 0)
        if not np.any(mask):
            continue
        cls_label = cls.replace('_', ' ').capitalize()
        ax2.semilogy(
            inv_T[mask], J_arr_T[mask],
            ls='none',
            color=sty['color'],
            marker=sty['marker'],
            ms=sty['ms'],
            alpha=sty['alpha'],
            label=cls_label if cls_label not in plotted_class_labels
                  else '_nolegend_'
        )
        plotted_class_labels.add(cls_label)

    # Arrhenius fit on pure points only
    if np.sum(pure) < 3:
        print(f"  ⚠  {label}: only {np.sum(pure)} pure T points "
              f"(fraction > 90%) — fit skipped")
        arrhenius_results[label] = None
        continue

    slope, intercept, r_val, *_ = stats.linregress(
        inv_T[pure], np.log(J_arr_T[pure])
    )
    E_a = -slope * R * 1000

    arrhenius_results[label] = {
        'J':            J_arr_T,
        'valid':        pure,
        'slope':        slope,
        'intercept':    intercept,
        'r_sq':         r_val**2,
        'E_a':          E_a,
        'P_fixed':      P_fixed,
        'color':        color,
        'n_pure':       int(np.sum(pure)),
        'T_pure_range': (temperatures_K[pure].min() - 273.15,
                         temperatures_K[pure].max() - 273.15),
        'class_arr':    class_arr,
    }

    # Fitted line through pure points only
    ax2.semilogy(
        inv_T[pure],
        np.exp(slope * inv_T[pure] + intercept),
        color=color, ls='--', lw=2.0, alpha=0.9,
        label=f'Fit {label}  (pure pts)'
    )

# ── Dual x-axis ───────────────────────────────────────────────────────────────
ax2_top = ax2.twiny()
ax2_top.set_xlim(ax2.get_xlim())
T_ticks_C = np.array([200, 300, 400, 500, 600, 700, 800, 900])
T_ticks_K = T_ticks_C + 273.15
in_range  = (T_ticks_K >= T_RANGE[0]) & (T_ticks_K <= T_RANGE[1])
if np.any(in_range):
    ax2_top.set_xticks(1000.0 / T_ticks_K[in_range])
    ax2_top.set_xticklabels([f'{t}' for t in T_ticks_C[in_range]])
ax2_top.set_xlabel('Temperature (°C)', fontsize=STYLE['fontsize_axis'])
ax2_top.tick_params(labelsize=STYLE['fontsize_tick'])

# ── Annotation box ────────────────────────────────────────────────────────────
E_refs    = {'surface': E_REF_SURFACE, 'oxide': E_REF_OXIDE, 'metal': E_REF_METAL}
ref_names = {'surface': 'E_diss', 'oxide': 'E_D_ox+H_sol', 'metal': 'E_D+H_s'}
annot     = []

for lbl in ('surface', 'oxide', 'metal'):
    res = arrhenius_results.get(lbl)
    E_r = E_refs[lbl]
    rn  = ref_names[lbl]
    if res is None:
        annot.append(f'{lbl.capitalize()}: not extractable')
        annot.append(f'  Expected ({rn}): {E_r/1000:.3f} kJ/mol')
    else:
        match = abs(res['E_a'] - E_r) < 3000
        annot.append(f'{lbl.capitalize()}-limited:')
        annot.append(f'  Extracted: {res["E_a"]/1000:.3f} kJ/mol')
        annot.append(f'  Expected ({rn}): {E_r/1000:.3f} kJ/mol')
        annot.append(f'  R²={res["r_sq"]:.4f}  {"✓" if match else "⚠"}')
        annot.append(f'  T range: {res["T_pure_range"][0]:.3f}–'
                     f'{res["T_pure_range"][1]:.3f}°C  '
                     f'({res["n_pure"]} pure pts)')
    annot.append('')

ax2.text(
    0.97, 0.97, '\n'.join(annot).rstrip(),
    transform=ax2.transAxes, fontsize=STYLE['fontsize_annotation']-1,
    va='top', ha='right',
    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9)
)

ax2.set_xlabel('1000/T (K⁻¹)',             fontsize=STYLE['fontsize_axis'])
ax2.set_ylabel('Flux $J_{ss}$ (mol/m²/s)', fontsize=STYLE['fontsize_axis'])
ax2.set_title('(B) Arrhenius: Apparent $E_a$ per Regime',
              fontsize=STYLE['fontsize_title'])
ax2.legend(fontsize=STYLE['fontsize_legend']-2, loc='lower left',
           ncol=2, title='Symbol size = purity')
ax2.grid(True, which='both', alpha=STYLE['grid_alpha'])
ax2.tick_params(labelsize=STYLE['fontsize_tick'])

# ─────────────────────────────────────────────────────────────────────────────
# (C) RESISTANCE FRACTIONS vs PRESSURE
# ─────────────────────────────────────────────────────────────────────────────
ax3 = axes[1, 0]

ax3.semilogx(
    P_RANGE_ARR[valid], frac_surface_arr[valid] * 100,
    color=CURVE_STYLES['surface_region']['color'],
    lw=STYLE['linewidth'], label='Surface (dissociation)'
)
ax3.semilogx(
    P_RANGE_ARR[valid], frac_oxide_arr[valid] * 100,
    color=CURVE_STYLES['oxide_region']['color'],
    lw=STYLE['linewidth'], label='Oxide (diffusion)'
)
ax3.semilogx(
    P_RANGE_ARR[valid], frac_metal_arr[valid] * 100,
    color=CURVE_STYLES['metal_region']['color'],
    lw=STYLE['linewidth'], label='Metal (diffusion)'
)
ax3.axhline(
    50,
    color=CURVE_STYLES['threshold']['color'],
    ls=CURVE_STYLES['threshold']['ls'],
    lw=CURVE_STYLES['threshold']['lw'],
    alpha=CURVE_STYLES['threshold']['alpha'],
    label='50% threshold'
)

# Crossover annotations — up to two for three-resistance system
if P_cross_surf_oxide is not None:
    ax3.axvline(P_cross_surf_oxide, color='gray', ls=':', lw=1.5, alpha=0.7)
    ax3.text(
        P_cross_surf_oxide * 1.5, 53,
        f'S→O\n{P_cross_surf_oxide:.1e} Pa',
        fontsize=STYLE['fontsize_annotation']-2, color='gray'
    )
if P_cross_oxide_metal is not None:
    ax3.axvline(P_cross_oxide_metal, color='gray', ls=':', lw=1.5, alpha=0.7)
    ax3.text(
        P_cross_oxide_metal * 1.5, 53,
        f'O→M\n{P_cross_oxide_metal:.1e} Pa',
        fontsize=STYLE['fontsize_annotation']-2, color='gray'
    )

if P_cross_surf_oxide is None and P_cross_oxide_metal is None:
    dominant = ('Surface' if n_surface >= n_oxide and n_surface >= n_metal
                else 'Oxide' if n_oxide >= n_metal else 'Metal')
    ax3.text(
        0.05, 0.50,
        f'No crossover in range\n{dominant}-limited throughout',
        transform=ax3.transAxes,
        fontsize=STYLE['fontsize_annotation'],
        va='center',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9)
    )

ax3.set_xlabel('Upstream Pressure $P_{up}$ (Pa)', fontsize=STYLE['fontsize_axis'])
ax3.set_ylabel('Resistance Fraction (%)',          fontsize=STYLE['fontsize_axis'])
ax3.set_title('(C) Rate-Limiting Step Analysis',   fontsize=STYLE['fontsize_title'])
ax3.set_ylim(0, 105)
ax3.legend(fontsize=STYLE['fontsize_legend'], loc='center right')
ax3.grid(True, alpha=STYLE['grid_alpha'])
ax3.tick_params(labelsize=STYLE['fontsize_tick'])

# ─────────────────────────────────────────────────────────────────────────────
# (D) LIMIT CHECK — k_diss → ∞ recovers series resistance result
# ─────────────────────────────────────────────────────────────────────────────
ax4 = axes[1, 1]

# Series resistance reference (no surface kinetics)
# J = (sqrt(P_up) - sqrt(P_down)) / (1/alpha + 1/beta)
J_series = ((np.sqrt(P_RANGE_ARR) - np.sqrt(P_down)) /
            (1.0/alpha + 1.0/beta))

# L2+L6 at high k_diss
J_high_k = []
for P_up in P_RANGE_ARR:
    try:
        r = solve_steady_state_flux_direct(
            P_up, P_down, L_m,
            1e-3, K_eq,
            D_ox, K_ox, L_ox,
            D_m, K_s_m
        )
        J_high_k.append(r['J_ss'])
    except Exception:
        J_high_k.append(np.nan)
J_high_k = np.array(J_high_k)

valid_parity = ~np.isnan(J_high_k) & (J_series > 0) & (J_high_k > 0)

ax4.loglog(
    J_series[valid_parity], J_high_k[valid_parity],
    'o', color=CURVE_STYLES['L2_L6']['color'],
    markersize=6, alpha=0.8,
    label='L2+L6  (k_diss = 1e-3)'
)

J_min = min(J_series[valid_parity].min(), J_high_k[valid_parity].min()) * 0.5
J_max = max(J_series[valid_parity].max(), J_high_k[valid_parity].max()) * 2.0
ax4.loglog(
    [J_min, J_max], [J_min, J_max],
    color=CURVE_STYLES['parity']['color'],
    ls=CURVE_STYLES['parity']['ls'],
    lw=CURVE_STYLES['parity']['lw'],
    alpha=CURVE_STYLES['parity']['alpha'],
    label='Perfect agreement'
)

rel_errors = (np.abs(J_high_k[valid_parity] - J_series[valid_parity])
              / J_series[valid_parity] * 100)
max_err   = rel_errors.max()
mean_err  = rel_errors.mean()
box_color = 'lightgreen' if max_err < 1.0 else 'lightyellow'

ax4.text(
    0.05, 0.95,
    f'k_diss = 1e-3\n'
    f'Max error  = {max_err:.2e}%\n'
    f'Mean error = {mean_err:.2e}%\n'
    f'{"✓ VALIDATED" if max_err < 1.0 else "⚠ CHECK k_diss"}',
    transform=ax4.transAxes, fontsize=STYLE['fontsize_annotation'],
    va='top',
    bbox=dict(boxstyle='round', facecolor=box_color, alpha=0.9)
)

ax4.set_xlabel('Series-Resistance Flux $J_{ref}$ (mol/m²/s)', fontsize=STYLE['fontsize_axis'])
ax4.set_ylabel('L2+L6 Flux $J_{L2+L6}$ (mol/m²/s)',           fontsize=STYLE['fontsize_axis'])
ax4.set_title(r'(D) Limit Check: $k_{diss} \to \infty$ → Series Resistance',
              fontsize=STYLE['fontsize_title'])
ax4.set_aspect('equal', adjustable='box')
ax4.legend(fontsize=STYLE['fontsize_legend'], loc='lower right')
ax4.grid(True, which='both', alpha=STYLE['grid_alpha'])
ax4.tick_params(labelsize=STYLE['fontsize_tick'])

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
save_figure(fig, "L2L6_core_validation", FIGS)

print("\n" + "=" * 60)
print("L2+L6 FIGURE 1 — VALIDATION SUMMARY")
print("=" * 60)
print(f"  Net log-log slope:       {slope_net:.4f}")
print(f"  θ range:                 "
      f"{np.nanmin(theta_arr):.4f} → {np.nanmax(theta_arr):.4f}")
print(f"  Series limit error:      {max_err:.2e}%  "
      f"({'✓' if max_err < 1.0 else '⚠'})")
print(f"\n  Arrhenius results:")
for lbl in ('surface', 'oxide', 'metal'):
    res = arrhenius_results.get(lbl)
    E_r = E_refs[lbl]
    if res is None:
        print(f"    {lbl.capitalize()}: not extractable")
    else:
        match = abs(res['E_a'] - E_r) < 3000
        print(f"    {lbl.capitalize()}: extracted={res['E_a']/1000:.3f} "
              f"expected={E_r/1000:.3f} kJ/mol  "
              f"{'✓' if match else '⚠'}")
print("=" * 60)

/var/folders/d4/mt_nqf_x42vgxs9t9mw1l59m0000gn/T/ipykernel_20394/449845170.py:487: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



L2+L6 FIGURE 1 — VALIDATION SUMMARY
  Net log-log slope:       0.7564
  θ range:                 0.0000 → 0.9407
  Series limit error:      4.45e-01%  (✓)

  Arrhenius results:
    Surface: extracted=59.172 expected=57.950 kJ/mol  ✓
    Oxide: not extractable
    Metal: not extractable


In [20]:
# =============================================================================
# CELL 6 — DATAFRAME OUTPUT
# =============================================================================
df = pd.DataFrame(rows)

df_display = df.copy()
for col in df_display.columns:
    if col in ('Rate-Limiting', 'Error'):
        continue
    if df_display[col].dtype == float:
        if 'fraction' in col.lower():
            df_display[col] = df_display[col].round(2)
        elif 'theta' in col.lower():
            df_display[col] = df_display[col].round(6)
        else:
            df_display[col] = df_display[col].apply(
                lambda x: f'{x:.4e}' if pd.notna(x) else 'NaN'
            )

n_surface_df = (df['Rate-Limiting'] == 'SURFACE').sum()
n_oxide_df   = (df['Rate-Limiting'] == 'OXIDE').sum()
n_metal_df   = (df['Rate-Limiting'] == 'METAL').sum()
n_mixed_df   = (df['Rate-Limiting'] == 'MIXED').sum()
n_error_df   = (df['Rate-Limiting'] == 'ERROR').sum()

print("=" * 60)
print("L2+L6 RESULTS DATAFRAME")
print("=" * 60)
print(f"  Oxide:       {OXIDE_KEY}")
print(f"  Metal:       {METAL_KEY}")
print(f"  T:           {T_op-273.15:.3f}°C  ({T_op} K)")
print(f"  L_ox:        {L_ox*1e6:.3f} μm")
print(f"  L_m:         {L_m*1e3:.3f} mm")
print(f"  k_diss:      {k_diss:.3e}  mol/m²/s/Pa")
print(f"  K_eq:        {K_eq:.3e}  Pa⁻¹")
print(f"  α:           {alpha:.3e}  mol/m²/s/Pa^0.5")
print(f"  β:           {beta:.3e}  mol/m²/s/Pa^0.5")
print(f"  α/β:         {alpha/beta:.3e}")
print(f"\n  Total rows:        {len(df)}")
print(f"  Surface-limited:   {n_surface_df}")
print(f"  Oxide-limited:     {n_oxide_df}")
print(f"  Metal-limited:     {n_metal_df}")
print(f"  Mixed:             {n_mixed_df}")
if n_error_df > 0:
    print(f"  Errors:            {n_error_df}  ⚠")
print("=" * 60)

display(df_display)

L2+L6 RESULTS DATAFRAME
  Oxide:       Cr2O3_sample4
  Metal:       metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985
  T:           599.850°C  (873 K)
  L_ox:        1.000 μm
  L_m:         1.000 mm
  k_diss:      2.370e-09  mol/m²/s/Pa
  K_eq:        2.799e-05  Pa⁻¹
  α:           4.002e-09  mol/m²/s/Pa^0.5
  β:           9.534e-09  mol/m²/s/Pa^0.5
  α/β:         4.197e-01

  Total rows:        40
  Surface-limited:   20
  Oxide-limited:     19
  Metal-limited:     0
  Mixed:             1


,P_up (Pa),P_int (Pa),J_ss (mol/m²/s),θ_surface,α_oxide,β_metal,R_surface,R_oxide,R_metal,R_total,fraction_surface (%),fraction_oxide (%),fraction_metal (%),Rate-Limiting
0,1.0000e-07,6.1800e-16,2.3702e-16,4.4487e-10,4.0017e-09,9.5343e-09,4.2191e+15,2.4990e+08,1.0488e+08,4.2191e+15,100.00,0.00,0.00,SURFACE
1,2.2855e-07,3.2280e-15,5.4170e-16,1.0167e-09,4.0017e-09,9.5343e-09,1.8460e+15,2.4990e+08,1.0488e+08,1.8460e+15,100.00,0.00,0.00,SURFACE
2,5.2233e-07,1.6861e-14,1.2380e-15,2.3237e-09,4.0017e-09,9.5343e-09,8.0773e+14,2.4990e+08,1.0488e+08,8.0773e+14,100.00,0.00,0.00,SURFACE
3,1.1938e-06,8.8071e-14,2.8295e-15,5.3108e-09,4.0017e-09,9.5343e-09,3.5342e+14,2.4990e+08,1.0488e+08,3.5342e+14,100.00,0.00,0.00,SURFACE
4,2.7283e-06,4.6002e-13,6.4667e-15,1.2138e-08,4.0017e-09,9.5343e-09,1.5464e+14,2.4990e+08,1.0488e+08,1.5464e+14,100.00,0.00,0.00,SURFACE
5,6.2355e-06,2.4028e-12,1.4779e-14,2.7740e-08,4.0017e-09,9.5343e-09,6.7662e+13,2.4990e+08,1.0488e+08,6.7662e+13,100.00,0.00,0.00,SURFACE
6,1.4251e-05,1.2551e-11,3.3777e-14,6.3399e-08,4.0017e-09,9.5343e-09,2.9605e+13,2.4990e+08,1.0488e+08,2.9606e+13,100.00,0.00,0.00,SURFACE
7,3.2570e-05,6.5555e-11,7.7196e-14,1.4489e-07,4.0017e-09,9.5343e-09,1.2954e+13,2.4990e+08,1.0488e+08,1.2954e+13,100.00,0.00,0.00,SURFACE
8,7.4438e-05,3.4240e-10,1.7642e-13,3.3114e-07,4.0017e-09,9.5343e-09,5.6679e+12,2.4990e+08,1.0488e+08,5.6683e+12,99.99,0.00,0.00,SURFACE
9,1.7013e-04,1.7882e-09,4.0318e-13,7.5675e-07,4.0017e-09,9.5343e-09,2.4800e+12,2.4990e+08,1.0488e+08,2.4803e+12,99.99,0.01,0.00,SURFACE


## Level 3+ L6: Surface Kinetics with Mixed Regimes

In [21]:
# =============================================================================
# CELL 1 — SETUP
# Run once. All imports and helper functions.
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import brentq
import pandas as pd
from itertools import groupby
from operator import itemgetter
from IPython.display import display

from calculations.surface_kinetics import (
    calculate_mixed_defect_flux_L6,
    calculate_path_flux_L6,
    solve_steady_state_flux_direct,
)
from calculations.config.model_config import (
    R, F,
    METALS, OXIDES, CONDITIONS, SUGGESTED_RANGES_LEVEL5,
    PLOT_STYLE as STYLE,
    CURVE_STYLES,
    apply_style,
    build_simulation_config,
)

# ── Simulation config ─────────────────────────────────────────────────────────
SIM       = build_simulation_config()

METAL_KEY  = SIM['metal_name']
OXIDE_KEY  = SIM['oxide_name']
METAL      = METALS[METAL_KEY]
OXIDE      = OXIDES[OXIDE_KEY]
SK_OX      = OXIDE['surface_kinetics']     # oxide surface kinetics
SK_M       = METAL['surface_kinetics']     # metal surface kinetics (pinhole)

# ── Pressure sweep ────────────────────────────────────────────────────────────
P_RANGE_ARR = np.logspace(
    np.log10(CONDITIONS['P_range'][0]),
    np.log10(CONDITIONS['P_range'][1]),
    CONDITIONS['n_points']
)
P_DOWN      = SIM['P_downstream']
T_RANGE     = CONDITIONS['T_range']
N_T         = CONDITIONS['n_points']

# ── Arrhenius helper functions — oxide ───────────────────────────────────────
def get_k_diss_at_T(T_K):
    return SK_OX['k_diss_ref'] * np.exp(
        (-SK_OX['E_diss'] / R) * (1/T_K - 1/SK_OX['T_ref'])
    )

def get_K_eq_at_T(T_K):
    return SK_OX['K_eq_ref'] * np.exp(
        (-SK_OX['H_eq'] / R) * (1/T_K - 1/SK_OX['T_ref'])
    )

def get_D_ox_at_T(T_K):
    return OXIDE['D_ox_ref'] * np.exp(
        (-OXIDE['E_D_ox'] / R) * (1/T_K - 1/OXIDE['T_ref'])
    )

def get_K_ox_at_T(T_K):
    return OXIDE['K_ox_ref'] * np.exp(
        (-OXIDE['H_sol_ox'] / R) * (1/T_K - 1/OXIDE['T_ref'])
    )

# ── Arrhenius helper functions — metal ───────────────────────────────────────
def get_D_m_at_T(T_K):
    return METAL['D_ref'] * np.exp(
        (-METAL['E_D'] / R) * (1/T_K - 1/METAL['T_ref'])
    )

def get_K_s_m_at_T(T_K):
    return METAL['K_s_ref'] * np.exp(
        (-METAL['H_s'] / R) * (1/T_K - 1/METAL['T_ref'])
    )

def get_k_diss_metal_at_T(T_K):
    """Metal surface k_diss — used for pinhole paths when not Sieverts limit."""
    return SK_M['k_diss_metal_ref'] * np.exp(
        (-SK_M['E_diss_metal'] / R) * (1/T_K - 1/SK_M['T_ref'])
    )

def get_K_eq_metal_at_T(T_K):
    """Metal surface K_eq — used for pinhole paths when not Sieverts limit."""
    return SK_M['K_eq_metal_ref'] * np.exp(
        (-SK_M['H_eq_metal'] / R) * (1/T_K - 1/SK_M['T_ref'])
    )

# ── Known reference activation energies ──────────────────────────────────────
E_REF_SURFACE = SK_OX['E_diss']
E_REF_OXIDE   = OXIDE['E_D_ox'] + OXIDE['H_sol_ox']
E_REF_METAL   = METAL['E_D']    + METAL['H_s']

print("=" * 60)
print("SETUP COMPLETE")
print("=" * 60)
print(f"  Metal:           {METAL_KEY}")
print(f"  Oxide:           {OXIDE_KEY}")
print(f"  E_ref surface:   {E_REF_SURFACE/1000:.3f} kJ/mol  (E_diss)")
print(f"  E_ref oxide:     {E_REF_OXIDE/1000:.3f} kJ/mol  (E_D_ox + H_sol_ox)")
print(f"  E_ref metal:     {E_REF_METAL/1000:.3f} kJ/mol  (E_D + H_s)")
print(f"  P range:         {P_RANGE_ARR[0]:.0e} – {P_RANGE_ARR[-1]:.0e} Pa")
print(f"  T range:         {T_RANGE[0]} – {T_RANGE[1]} K")

SETUP COMPLETE
  Metal:           metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985
  Oxide:           Cr2O3_sample4
  E_ref surface:   57.950 kJ/mol  (E_diss)
  E_ref oxide:     234.000 kJ/mol  (E_D_ox + H_sol_ox)
  E_ref metal:     79.400 kJ/mol  (E_D + H_s)
  P range:         1e-07 – 1e+07 Pa
  T range:         573 – 1273 K


In [22]:
# =============================================================================
# CELL 2 — PARAMETERS
# All physics values at operating temperature, pulled from config.
# Change values in config and rerun from this cell downward.
# =============================================================================

# ── Operating conditions ──────────────────────────────────────────────────────
T_op   = SIM['T_operating']
L_ox   = SIM['L_oxide']
L_m    = SIM['L_metal']
P_down = SIM['P_downstream']

# ── Surface kinetics at T_op — oxide surface ──────────────────────────────────
k_diss   = get_k_diss_at_T(T_op)
K_eq     = get_K_eq_at_T(T_op)
k_recomb = k_diss / K_eq

# ── Oxide transport at T_op ───────────────────────────────────────────────────
D_ox = get_D_ox_at_T(T_op)
K_ox = get_K_ox_at_T(T_op)

# ── Metal transport at T_op ───────────────────────────────────────────────────
D_m   = get_D_m_at_T(T_op)
K_s_m = get_K_s_m_at_T(T_op)

# ── Metal surface kinetics at T_op — for pinhole paths ───────────────────────
k_diss_metal = get_k_diss_metal_at_T(T_op)
K_eq_metal   = get_K_eq_metal_at_T(T_op)

# ── Derived permeances ────────────────────────────────────────────────────────
alpha_intact = D_ox * K_ox / L_ox
beta         = D_m  * K_s_m / L_m

# ── Defect configuration — from SIM (built by build_simulation_config) ────────
defect_config       = SIM['defect_config']
use_sieverts_pinhole = SIM['oxide_defects'].get('use_sieverts_pinhole', False)

# Pinhole kinetics: None triggers Sieverts limit inside the solver
k_diss_metal_arg = None if use_sieverts_pinhole else k_diss_metal
K_eq_metal_arg   = None if use_sieverts_pinhole else K_eq_metal

# ── Validation sweep parameters — from config ─────────────────────────────────
f_compare       = CONDITIONS['defect_fractions_compare']   # panel (A) sweep
f_defect_min    = CONDITIONS['defect_fraction_range'][0]               # panel (B) sweep
f_defect_max    = CONDITIONS['defect_fraction_range'][1]
n_defect_points = CONDITIONS['n_points']
f_limit_check   = CONDITIONS['defect_limit_check']       # panel (D) limit

# ── Print verification ────────────────────────────────────────────────────────
print("=" * 60)
print(f"L3+L6 PARAMETERS AT T = {T_op-273.15:.3f}°C  ({T_op} K)")
print("=" * 60)

print(f"\n  Geometry")
print(f"    L_ox     = {L_ox:.3e}  m")
print(f"    L_m      = {L_m:.3e}  m")
print(f"    P_down   = {P_down:.1e}  Pa")

print(f"\n  Oxide surface kinetics  [config: oxide surface_kinetics]")
print(f"    k_diss   = {k_diss:.3e}  mol/m²/s/Pa")
print(f"    K_eq     = {K_eq:.3e}  Pa⁻¹")
print(f"    k_recomb = {k_recomb:.3e}  mol/m²/s")

print(f"\n  Oxide transport  [config: Arrhenius]")
print(f"    D_ox     = {D_ox:.3e}  m²/s")
print(f"    K_ox     = {K_ox:.3e}  mol/m³/Pa^0.5")

print(f"\n  Metal transport  [config: Arrhenius]")
print(f"    D_m      = {D_m:.3e}  m²/s")
print(f"    K_s_m    = {K_s_m:.3e}  mol/m³/Pa^0.5")

print(f"\n  Metal surface kinetics  [config: metal surface_kinetics]")
print(f"    k_diss_metal = {k_diss_metal:.3e}  mol/m²/s/Pa")
print(f"    K_eq_metal   = {K_eq_metal:.3e}  Pa⁻¹")
print(f"    Pinhole mode: {'Sieverts limit' if use_sieverts_pinhole else 'Metal surface kinetics'}")

print(f"\n  Derived permeances")
print(f"    α_intact = {alpha_intact:.3e}  mol/m²/s/Pa^0.5")
print(f"    β        = {beta:.3e}  mol/m²/s/Pa^0.5")
print(f"    α/β      = {alpha_intact/beta:.3e}")

print(f"\n  Defect configuration  [config: OXIDE_DEFECTS]")
total_f = sum(v['area_fraction'] for v in defect_config.values())
for path, cfg in defect_config.items():
    extras = ''
    if 'thickness_factor' in cfg:
        extras = f"  γ = {cfg['thickness_factor']}"
        alpha_path = alpha_intact / cfg['thickness_factor']
        extras += f"  → α_crack = {alpha_path:.2e}"
    if 'diffusivity_factor' in cfg:
        extras = f"  δ = {cfg['diffusivity_factor']}"
        alpha_path = cfg['diffusivity_factor'] * alpha_intact
        extras += f"  → α_GB = {alpha_path:.2e}"
    print(f"    {path:<16} f = {cfg['area_fraction']*100:.3f}%{extras}")
print(f"    {'intact':<16} f = {(1-total_f)*100:.3f}%  (= 1 - total defect)")
print(f"    Total defect fraction = {total_f*100:.3f}%")

if total_f > 0.5:
    print(f"\n  ⚠  WARNING: total defect fraction = {total_f*100:.3f}% — unusually high")
if total_f == 0:
    print(f"\n  ⚠  WARNING: all defect fractions are zero — L3+L6 = L2+L6")

print(f"\n  Reference activation energies  [config]")
print(f"    E_diss        = {E_REF_SURFACE/1000:.3f} kJ/mol")
print(f"    E_D_ox+H_sol  = {E_REF_OXIDE/1000:.3f} kJ/mol")
print(f"    E_D+H_s       = {E_REF_METAL/1000:.3f} kJ/mol")

print("\n" + "=" * 60)

L3+L6 PARAMETERS AT T = 599.850°C  (873 K)

  Geometry
    L_ox     = 1.000e-06  m
    L_m      = 1.000e-03  m
    P_down   = 0.0e+00  Pa

  Oxide surface kinetics  [config: oxide surface_kinetics]
    k_diss   = 2.370e-09  mol/m²/s/Pa
    K_eq     = 2.799e-05  Pa⁻¹
    k_recomb = 8.468e-05  mol/m²/s

  Oxide transport  [config: Arrhenius]
    D_ox     = 1.395e-17  m²/s
    K_ox     = 2.869e+02  mol/m³/Pa^0.5

  Metal transport  [config: Arrhenius]
    D_m      = 2.487e-10  m²/s
    K_s_m    = 3.834e-02  mol/m³/Pa^0.5

  Metal surface kinetics  [config: metal surface_kinetics]
    k_diss_metal = 4.611e-07  mol/m²/s/Pa
    K_eq_metal   = 8.212e-04  Pa⁻¹
    Pinhole mode: Metal surface kinetics

  Derived permeances
    α_intact = 4.002e-09  mol/m²/s/Pa^0.5
    β        = 9.534e-09  mol/m²/s/Pa^0.5
    α/β      = 4.197e-01

  Defect configuration  [config: OXIDE_DEFECTS]
    pinhole          f = 1.000%
    crack            f = 0.500%  γ = 0.1  → α_crack = 4.00e-08
    grain_boundary   f 

In [23]:
# =============================================================================
# CELL 3 — COMPUTE
# Single-pass loop over full pressure range.
# Builds all arrays, regime summary, and Arrhenius pressure selection.
# =============================================================================

# ── Single-pass loop ──────────────────────────────────────────────────────────
rows              = []
J_arr             = []
J_intact_arr      = []
J_pinhole_arr     = []
J_crack_arr       = []
J_gb_arr          = []
frac_intact_arr   = []
frac_pinhole_arr  = []
frac_crack_arr    = []
frac_gb_arr       = []
enhancement_arr   = []
dominant_path_arr = []
frac_surface_arr  = []
frac_oxide_arr    = []
frac_metal_arr    = []
rate_lim_arr      = []

for P_up in P_RANGE_ARR:
    try:
        r = calculate_mixed_defect_flux_L6(
            P_up, P_down, L_m,
            k_diss, K_eq,
            D_ox, K_ox, L_ox,
            D_m, K_s_m,
            defect_config,
            k_diss_metal=k_diss_metal_arg,
            K_eq_metal=K_eq_metal_arg,
        )

        J_total      = r['J_total']
        enhancement  = r['enhancement_factor']
        dom_path     = r['dominant_path']
        fb           = r['flux_breakdown']

        # Per-path flux fractions
        frac_intact  = fb.get('intact',         {}).get('fraction_of_total', 0.0)
        frac_pinhole = fb.get('pinhole',         {}).get('fraction_of_total', 0.0)
        frac_crack   = fb.get('crack',           {}).get('fraction_of_total', 0.0)
        frac_gb      = fb.get('grain_boundary',  {}).get('fraction_of_total', 0.0)

        # Per-path absolute flux
        J_intact_v   = fb.get('intact',         {}).get('flux', np.nan)
        J_pinhole_v  = fb.get('pinhole',         {}).get('flux', np.nan)
        J_crack_v    = fb.get('crack',           {}).get('flux', np.nan)
        J_gb_v       = fb.get('grain_boundary',  {}).get('flux', np.nan)

        # Resistance fractions from dominant path
        if dom_path == 'intact':
            res = r['intact_path']['resistances']
        elif dom_path in r.get('defect_paths', {}):
            res = r['defect_paths'][dom_path]['path_result']['resistances']
        else:
            # Mixed — use intact path as proxy
            res = r['intact_path']['resistances']

        fs = res.get('fraction_surface', np.nan)
        fo = res.get('fraction_oxide',   np.nan)
        fm = res.get('fraction_metal',   np.nan)
        rl = r.get('system_rate_limiting', 'mixed')

        J_arr.append(J_total)
        J_intact_arr.append(J_intact_v)
        J_pinhole_arr.append(J_pinhole_v)
        J_crack_arr.append(J_crack_v)
        J_gb_arr.append(J_gb_v)
        frac_intact_arr.append(frac_intact)
        frac_pinhole_arr.append(frac_pinhole)
        frac_crack_arr.append(frac_crack)
        frac_gb_arr.append(frac_gb)
        enhancement_arr.append(enhancement)
        dominant_path_arr.append(dom_path)
        frac_surface_arr.append(fs)
        frac_oxide_arr.append(fo)
        frac_metal_arr.append(fm)
        rate_lim_arr.append(rl)

        rows.append({
            'P_up (Pa)':            P_up,
            'J_total (mol/m²/s)':   J_total,
            'J_intact':             J_intact_v,
            'J_pinhole':            J_pinhole_v,
            'J_crack':              J_crack_v,
            'J_grain_boundary':     J_gb_v,
            'frac_intact (%)':      frac_intact  * 100,
            'frac_pinhole (%)':     frac_pinhole * 100,
            'frac_crack (%)':       frac_crack   * 100,
            'frac_GB (%)':          frac_gb      * 100,
            'enhancement_factor':   enhancement,
            'dominant_path':        dom_path,
            'fraction_surface (%)': fs * 100 if not np.isnan(fs) else np.nan,
            'fraction_oxide (%)':   fo * 100 if not np.isnan(fo) else np.nan,
            'fraction_metal (%)':   fm * 100 if not np.isnan(fm) else np.nan,
            'system_rate_limiting': rl.upper(),
        })

    except Exception as e:
        for lst in [J_arr, J_intact_arr, J_pinhole_arr, J_crack_arr,
                    J_gb_arr, frac_intact_arr, frac_pinhole_arr,
                    frac_crack_arr, frac_gb_arr, enhancement_arr,
                    frac_surface_arr, frac_oxide_arr, frac_metal_arr]:
            lst.append(np.nan)
        dominant_path_arr.append('error')
        rate_lim_arr.append('error')
        rows.append({'P_up (Pa)': P_up, 'system_rate_limiting': 'ERROR',
                     'Error': str(e)})

# ── Convert to arrays ─────────────────────────────────────────────────────────
J_arr            = np.array(J_arr)
J_intact_arr     = np.array(J_intact_arr)
J_pinhole_arr    = np.array(J_pinhole_arr)
J_crack_arr      = np.array(J_crack_arr)
J_gb_arr         = np.array(J_gb_arr)
frac_intact_arr  = np.array(frac_intact_arr)
frac_pinhole_arr = np.array(frac_pinhole_arr)
frac_crack_arr   = np.array(frac_crack_arr)
frac_gb_arr      = np.array(frac_gb_arr)
enhancement_arr  = np.array(enhancement_arr)
frac_surface_arr = np.array(frac_surface_arr)
frac_oxide_arr   = np.array(frac_oxide_arr)
frac_metal_arr   = np.array(frac_metal_arr)
rate_lim_arr     = np.array(rate_lim_arr)
dominant_path_arr= np.array(dominant_path_arr)
valid            = ~np.isnan(J_arr)

# ── Rate-limiting classification (dominant path) ──────────────────────────────
rl_arr = np.where(frac_surface_arr > 0.5, 'surface',
         np.where(frac_oxide_arr   > 0.5, 'oxide',
         np.where(frac_metal_arr   > 0.5, 'metal', 'mixed')))

# ── Regime summary ────────────────────────────────────────────────────────────
n_surface = np.sum((frac_surface_arr > 0.5) & valid)
n_oxide   = np.sum((frac_oxide_arr   > 0.5) & valid)
n_metal   = np.sum((frac_metal_arr   > 0.5) & valid)
n_mixed   = np.sum(
    (frac_surface_arr <= 0.5) & (frac_oxide_arr <= 0.5) &
    (frac_metal_arr   <= 0.5) & valid
)

surf_dom_mask  = (frac_surface_arr > 0.90) & valid
oxide_dom_mask = (frac_oxide_arr   > 0.90) & valid
metal_dom_mask = (frac_metal_arr   > 0.90) & valid
P_surf_dom     = P_RANGE_ARR[surf_dom_mask]
P_oxide_dom    = P_RANGE_ARR[oxide_dom_mask]
P_metal_dom    = P_RANGE_ARR[metal_dom_mask]

# ── Arrhenius pressure selection ──────────────────────────────────────────────
P_for_surface_Ea = np.median(P_surf_dom)  if len(P_surf_dom)  > 3 else None
P_for_oxide_Ea   = np.median(P_oxide_dom) if len(P_oxide_dom) > 3 else None
P_for_metal_Ea   = np.median(P_metal_dom) if len(P_metal_dom) > 3 else None

# ── Dominant path summary ─────────────────────────────────────────────────────
dom_counts = {}
for p in ['intact', 'pinhole', 'crack', 'grain_boundary', 'mixed', 'error']:
    dom_counts[p] = np.sum(dominant_path_arr == p)

# ── Crossover pressures (diagnostic) ─────────────────────────────────────────
def find_crossover(frac_a, frac_b, valid_mask):
    vc   = valid_mask & ~np.isnan(frac_a) & ~np.isnan(frac_b)
    if np.sum(vc) < 2:
        return None
    diff = frac_a[vc] - frac_b[vc]
    Pv   = P_RANGE_ARR[vc]
    sc   = np.where(np.diff(np.sign(diff)))[0]
    if len(sc) == 0:
        return None
    idx = sc[0]
    return Pv[idx] if diff[idx] > 0 else None

P_cross_surf_oxide = find_crossover(frac_surface_arr, frac_oxide_arr, valid)
P_cross_surf_metal = find_crossover(frac_surface_arr, frac_metal_arr, valid)
P_cross_oxide_metal= find_crossover(frac_oxide_arr,   frac_metal_arr, valid)

# ── Print regime summary ──────────────────────────────────────────────────────
print("=" * 60)
print(f"L3+L6 REGIME SUMMARY  (T = {T_op-273.15:.3f}°C, {np.sum(valid)} valid points)")
print("=" * 60)

print(f"\n  Resistance regime (dominant path fractions > 50%)")
print(f"    Surface-limited:  {n_surface:3d} points")
print(f"    Oxide-limited:    {n_oxide:3d} points")
print(f"    Metal-limited:    {n_metal:3d} points")
print(f"    Mixed:            {n_mixed:3d} points")

print(f"\n  Dominant oxide path")
for path, count in dom_counts.items():
    if count > 0:
        print(f"    {path:<18} {count:3d} points")

print(f"\n  Dominant regions (fraction > 90%) — used for Arrhenius")
for label, dom, P_ea in [
    ('Surface', P_surf_dom,  P_for_surface_Ea),
    ('Oxide',   P_oxide_dom, P_for_oxide_Ea),
    ('Metal',   P_metal_dom, P_for_metal_Ea),
]:
    if len(dom) > 3:
        print(f"    {label}-dominant: {len(dom):3d} points  "
              f"P = {dom.min():.1e} – {dom.max():.1e} Pa  "
              f"→ P_for_{label.lower()}_Ea = {P_ea:.2e} Pa")
    else:
        print(f"    {label}-dominant: {len(dom):3d} points  "
              f"(insufficient — {label.lower()} Ea not extractable)")

print(f"\n  Crossover pressures (diagnostic only)")
for desc, P_co in [
    ('Surface → Oxide', P_cross_surf_oxide),
    ('Surface → Metal', P_cross_surf_metal),
    ('Oxide   → Metal', P_cross_oxide_metal),
]:
    val = f'{P_co:.2e} Pa' if P_co is not None else 'not found'
    print(f"    {desc}:  {val}")

print(f"\n  Enhancement factor range:")
valid_enh = enhancement_arr[valid & ~np.isnan(enhancement_arr)]
if len(valid_enh) > 0:
    print(f"    Min = {valid_enh.min():.3f}  Max = {valid_enh.max():.3f}")

print(f"\n  J range:  {np.nanmin(J_arr):.2e} → {np.nanmax(J_arr):.2e} mol/m²/s")
print("=" * 60)

L3+L6 REGIME SUMMARY  (T = 599.850°C, 40 valid points)

  Resistance regime (dominant path fractions > 50%)
    Surface-limited:   20 points
    Oxide-limited:     19 points
    Metal-limited:      0 points
    Mixed:              1 points

  Dominant oxide path
    intact              26 points
    mixed               14 points

  Dominant regions (fraction > 90%) — used for Arrhenius
    Surface-dominant:  18 points  P = 1.0e-07 – 1.3e-01 Pa  → P_for_surface_Ea = 1.22e-04 Pa
    Oxide-dominant:   0 points  (insufficient — oxide Ea not extractable)
    Metal-dominant:   0 points  (insufficient — metal Ea not extractable)

  Crossover pressures (diagnostic only)
    Surface → Oxide:  1.51e+00 Pa
    Surface → Metal:  3.46e+00 Pa
    Oxide   → Metal:  not found

  Enhancement factor range:
    Min = 1.040  Max = 2.935

  J range:  6.96e-16 → 8.79e-06 mol/m²/s


In [24]:
# =============================================================================
# CELL 4 — FIGURE 1: Core Validation (2×2)
# =============================================================================
"""
Figure 1 validates the L3+L6 model from four angles at operating temperature.

Panel (A) — Flux vs Pressure: Defect Fraction Sweep
    Question: How does increasing defect fraction shift the flux curve and
    change the pressure-dependent slope?
    Plots J_total vs P_up for four defect fractions: f=0 (L2+L6 baseline,
    no defects), f=0.1%, f=1%, f=10%. Each curve uses the same defect type
    mix from config, scaled by the fraction multiplier. The f=0 baseline
    uses solve_steady_state_flux_direct directly. Each curve is colored by
    its dominant regime using thick colored segments. As defect fraction
    increases the curve shifts upward and the slope changes — pinholes pull
    it toward slope=0.5 (Sieverts metal), while oxide defects flatten the
    oxide-limited regime. The intact baseline is shown as a dashed line
    for reference on all curves.

Panel (B) — Flux vs Defect Fraction
    Question: Which defect type causes the most enhancement per unit area
    fraction, and where does defect-dominated transport take over?
    Fixed P and T (from CONDITIONS['P_upstream']). Sweeps defect
    fraction from f_defect_min to f_defect_max. Three curves — one per
    defect type (pinhole, crack, grain_boundary) — each tested in isolation
    with the other two set to zero. Log-log plot. The crossover fraction
    where defect flux exceeds intact flux is annotated for each type.
    Mirrors L3 panel (B) from the proposal exactly.

Panel (C) — Resistance Fractions vs Pressure
    Question: Which step controls the rate in the dominant path, and how
    does the balance shift with pressure?
    Consistent style with L1+L6, L2a+L6, L2+L6 — semilogx lines per
    fraction, 50% threshold, crossover vertical lines. Fractions are from
    the dominant path at each pressure point. Two crossover lines (S→O,
    O→M) annotated if present.

Panel (D) — Limit Check: f→0 recovers L2+L6
    Question: When all defect fractions go to zero, does L3+L6 recover
    the L2+L6 result exactly?
    Sets all defect fractions to f_defect_limit_check (≈1e-10) from
    the CONDITIONS defect entries. Computes J_L3+L6(f→0) and compares to
    J_L2+L6 from solve_steady_state_flux_direct. Parity plot — perfect
    agreement is a diagonal line. Green box if max error < 1%.
    Validates hierarchical consistency: L3+L6 collapses to L2+L6
    when the oxide is perfect.
"""

props = dict(boxstyle='round', facecolor='wheat', alpha=0.9)

fig, axes = plt.subplots(2, 2, figsize=STYLE['figsize'])
fig.suptitle(
    f'L3+L6: Defective Oxide + Surface Kinetics — Core Validation\n'
    f'{OXIDE_KEY} + {METAL_KEY}  |  T = {T_op-273.15:.3f}°C  |  '
    f'Total defect = {sum(v["area_fraction"] for v in defect_config.values())*100:.2f}%',
    fontsize=STYLE['fontsize_suptitle']-1, fontweight='bold', y=0.98
)

# ─────────────────────────────────────────────────────────────────────────────
# (A) FLUX vs PRESSURE — defect fraction sweep with regime overlays
# ─────────────────────────────────────────────────────────────────────────────
ax1 = axes[0, 0]

# Local pressure array — fixed length, never captured from outer scope
P_A = np.logspace(
    np.log10(CONDITIONS['P_range'][0]),
    np.log10(CONDITIONS['P_range'][1]),
    CONDITIONS['n_points']
)
print(f"Panel (A): {len(P_A)} pressure points  "
      f"{P_A[0]:.1e} – {P_A[-1]:.1e} Pa")

# ── Regime overlay helper ─────────────────────────────────────────────────────
def plot_regime_overlay(ax, P_arr, J_arr, frac_s, frac_o, frac_m, valid_mask):
    """
    Overlay thick colored segments + slope annotation boxes on a flux curve.
    Identical pattern to L1+L6, L2a+L6, L2+L6 panel (A).
    """
    rl = np.where(frac_s > 0.5, 'surface',
         np.where(frac_o > 0.5, 'oxide',
         np.where(frac_m > 0.5, 'metal', 'mixed')))

    region_styles = [
        ('surface', CURVE_STYLES['surface_region']['color'], 'Surface-limited'),
        ('oxide',   CURVE_STYLES['oxide_region']['color'],   'Oxide-limited'),
        ('metal',   CURVE_STYLES['metal_region']['color'],   'Metal-limited'),
        ('mixed',   CURVE_STYLES['mixed_region']['color'],   'Mixed'),
    ]

    for regime, color, label in region_styles:
        mask = (rl == regime) & valid_mask
        if not np.any(mask):
            continue
        idxs = np.where(mask)[0]
        for _, g in groupby(enumerate(idxs), lambda x: x[0] - x[1]):
            group = list(map(itemgetter(1), g))
            if len(group) < 3:
                continue
            P_seg = P_arr[group]
            J_seg = J_arr[group]
            ax.loglog(P_seg, J_seg, color=color, lw=4, alpha=0.7)
            slope_seg, *_ = stats.linregress(
                np.log10(P_seg), np.log10(np.abs(J_seg))
            )
            mid = len(group) // 2
            ax.text(
                P_seg[mid], J_seg[mid],
                f'{label}\nSlope={slope_seg:.2f}',
                color=color,
                fontsize=STYLE['fontsize_annotation']-1,
                fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
            )

# ── f=0 baseline — perfect oxide (L2+L6) ─────────────────────────────────────
J_baseline = np.full(len(P_A), np.nan)
fs_base    = np.full(len(P_A), np.nan)
fo_base    = np.full(len(P_A), np.nan)
fm_base    = np.full(len(P_A), np.nan)

for i, P_up in enumerate(P_A):
    try:
        r = solve_steady_state_flux_direct(
            P_up, P_down, L_m,
            k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m
        )
        J_baseline[i] = r['J_ss']
        fs_base[i]    = r['resistances']['fraction_surface']
        fo_base[i]    = r['resistances']['fraction_oxide']
        fm_base[i]    = r['resistances']['fraction_metal']
    except Exception:
        pass

v_base = ~np.isnan(J_baseline) & (J_baseline > 0)

ax1.loglog(
    P_A[v_base], J_baseline[v_base],
    color='black', ls='-', lw=STYLE['linewidth'],
    label='f=0  (L2+L6, perfect oxide)'
)
#plot_regime_overlay(ax1, P_A, J_baseline, fs_base, fo_base, fm_base, v_base)

# ── Intermediate defect fractions ─────────────────────────────────────────────
f_cases = [
    (0.001, CURVE_STYLES['f_001']['color'], '--', 'f=0.1%'),
    (0.01,  CURVE_STYLES['f_01']['color'],  '-.', 'f=1%'),
    (0.1,   CURVE_STYLES['f_10']['color'],  ':',  'f=10%'),
]

base_total = sum(v['area_fraction'] for v in defect_config.values())

for f_total, color, ls, label in f_cases:
    if base_total == 0:
        print(f"  ⚠  base_total = 0 — skipping {label}")
        continue

    scale     = f_total / base_total
    dc_scaled = {
        path: {**cfg, 'area_fraction': cfg['area_fraction'] * scale}
        for path, cfg in defect_config.items()
    }

    # Pre-allocated fixed-length arrays — cannot grow beyond len(P_A)
    J_f  = np.full(len(P_A), np.nan)
    fs_f = np.full(len(P_A), np.nan)
    fo_f = np.full(len(P_A), np.nan)
    fm_f = np.full(len(P_A), np.nan)

    for i, P_up in enumerate(P_A):
        try:
            r = calculate_mixed_defect_flux_L6(
                P_up, P_down, L_m,
                k_diss, K_eq,
                D_ox, K_ox, L_ox,
                D_m, K_s_m,
                dc_scaled,
                k_diss_metal=k_diss_metal_arg,
                K_eq_metal=K_eq_metal_arg,
            )
            J_f[i] = r['J_total']

            dom = r.get('dominant_path', 'intact')
            if dom == 'intact':
                res = r['intact_path']['resistances']
            elif dom in r.get('defect_paths', {}):
                res = r['defect_paths'][dom]['path_result']['resistances']
            else:
                res = r['intact_path']['resistances']

            fs_f[i] = res.get('fraction_surface', np.nan)
            fo_f[i] = res.get('fraction_oxide',   np.nan)
            fm_f[i] = res.get('fraction_metal',   np.nan)

        except Exception:
            pass    # array already filled with nan at index i

    v_f = ~np.isnan(J_f) & (J_f > 0)

    if not np.any(v_f):
        print(f"  ⚠  {label}: no valid points — check solver for dc_scaled")
        continue

    ax1.loglog(
        P_A[v_f], J_f[v_f],
        color='black', ls=ls, lw=STYLE['linewidth']-0.5,
        alpha=0.4, label=label
    )
    #plot_regime_overlay(ax1, P_A, J_f, fs_f, fo_f, fm_f, v_f)

# ── f=1 baseline — fully defective oxide ─────────────────────────────────────
# Dominant defect type = highest area fraction in config
dom_defect_type = max(
    defect_config.keys(),
    key=lambda k: defect_config[k]['area_fraction']
)

if dom_defect_type == 'pinhole':
    alpha_f1 = np.inf
elif dom_defect_type == 'crack':
    alpha_f1 = alpha_intact / defect_config['crack'].get('thickness_factor', 0.1)
elif dom_defect_type == 'grain_boundary':
    alpha_f1 = (defect_config['grain_boundary'].get('diffusivity_factor', 100.0)
                * alpha_intact)

J_f1  = np.full(len(P_A), np.nan)
fs_f1 = np.full(len(P_A), np.nan)
fo_f1 = np.full(len(P_A), np.nan)
fm_f1 = np.full(len(P_A), np.nan)

for i, P_up in enumerate(P_A):
    try:
        r = calculate_path_flux_L6(
            P_up, P_down, L_m,
            k_diss, K_eq,
            alpha_f1,
            D_m, K_s_m,
            path_type=dom_defect_type,
            k_diss_metal=k_diss_metal_arg if dom_defect_type == 'pinhole' else None,
            K_eq_metal=K_eq_metal_arg     if dom_defect_type == 'pinhole' else None,
        )
        J_f1[i]  = r['flux']
        fs_f1[i] = r['resistances']['fraction_surface']
        fo_f1[i] = r['resistances']['fraction_oxide']
        fm_f1[i] = r['resistances']['fraction_metal']
    except Exception:
        pass

v_f1 = ~np.isnan(J_f1) & (J_f1 > 0)

ax1.loglog(
    P_A[v_f1], J_f1[v_f1],
    color='gray', ls='-', lw=STYLE['linewidth'],
    label=f'f=1  ({dom_defect_type}, fully defective)'
)
#plot_regime_overlay(ax1, P_A, J_f1, fs_f1, fo_f1, fm_f1, v_f1)

# ── Net slope — f=0 baseline ──────────────────────────────────────────────────
slope_net, *_ = stats.linregress(
    np.log10(P_A[v_base]), np.log10(np.abs(J_baseline[v_base]))
)

ax1.text(
    0.98, 0.02, f'Net slope (f=0) = {slope_net:.2f}',
    transform=ax1.transAxes, ha='right', va='bottom',
    fontsize=STYLE['fontsize_annotation'], fontweight='bold',
    bbox=dict(boxstyle='square', fc='wheat', ec='gray', alpha=1)
)

ax1.set_xlabel('Upstream Pressure $P_{up}$ (Pa)',           fontsize=STYLE['fontsize_axis'])
ax1.set_ylabel('Flux $J_{total}$ (mol/m²/s)',                fontsize=STYLE['fontsize_axis'])
ax1.set_title('(A) Flux vs Pressure: Defect Fraction Sweep', fontsize=STYLE['fontsize_title'])
ax1.legend(fontsize=STYLE['fontsize_legend']-1, loc='upper left')
ax1.grid(True, which='both', alpha=STYLE['grid_alpha'])
ax1.tick_params(labelsize=STYLE['fontsize_tick'])

# ─────────────────────────────────────────────────────────────────────────────
# (B) FLUX vs DEFECT FRACTION — per defect type
# ─────────────────────────────────────────────────────────────────────────────
ax2 = axes[0, 1]

P_fixed_B  = CONDITIONS['P_upstream']
f_sweep    = np.logspace(
    np.log10(f_defect_min),
    np.log10(f_defect_max),
    n_defect_points
)

# L2+L6 reference at P_fixed_B
try:
    r_ref = solve_steady_state_flux_direct(
        P_fixed_B, P_down, L_m,
        k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m
    )
    J_L2L6_ref = r_ref['J_ss']
except Exception:
    J_L2L6_ref = np.nan

ax2.axhline(
    J_L2L6_ref, color='black', ls='--', lw=1.5, alpha=0.7,
    label=f'L2+L6 baseline  (f=0)'
)

defect_type_styles = [
    ('pinhole',        CURVE_STYLES['pinhole'],  {'area_fraction': 0}),
    ('crack',          CURVE_STYLES['crack'],    {'area_fraction': 0,
                                                  'thickness_factor': SIM['oxide_defects'].get('thickness_factor', 0.1)}),
    ('grain_boundary', CURVE_STYLES['gb'],       {'area_fraction': 0,
                                                  'diffusivity_factor': SIM['oxide_defects'].get('diffusivity_factor', 100.0)}),
]

for defect_type, sty, base_cfg in defect_type_styles:
    J_sweep = []
    for f in f_sweep:
        dc_single = {defect_type: {**base_cfg, 'area_fraction': f}}
        try:
            r = calculate_mixed_defect_flux_L6(
                P_fixed_B, P_down, L_m,
                k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m,
                dc_single,
                k_diss_metal=k_diss_metal_arg if defect_type == 'pinhole' else None,
                K_eq_metal=K_eq_metal_arg   if defect_type == 'pinhole' else None,
            )
            J_sweep.append(r['J_total'])
        except Exception:
            J_sweep.append(np.nan)

    J_sweep = np.array(J_sweep)
    v_sw    = ~np.isnan(J_sweep) & (J_sweep > 0)
    if not np.any(v_sw):
        continue

    ax2.loglog(
        f_sweep[v_sw] * 100, J_sweep[v_sw],
        color=sty['color'], ls=sty['ls'],
        lw=STYLE['linewidth'],
        marker=sty['marker'], ms=sty['ms'], markevery=5,
        label=defect_type.replace('_', ' ').capitalize()
    )

    # Crossover fraction where J > J_L2L6_ref
    if not np.isnan(J_L2L6_ref):
        cross = np.where(J_sweep[v_sw] > J_L2L6_ref)[0]
        if len(cross) > 0:
            f_cross = f_sweep[v_sw][cross[0]]
            ax2.axvline(
                f_cross * 100, color=sty['color'],
                ls=':', lw=1.5, alpha=0.6
            )
            ax2.text(
                f_cross * 100, J_L2L6_ref * 2,
                f'{f_cross*100:.2f}%',
                color=sty['color'],
                fontsize=STYLE['fontsize_annotation']-1,
                ha='center'
            )

ax2.set_xlabel('Defect Fraction (%)',          fontsize=STYLE['fontsize_axis'])
ax2.set_ylabel('Flux $J_{total}$ (mol/m²/s)',  fontsize=STYLE['fontsize_axis'])
ax2.set_title(f'(B) Flux vs Defect Fraction\n'
              f'(P = {P_fixed_B:.0e} Pa, T = {T_op-273.15:.3f}°C)',
              fontsize=STYLE['fontsize_title'])
ax2.legend(fontsize=STYLE['fontsize_legend']-1, loc='upper left')
ax2.grid(True, which='both', alpha=STYLE['grid_alpha'])
ax2.tick_params(labelsize=STYLE['fontsize_tick'])

# ─────────────────────────────────────────────────────────────────────────────
# (C) RESISTANCE FRACTIONS vs PRESSURE
# ─────────────────────────────────────────────────────────────────────────────
ax3 = axes[1, 0]

ax3.semilogx(
    P_RANGE_ARR[valid], frac_surface_arr[valid] * 100,
    color=CURVE_STYLES['surface_region']['color'],
    lw=STYLE['linewidth'], label='Surface (dissociation)'
)
ax3.semilogx(
    P_RANGE_ARR[valid], frac_oxide_arr[valid] * 100,
    color=CURVE_STYLES['oxide_region']['color'],
    lw=STYLE['linewidth'], label='Oxide (diffusion)'
)
ax3.semilogx(
    P_RANGE_ARR[valid], frac_metal_arr[valid] * 100,
    color=CURVE_STYLES['metal_region']['color'],
    lw=STYLE['linewidth'], label='Metal (diffusion)'
)
ax3.axhline(
    50,
    color=CURVE_STYLES['threshold']['color'],
    ls=CURVE_STYLES['threshold']['ls'],
    lw=CURVE_STYLES['threshold']['lw'],
    alpha=CURVE_STYLES['threshold']['alpha'],
    label='50% threshold'
)

if P_cross_surf_oxide is not None:
    ax3.axvline(P_cross_surf_oxide, color='gray', ls=':', lw=1.5, alpha=0.7)
    ax3.text(
        P_cross_surf_oxide * 1.5, 53,
        f'S→O\n{P_cross_surf_oxide:.1e} Pa',
        fontsize=STYLE['fontsize_annotation']-2, color='gray'
    )
if P_cross_oxide_metal is not None:
    ax3.axvline(P_cross_oxide_metal, color='gray', ls=':', lw=1.5, alpha=0.7)
    ax3.text(
        P_cross_oxide_metal * 1.5, 53,
        f'O→M\n{P_cross_oxide_metal:.1e} Pa',
        fontsize=STYLE['fontsize_annotation']-2, color='gray'
    )

if P_cross_surf_oxide is None and P_cross_oxide_metal is None:
    dominant = ('Surface' if n_surface >= n_oxide and n_surface >= n_metal
                else 'Oxide' if n_oxide >= n_metal else 'Metal')
    ax3.text(
        0.05, 0.50,
        f'No crossover in range\n{dominant}-limited throughout',
        transform=ax3.transAxes,
        fontsize=STYLE['fontsize_annotation'],
        va='center',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9)
    )

ax3.text(
    0.05, 0.95,
    'Fractions from dominant\noxide path at each P',
    transform=ax3.transAxes,
    fontsize=STYLE['fontsize_annotation']-1, va='top',
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8)
)

ax3.set_xlabel('Upstream Pressure $P_{up}$ (Pa)', fontsize=STYLE['fontsize_axis'])
ax3.set_ylabel('Resistance Fraction (%)',          fontsize=STYLE['fontsize_axis'])
ax3.set_title('(C) Rate-Limiting Step Analysis\n(dominant path)',
              fontsize=STYLE['fontsize_title'])
ax3.set_ylim(0, 105)
ax3.legend(fontsize=STYLE['fontsize_legend'], loc='center right')
ax3.grid(True, alpha=STYLE['grid_alpha'])
ax3.tick_params(labelsize=STYLE['fontsize_tick'])

# ─────────────────────────────────────────────────────────────────────────────
# (D) LIMIT CHECK — f→0 recovers L2+L6
# ─────────────────────────────────────────────────────────────────────────────
ax4 = axes[1, 1]

# Build near-zero defect config
dc_zero = {}
for path, cfg in defect_config.items():
    dc_zero[path] = {**cfg, 'area_fraction': f_limit_check}

# L3+L6 at f→0
J_f_zero = []
for P_up in P_RANGE_ARR:
    try:
        r = calculate_mixed_defect_flux_L6(
            P_up, P_down, L_m,
            k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m,
            dc_zero,
            k_diss_metal=k_diss_metal_arg,
            K_eq_metal=K_eq_metal_arg,
        )
        J_f_zero.append(r['J_total'])
    except Exception:
        J_f_zero.append(np.nan)
J_f_zero = np.array(J_f_zero)

# L2+L6 reference
J_L2L6 = []
for P_up in P_RANGE_ARR:
    try:
        r = solve_steady_state_flux_direct(
            P_up, P_down, L_m,
            k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m
        )
        J_L2L6.append(r['J_ss'])
    except Exception:
        J_L2L6.append(np.nan)
J_L2L6 = np.array(J_L2L6)

valid_parity = (~np.isnan(J_f_zero) & ~np.isnan(J_L2L6) &
                (J_f_zero > 0) & (J_L2L6 > 0))

ax4.loglog(
    J_L2L6[valid_parity], J_f_zero[valid_parity],
    'o', color=CURVE_STYLES['L3_L6']['color'],
    markersize=6, alpha=0.8,
    label=f'L3+L6  (f → {f_limit_check:.0e})'
)

J_min = min(J_L2L6[valid_parity].min(), J_f_zero[valid_parity].min()) * 0.5
J_max = max(J_L2L6[valid_parity].max(), J_f_zero[valid_parity].max()) * 2.0
ax4.loglog(
    [J_min, J_max], [J_min, J_max],
    color=CURVE_STYLES['parity']['color'],
    ls=CURVE_STYLES['parity']['ls'],
    lw=CURVE_STYLES['parity']['lw'],
    alpha=CURVE_STYLES['parity']['alpha'],
    label='Perfect agreement'
)

rel_errors = (np.abs(J_f_zero[valid_parity] - J_L2L6[valid_parity])
              / J_L2L6[valid_parity] * 100)
max_err   = rel_errors.max()
mean_err  = rel_errors.mean()
box_color = 'lightgreen' if max_err < 1.0 else 'lightyellow'

ax4.text(
    0.05, 0.95,
    f'f = {f_limit_check:.0e}  (all paths)\n'
    f'Max error  = {max_err:.2e}%\n'
    f'Mean error = {mean_err:.2e}%\n'
    f'{"✓ VALIDATED" if max_err < 1.0 else "⚠ CHECK"}',
    transform=ax4.transAxes, fontsize=STYLE['fontsize_annotation'],
    va='top',
    bbox=dict(boxstyle='round', facecolor=box_color, alpha=0.9)
)

ax4.set_xlabel('L2+L6 Flux $J_{L2+L6}$ (mol/m²/s)',       fontsize=STYLE['fontsize_axis'])
ax4.set_ylabel('L3+L6 Flux $J_{L3+L6}$ (f→0) (mol/m²/s)', fontsize=STYLE['fontsize_axis'])
ax4.set_title('(D) Limit Check: $f \\to 0$ → L2+L6',       fontsize=STYLE['fontsize_title'])
ax4.set_aspect('equal', adjustable='box')
ax4.legend(fontsize=STYLE['fontsize_legend'], loc='lower right')
ax4.grid(True, which='both', alpha=STYLE['grid_alpha'])
ax4.tick_params(labelsize=STYLE['fontsize_tick'])

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
save_figure(fig, "L3L6_core_validation", FIGS)

print("\n" + "=" * 60)
print("L3+L6 FIGURE 1 — VALIDATION SUMMARY")
print("=" * 60)
print(f"  Net log-log slope:     {slope_net:.4f}")
print(f"  Enhancement range:     "
      f"{np.nanmin(enhancement_arr):.3f} → {np.nanmax(enhancement_arr):.3f}")
print(f"  Limit check error:     {max_err:.2e}%  "
      f"({'✓' if max_err < 1.0 else '⚠'})")
if not np.isnan(J_L2L6_ref):
    print(f"\n  Flux at P={P_fixed_B:.0e} Pa:")
    print(f"    L2+L6 baseline:  {J_L2L6_ref:.3e} mol/m²/s")
    for defect_type, sty, _ in defect_type_styles:
        cfg = defect_config.get(defect_type, {})
        f_v = cfg.get('area_fraction', 0)
        if f_v > 0:
            print(f"    {defect_type:<16} f={f_v*100:.2f}%: "
                  f"enhancement estimated from sweep")

Panel (A): 40 pressure points  1.0e-07 – 1.0e+07 Pa


/var/folders/d4/mt_nqf_x42vgxs9t9mw1l59m0000gn/T/ipykernel_20394/3701431159.py:525: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



L3+L6 FIGURE 1 — VALIDATION SUMMARY
  Net log-log slope:     0.7564
  Enhancement range:     1.040 → 2.935
  Limit check error:     4.38e-14%  (✓)

  Flux at P=1e+05 Pa:
    L2+L6 baseline:  8.796e-07 mol/m²/s
    pinhole          f=1.00%: enhancement estimated from sweep
    crack            f=0.50%: enhancement estimated from sweep
    grain_boundary   f=0.50%: enhancement estimated from sweep


In [26]:
# =============================================================================
# CELL 6 — DATAFRAME OUTPUT
# =============================================================================
df = pd.DataFrame(rows)

df_display = df.copy()
for col in df_display.columns:
    if col in ('system_rate_limiting', 'dominant_path', 'Error'):
        continue
    if df_display[col].dtype == float:
        if 'fraction' in col.lower():
            df_display[col] = df_display[col].round(2)
        elif 'enhancement' in col.lower():
            df_display[col] = df_display[col].round(4)
        else:
            df_display[col] = df_display[col].apply(
                lambda x: f'{x:.4e}' if pd.notna(x) else 'NaN'
            )

n_surface_df = (df['system_rate_limiting'] == 'SURFACE').sum()
n_oxide_df   = (df['system_rate_limiting'] == 'OXIDE').sum()
n_metal_df   = (df['system_rate_limiting'] == 'METAL').sum()
n_mixed_df   = (df['system_rate_limiting'] == 'MIXED').sum()
n_error_df   = (df['system_rate_limiting'] == 'ERROR').sum()

n_intact_df  = (df['dominant_path'] == 'intact').sum()
n_pin_df     = (df['dominant_path'] == 'pinhole').sum()
n_crack_df   = (df['dominant_path'] == 'crack').sum()
n_gb_df      = (df['dominant_path'] == 'grain_boundary').sum()

print("=" * 60)
print("L3+L6 RESULTS DATAFRAME")
print("=" * 60)
print(f"  Oxide:       {OXIDE_KEY}")
print(f"  Metal:       {METAL_KEY}")
print(f"  T:           {T_op-273.15:.3f}°C  ({T_op} K)")
print(f"  L_ox:        {L_ox*1e6:.3f} μm")
print(f"  L_m:         {L_m*1e3:.3f} mm")
print(f"  k_diss:      {k_diss:.3e}  mol/m²/s/Pa")
print(f"  K_eq:        {K_eq:.3e}  Pa⁻¹")
print(f"  α_intact:    {alpha_intact:.3e}  mol/m²/s/Pa^0.5")
print(f"  β:           {beta:.3e}  mol/m²/s/Pa^0.5")

print(f"\n  Defect configuration")
for path, cfg in defect_config.items():
    print(f"    {path:<16} f = {cfg['area_fraction']*100:.3f}%")

print(f"\n  Total rows:          {len(df)}")
print(f"\n  Resistance regime (dominant path):")
print(f"    Surface-limited:   {n_surface_df}")
print(f"    Oxide-limited:     {n_oxide_df}")
print(f"    Metal-limited:     {n_metal_df}")
print(f"    Mixed:             {n_mixed_df}")
if n_error_df > 0:
    print(f"    Errors:            {n_error_df}  ⚠")

print(f"\n  Dominant oxide path:")
print(f"    Intact:            {n_intact_df}")
print(f"    Pinhole:           {n_pin_df}")
print(f"    Crack:             {n_crack_df}")
print(f"    Grain boundary:    {n_gb_df}")

print(f"\n  Enhancement factor:")
valid_enh = df['enhancement_factor'].dropna()
if len(valid_enh) > 0:
    print(f"    Min = {valid_enh.min():.4f}")
    print(f"    Max = {valid_enh.max():.4f}")
    print(f"    Mean = {valid_enh.mean():.4f}")

print("=" * 60)

display(df_display)

L3+L6 RESULTS DATAFRAME
  Oxide:       Cr2O3_sample4
  Metal:       metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985
  T:           599.850°C  (873 K)
  L_ox:        1.000 μm
  L_m:         1.000 mm
  k_diss:      2.370e-09  mol/m²/s/Pa
  K_eq:        2.799e-05  Pa⁻¹
  α_intact:    4.002e-09  mol/m²/s/Pa^0.5
  β:           9.534e-09  mol/m²/s/Pa^0.5

  Defect configuration
    pinhole          f = 1.000%
    crack            f = 0.500%
    grain_boundary   f = 0.500%

  Total rows:          40

  Resistance regime (dominant path):
    Surface-limited:   6
    Oxide-limited:     19
    Metal-limited:     0
    Mixed:             1

  Dominant oxide path:
    Intact:            26
    Pinhole:           0
    Crack:             0
    Grain boundary:    0

  Enhancement factor:
    Min = 1.0401
    Max = 2.9349
    Mean = 1.5898


,P_up (Pa),J_total (mol/m²/s),J_intact,J_pinhole,J_crack,J_grain_boundary,frac_intact (%),frac_pinhole (%),frac_crack (%),frac_GB (%),enhancement_factor,dominant_path,fraction_surface (%),fraction_oxide (%),fraction_metal (%),system_rate_limiting
0,1.0000e-07,6.9563e-16,2.3702e-16,4.6098e-14,2.3702e-16,2.3702e-16,3.3391e+01,6.6268e+01,1.7036e-01,1.7036e-01,2.9349,mixed,100.00,0.00,0.00,MIXED (MULTIPLE PATHS)
1,2.2855e-07,1.5895e-15,5.4170e-16,1.0532e-13,5.4170e-16,5.4170e-16,3.3398e+01,6.6261e+01,1.7040e-01,1.7040e-01,2.9343,mixed,100.00,0.00,0.00,MIXED (MULTIPLE PATHS)
2,5.2233e-07,3.6311e-15,1.2380e-15,2.4055e-13,1.2380e-15,1.2380e-15,3.3413e+01,6.6246e+01,1.7047e-01,1.7047e-01,2.9330,mixed,100.00,0.00,0.00,MIXED (MULTIPLE PATHS)
3,1.1938e-06,8.2902e-15,2.8295e-15,5.4890e-13,2.8295e-15,2.8295e-15,3.3448e+01,6.6211e+01,1.7065e-01,1.7065e-01,2.9300,mixed,100.00,0.00,0.00,MIXED (MULTIPLE PATHS)
4,2.7283e-06,1.8903e-14,6.4667e-15,1.2501e-12,6.4667e-15,6.4667e-15,3.3526e+01,6.6132e+01,1.7105e-01,1.7105e-01,2.9231,mixed,100.00,0.00,0.00,MIXED (MULTIPLE PATHS)
5,6.2355e-06,4.2975e-14,1.4779e-14,2.8343e-12,1.4779e-14,1.4779e-14,3.3703e+01,6.5953e+01,1.7195e-01,1.7195e-01,2.9078,mixed,100.00,0.00,0.00,MIXED (MULTIPLE PATHS)
6,1.4251e-05,9.7092e-14,3.3777e-14,6.3652e-12,3.3778e-14,3.3778e-14,3.4093e+01,6.5559e+01,1.7395e-01,1.7395e-01,2.8745,mixed,100.00,0.00,0.00,MIXED (MULTIPLE PATHS)
7,3.2570e-05,2.1662e-13,7.7196e-14,1.4020e-11,7.7197e-14,7.7197e-14,3.4924e+01,6.4720e+01,1.7819e-01,1.7819e-01,2.8061,mixed,100.00,0.00,0.00,MIXED (MULTIPLE PATHS)
8,7.4438e-05,4.7276e-13,1.7642e-13,2.9810e-11,1.7643e-13,1.7643e-13,3.6572e+01,6.3055e+01,1.8660e-01,1.8660e-01,2.6797,mixed,99.99,0.00,0.00,MIXED (MULTIPLE PATHS)
9,1.7013e-04,1.0001e-12,4.0318e-13,6.0100e-11,4.0322e-13,4.0322e-13,3.9506e+01,6.0091e+01,2.0158e-01,2.0158e-01,2.4806,mixed,99.99,0.01,0.00,MIXED (MULTIPLE PATHS)


### L2a + L4 + L6: Mixed Surface Kinetics + Mixed Bulk Diffusion

In [27]:
# =============================================================================
# CELL 1 — SETUP
# Run once. All imports and helper functions.
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import brentq
import pandas as pd
from itertools import groupby
from operator import itemgetter
from IPython.display import display

from calculations.surface_kinetics import (
    calculate_defective_metal_flux_L6,
    solve_steady_state_flux_direct,
)
from calculations.defective_metal import metal_diffusivity
from calculations.config.model_config import (
    R, F,
    METALS, OXIDES, CONDITIONS, MICROSTRUCTURE, MODEL_OPTIONS, SUGGESTED_RANGES_LEVEL5,
    PLOT_STYLE as STYLE,
    CURVE_STYLES,
    apply_style,
    build_simulation_config,
)

# ── Simulation config ─────────────────────────────────────────────────────────
SIM       = build_simulation_config()

METAL_KEY = SIM['metal_name']
OXIDE_KEY = SIM['oxide_name']
METAL     = METALS[METAL_KEY]
OXIDE     = OXIDES[OXIDE_KEY]
SK_OX     = OXIDE['surface_kinetics']

# ── Pressure sweep ────────────────────────────────────────────────────────────
P_RANGE_ARR = np.logspace(
    np.log10(CONDITIONS['P_range'][0]),
    np.log10(CONDITIONS['P_range'][1]),
    CONDITIONS['n_points']
)
P_DOWN      = SIM['P_downstream']
T_RANGE     = CONDITIONS['T_range']
N_T         = CONDITIONS['n_points']

# ── Arrhenius helper functions — oxide ───────────────────────────────────────
def get_k_diss_at_T(T_K):
    return SK_OX['k_diss_ref'] * np.exp(
        (-SK_OX['E_diss'] / R) * (1/T_K - 1/SK_OX['T_ref'])
    )

def get_K_eq_at_T(T_K):
    return SK_OX['K_eq_ref'] * np.exp(
        (-SK_OX['H_eq'] / R) * (1/T_K - 1/SK_OX['T_ref'])
    )

def get_D_ox_at_T(T_K):
    return OXIDE['D_ox_ref'] * np.exp(
        (-OXIDE['E_D_ox'] / R) * (1/T_K - 1/OXIDE['T_ref'])
    )

def get_K_ox_at_T(T_K):
    return OXIDE['K_ox_ref'] * np.exp(
        (-OXIDE['H_sol_ox'] / R) * (1/T_K - 1/OXIDE['T_ref'])
    )

# ── Arrhenius helper functions — metal ───────────────────────────────────────
def get_D_m_at_T(T_K):
    return METAL['D_ref'] * np.exp(
        (-METAL['E_D'] / R) * (1/T_K - 1/METAL['T_ref'])
    )

def get_K_s_m_at_T(T_K):
    return METAL['K_s_ref'] * np.exp(
        (-METAL['H_s'] / R) * (1/T_K - 1/METAL['T_ref'])
    )

# ── D_eff helper ──────────────────────────────────────────────────────────────
def get_D_eff_at_T(T_K, gb_on, microstructure):
    """Metal diffusivity at T_K, with or without grain-boundary enhancement."""
    return metal_diffusivity(get_D_m_at_T(T_K), T_K, microstructure,
                             enable_gb_enhancement=gb_on)

# ── Known reference activation energies ──────────────────────────────────────
E_REF_SURFACE = SK_OX['E_diss']
E_REF_OXIDE   = OXIDE['E_D_ox'] + OXIDE['H_sol_ox']
E_REF_METAL   = METAL['E_D']    + METAL['H_s']

print("=" * 60)
print("SETUP COMPLETE")
print("=" * 60)
print(f"  Metal:           {METAL_KEY}")
print(f"  Oxide:           {OXIDE_KEY}")
print(f"  E_ref surface:   {E_REF_SURFACE/1000:.3f} kJ/mol  (E_diss)")
print(f"  E_ref oxide:     {E_REF_OXIDE/1000:.3f} kJ/mol  (E_D_ox + H_sol_ox)")
print(f"  E_ref metal:     {E_REF_METAL/1000:.3f} kJ/mol  (E_D + H_s)")
print(f"  P range:         {P_RANGE_ARR[0]:.0e} – {P_RANGE_ARR[-1]:.0e} Pa")
print(f"  T range:         {T_RANGE[0]} – {T_RANGE[1]} K")
print(f"  GB enhancement:  {'on' if MODEL_OPTIONS['enable_gb_enhancement'] else 'off'}  (MODEL_OPTIONS)")

SETUP COMPLETE
  Metal:           metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985
  Oxide:           Cr2O3_sample4
  E_ref surface:   57.950 kJ/mol  (E_diss)
  E_ref oxide:     234.000 kJ/mol  (E_D_ox + H_sol_ox)
  E_ref metal:     79.400 kJ/mol  (E_D + H_s)
  P range:         1e-07 – 1e+07 Pa
  T range:         573 – 1273 K
  GB enhancement:  on  (MODEL_OPTIONS)


In [28]:
# =============================================================================
# CELL 2 — PARAMETERS
# All physics values at operating temperature, pulled from config.
# Change values in config and rerun from this cell downward.
# =============================================================================

# ── Operating conditions ──────────────────────────────────────────────────────
T_op   = SIM['T_operating']
L_ox   = SIM['L_oxide']
L_m    = SIM['L_metal']
P_down = SIM['P_downstream']

# ── Surface kinetics at T_op ──────────────────────────────────────────────────
k_diss   = get_k_diss_at_T(T_op)
K_eq     = get_K_eq_at_T(T_op)
k_recomb = k_diss / K_eq

# ── Oxide transport at T_op ───────────────────────────────────────────────────
D_ox = get_D_ox_at_T(T_op)
K_ox = get_K_ox_at_T(T_op)

# ── Metal lattice transport at T_op ───────────────────────────────────────────
D_m   = get_D_m_at_T(T_op)
K_s_m = get_K_s_m_at_T(T_op)

# ── Microstructure — from config ──────────────────────────────────────────────
microstructure = SIM['microstructure']
gb_on          = MODEL_OPTIONS['enable_gb_enhancement']
gb_label       = 'GB on' if gb_on else 'GB off'
grain_size     = microstructure['grain_size']
gb_thickness   = microstructure['gb_thickness']
gb_type        = microstructure.get('gb_type', 'LAGB')

# ── D_eff at T_op — GB enhancement off and on ────────────────────────────────
D_eff_modes = {}
for m in (False, True):
    D_eff_modes[m] = get_D_eff_at_T(T_op, m, microstructure)

D_eff = D_eff_modes[gb_on]    # operating D_eff

# ── Derived permeances ────────────────────────────────────────────────────────
alpha    = D_ox  * K_ox / L_ox
beta_L   = D_m   * K_s_m / L_m    # lattice only
beta_eff = D_eff * K_s_m / L_m    # effective (with microstructure)

# ── Validation sweep parameters — from config ─────────────────────────────────
grain_size_sweep = np.logspace(*np.log10(SUGGESTED_RANGES_LEVEL5['grain_size']), CONDITIONS['n_points'])
T_grain_size     = CONDITIONS['T_operating']

# ── Print verification ────────────────────────────────────────────────────────
print("=" * 60)
print(f"L2+L4+L6 PARAMETERS AT T = {T_op-273.15:.3f}°C  ({T_op} K)")
print("=" * 60)

print(f"\n  Geometry")
print(f"    L_ox     = {L_ox:.3e}  m")
print(f"    L_m      = {L_m:.3e}  m")
print(f"    P_down   = {P_down:.1e}  Pa")

print(f"\n  Surface kinetics  [config: oxide surface_kinetics]")
print(f"    k_diss   = {k_diss:.3e}  mol/m²/s/Pa")
print(f"    K_eq     = {K_eq:.3e}  Pa⁻¹")
print(f"    k_recomb = {k_recomb:.3e}  mol/m²/s")

print(f"\n  Oxide transport  [config: Arrhenius]")
print(f"    D_ox     = {D_ox:.3e}  m²/s")
print(f"    K_ox     = {K_ox:.3e}  mol/m³/Pa^0.5")

print(f"\n  Metal lattice transport  [config: Arrhenius]")
print(f"    D_m      = {D_m:.3e}  m²/s")
print(f"    K_s_m    = {K_s_m:.3e}  mol/m³/Pa^0.5")

print(f"\n  Microstructure  [config: MICROSTRUCTURE]")
print(f"    GB enhancement   = {gb_label}  [config: MODEL_OPTIONS]")
print(f"    grain_size       = {grain_size*1e6:.3f} μm")
print(f"    gb_thickness     = {gb_thickness*1e9:.3f} nm")
print(f"    gb_type          = {gb_type}")

print(f"\n  D_eff at T_op — GB off / on")
for m, D in D_eff_modes.items():
    ratio  = D / D_m
    marker = ' ← active' if m == gb_on else ''
    print(f"    GB {'on ' if m else 'off'}  D_eff = {D:.3e} m²/s  "
          f"D_eff/D_m = {ratio:.4f}{marker}")

print(f"\n  Derived permeances")
print(f"    α        = {alpha:.3e}  mol/m²/s/Pa^0.5  (oxide)")
print(f"    β_latt   = {beta_L:.3e}  mol/m²/s/Pa^0.5  (lattice only)")
print(f"    β_eff    = {beta_eff:.3e}  mol/m²/s/Pa^0.5  ({gb_label})")
print(f"    α/β_eff  = {alpha/beta_eff:.3e}")

print(f"\n  Reference activation energies  [config]")
print(f"    E_diss        = {E_REF_SURFACE/1000:.3f} kJ/mol")
print(f"    E_D_ox+H_sol  = {E_REF_OXIDE/1000:.3f} kJ/mol")
print(f"    E_D+H_s       = {E_REF_METAL/1000:.3f} kJ/mol")

if not gb_on:
    print(f"\n  ⚠  GB enhancement off — L2+L4+L6 = L2+L6.")
if D_eff / D_m > 10:
    print(f"\n  ⚠  D_eff/D_m = {D_eff/D_m:.3f} — GB short-circuit dominates at T_op.")

print("\n" + "=" * 60)

L2+L4+L6 PARAMETERS AT T = 599.850°C  (873 K)

  Geometry
    L_ox     = 1.000e-06  m
    L_m      = 1.000e-03  m
    P_down   = 0.0e+00  Pa

  Surface kinetics  [config: oxide surface_kinetics]
    k_diss   = 2.370e-09  mol/m²/s/Pa
    K_eq     = 2.799e-05  Pa⁻¹
    k_recomb = 8.468e-05  mol/m²/s

  Oxide transport  [config: Arrhenius]
    D_ox     = 1.395e-17  m²/s
    K_ox     = 2.869e+02  mol/m³/Pa^0.5

  Metal lattice transport  [config: Arrhenius]
    D_m      = 2.487e-10  m²/s
    K_s_m    = 3.834e-02  mol/m³/Pa^0.5

  Microstructure  [config: MICROSTRUCTURE]
    GB enhancement   = GB on  [config: MODEL_OPTIONS]
    grain_size       = 100.000 μm
    gb_thickness     = 0.500 nm
    gb_type          = LAGB

  D_eff at T_op — GB off / on
    GB off  D_eff = 2.487e-10 m²/s  D_eff/D_m = 1.0000
    GB on   D_eff = 2.487e-10 m²/s  D_eff/D_m = 1.0001 ← active

  Derived permeances
    α        = 4.002e-09  mol/m²/s/Pa^0.5  (oxide)
    β_latt   = 9.534e-09  mol/m²/s/Pa^0.5  (lattice only

/Users/rwest/CoMoChEng Dropbox/Richard West/Northeastern/Research/Research-MHI-2026-hydrogen/Continuum_H_permeation/calculations/defective_metal.py:222: UserWarning: Temperature 873.0 K is outside the validated range [1023.0, 1223.0] K of metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985's diffusion data. The GB enhancement is extrapolated.
  warnings.warn(


In [29]:
# =============================================================================
# CELL 3 — COMPUTE
# Single-pass loop over full pressure range.
# Builds all arrays, regime summary, and Arrhenius pressure selection.
# =============================================================================

# ── Single-pass loop — pre-allocated fixed-length arrays ─────────────────────
rows             = []
J_arr            = np.full(len(P_RANGE_ARR), np.nan)
theta_arr        = np.full(len(P_RANGE_ARR), np.nan)
P_int_arr        = np.full(len(P_RANGE_ARR), np.nan)
D_eff_arr        = np.full(len(P_RANGE_ARR), np.nan)
frac_surface_arr = np.full(len(P_RANGE_ARR), np.nan)
frac_oxide_arr   = np.full(len(P_RANGE_ARR), np.nan)
frac_metal_arr   = np.full(len(P_RANGE_ARR), np.nan)
R_surface_arr    = np.full(len(P_RANGE_ARR), np.nan)
R_oxide_arr      = np.full(len(P_RANGE_ARR), np.nan)
R_metal_arr      = np.full(len(P_RANGE_ARR), np.nan)
R_total_arr      = np.full(len(P_RANGE_ARR), np.nan)
rate_lim_arr     = np.empty(len(P_RANGE_ARR), dtype=object)

for i, P_up in enumerate(P_RANGE_ARR):
    try:
        r = calculate_defective_metal_flux_L6(
            P_up, P_down,
            L_m,                              # thickness
            T_op,                             # temperature — 4th positional
            k_diss, K_eq,
            D_ox, K_ox, L_ox,
            D_m,                              # D_lattice
            K_s_m,
            microstructure_params=microstructure,
            enable_gb_enhancement=gb_on,
        )

        J_arr[i]            = r['flux']
        theta_arr[i]        = r['theta_surface']
        P_int_arr[i]        = r['P_int']
        D_eff_arr[i]        = r.get('D_eff', D_m)
        frac_surface_arr[i] = r['resistances']['fraction_surface']
        frac_oxide_arr[i]   = r['resistances']['fraction_oxide']
        frac_metal_arr[i]   = r['resistances']['fraction_metal']
        R_surface_arr[i]    = r['resistances']['R_surface']
        R_oxide_arr[i]      = r['resistances']['R_oxide']
        R_metal_arr[i]      = r['resistances']['R_metal']
        R_total_arr[i]      = r['resistances']['R_total']
        rate_lim_arr[i]     = r['rate_limiting']

        rows.append({
            'P_up (Pa)':            P_up,
            'P_int (Pa)':           r['P_int'],
            'J_ss (mol/m²/s)':      r['flux'],
            'θ_surface':            r['theta_surface'],
            'D_eff (m²/s)':         r.get('D_eff', D_m),
            'D_eff/D_m':            r.get('D_eff', D_m) / D_m,
            'α_oxide':              r.get('alpha', alpha),
            'β_eff':                r.get('beta_eff', beta_eff),
            'R_surface':            r['resistances']['R_surface'],
            'R_oxide':              r['resistances']['R_oxide'],
            'R_metal':              r['resistances']['R_metal'],
            'R_total':              r['resistances']['R_total'],
            'fraction_surface (%)': r['resistances']['fraction_surface'] * 100,
            'fraction_oxide (%)':   r['resistances']['fraction_oxide']   * 100,
            'fraction_metal (%)':   r['resistances']['fraction_metal']   * 100,
            'Rate-Limiting':        r['rate_limiting'].upper(),
            'GB enhancement':       gb_label,
        })

    except Exception as e:
        rate_lim_arr[i] = 'error'
        rows.append({
            'P_up (Pa)':     P_up,
            'Rate-Limiting': 'ERROR',
            'GB enhancement': gb_label,
            'Error':         str(e),
        })

# ── Derived arrays ────────────────────────────────────────────────────────────
valid  = ~np.isnan(J_arr)

rl_arr = np.where(frac_surface_arr > 0.5, 'surface',
         np.where(frac_oxide_arr   > 0.5, 'oxide',
         np.where(frac_metal_arr   > 0.5, 'metal', 'mixed')))

# ── Regime summary ────────────────────────────────────────────────────────────
n_surface = np.sum((frac_surface_arr > 0.5) & valid)
n_oxide   = np.sum((frac_oxide_arr   > 0.5) & valid)
n_metal   = np.sum((frac_metal_arr   > 0.5) & valid)
n_mixed   = np.sum(
    (frac_surface_arr <= 0.5) & (frac_oxide_arr <= 0.5) &
    (frac_metal_arr   <= 0.5) & valid
)

surf_dom_mask  = (frac_surface_arr > 0.90) & valid
oxide_dom_mask = (frac_oxide_arr   > 0.90) & valid
metal_dom_mask = (frac_metal_arr   > 0.90) & valid
P_surf_dom     = P_RANGE_ARR[surf_dom_mask]
P_oxide_dom    = P_RANGE_ARR[oxide_dom_mask]
P_metal_dom    = P_RANGE_ARR[metal_dom_mask]

P_for_surface_Ea = np.median(P_surf_dom)  if len(P_surf_dom)  > 3 else None
P_for_oxide_Ea   = np.median(P_oxide_dom) if len(P_oxide_dom) > 3 else None
P_for_metal_Ea   = np.median(P_metal_dom) if len(P_metal_dom) > 3 else None

# ── Crossover pressures ───────────────────────────────────────────────────────
def find_crossover(frac_a, frac_b, valid_mask):
    vc   = valid_mask & ~np.isnan(frac_a) & ~np.isnan(frac_b)
    if np.sum(vc) < 2:
        return None
    diff = frac_a[vc] - frac_b[vc]
    Pv   = P_RANGE_ARR[vc]
    sc   = np.where(np.diff(np.sign(diff)))[0]
    if len(sc) == 0:
        return None
    idx = sc[0]
    return Pv[idx] if diff[idx] > 0 else None

P_cross_surf_oxide  = find_crossover(frac_surface_arr, frac_oxide_arr,  valid)
P_cross_surf_metal  = find_crossover(frac_surface_arr, frac_metal_arr,  valid)
P_cross_oxide_metal = find_crossover(frac_oxide_arr,   frac_metal_arr,  valid)

# ── Print regime summary ──────────────────────────────────────────────────────
print("=" * 60)
print(f"L2+L4+L6 REGIME SUMMARY")
print(f"T = {T_op-273.15:.3f}°C  |  {gb_label}  |  "
      f"{np.sum(valid)} valid points")
print("=" * 60)

print(f"\n  Resistance regime (fraction > 50%)")
print(f"    Surface-limited:  {n_surface:3d} points")
print(f"    Oxide-limited:    {n_oxide:3d} points")
print(f"    Metal-limited:    {n_metal:3d} points")
print(f"    Mixed:            {n_mixed:3d} points")

print(f"\n  Dominant regions (fraction > 90%) — used for Arrhenius")
for label, dom, P_ea in [
    ('Surface', P_surf_dom,  P_for_surface_Ea),
    ('Oxide',   P_oxide_dom, P_for_oxide_Ea),
    ('Metal',   P_metal_dom, P_for_metal_Ea),
]:
    if len(dom) > 3:
        print(f"    {label}-dominant: {len(dom):3d} points  "
              f"P = {dom.min():.1e} – {dom.max():.1e} Pa  "
              f"→ P_for_{label.lower()}_Ea = {P_ea:.2e} Pa")
    else:
        print(f"    {label}-dominant: {len(dom):3d} points  "
              f"(insufficient — {label.lower()} Ea not extractable)")

print(f"\n  Crossover pressures (diagnostic only)")
for desc, P_co in [
    ('Surface → Oxide', P_cross_surf_oxide),
    ('Surface → Metal', P_cross_surf_metal),
    ('Oxide   → Metal', P_cross_oxide_metal),
]:
    val = f'{P_co:.2e} Pa' if P_co is not None else 'not found'
    print(f"    {desc}:  {val}")

print(f"\n  D_eff at T_op")
print(f"    D_eff ({gb_label})  = {D_eff:.3e} m²/s")
print(f"    D_m   (lattice)     = {D_m:.3e} m²/s")
print(f"    D_eff/D_m           = {D_eff/D_m:.4f}")
d_eff_range = D_eff_arr[valid]
if len(d_eff_range) > 0:
    print(f"    D_eff range over P  = "
          f"{np.nanmin(d_eff_range):.3e} – {np.nanmax(d_eff_range):.3e} m²/s")

if np.sum(valid) > 0:
    print(f"\n  θ range:  {np.nanmin(theta_arr):.4f} → {np.nanmax(theta_arr):.4f}")
    print(f"  J range:  {np.nanmin(J_arr):.2e} → {np.nanmax(J_arr):.2e} mol/m²/s")
else:
    print(f"\n  ⚠  No valid points — check solver signature and config.")
print("=" * 60)

L2+L4+L6 REGIME SUMMARY
T = 599.850°C  |  GB on  |  40 valid points

  Resistance regime (fraction > 50%)
    Surface-limited:   20 points
    Oxide-limited:     19 points
    Metal-limited:      0 points
    Mixed:              1 points

  Dominant regions (fraction > 90%) — used for Arrhenius
    Surface-dominant:  18 points  P = 1.0e-07 – 1.3e-01 Pa  → P_for_surface_Ea = 1.22e-04 Pa
    Oxide-dominant:   0 points  (insufficient — oxide Ea not extractable)
    Metal-dominant:   0 points  (insufficient — metal Ea not extractable)

  Crossover pressures (diagnostic only)
    Surface → Oxide:  1.51e+00 Pa
    Surface → Metal:  3.46e+00 Pa
    Oxide   → Metal:  not found

  D_eff at T_op
    D_eff (GB on)  = 2.487e-10 m²/s
    D_m   (lattice)     = 2.487e-10 m²/s
    D_eff/D_m           = 1.0001
    D_eff range over P  = 2.487e-10 – 2.487e-10 m²/s

  θ range:  0.0000 → 0.9407
  J range:  2.37e-16 → 8.45e-06 mol/m²/s


In [30]:
# =============================================================================
# CELL 4 — FIGURE 1: Core Validation (2×2)
# =============================================================================
"""
Figure 1 validates the L2+L4+L6 model from four angles at operating
temperature. The key new dimension vs L2+L6 is grain-boundary (GB)
enhancement of the metal diffusivity.

Panel (A) — Flux vs Pressure: GB off vs GB on
    Question: How does GB enhancement shift the flux curve and regime
    boundaries relative to the pure lattice baseline?
    Two curves: GB off (L2+L6 baseline, black solid) and GB on
    (CURVE_STYLES L4_gb). Each curve has colored regime overlays and slope
    annotation boxes using the identical pattern from all previous levels.
    Net slope annotation box references the active config setting.

Panel (B) — D_eff/D_m vs Temperature: GB off vs GB on
    Question: How much does GB enhancement raise the effective diffusivity
    above the lattice value across the temperature range?
    Sweeps temperature from T_RANGE[0] to T_RANGE[1]. Computes D_eff with
    GB enhancement off and on at each T via get_D_eff_at_T. GBs push the
    ratio above 1, most strongly at low T (short-circuit).

Panel (C) — Resistance Fractions vs Pressure
    Question: Which step controls the rate at operating conditions?
    Consistent style with all previous levels — semilogx lines per
    fraction, 50% threshold, S→O and O→M crossover vertical lines.
    Uses the active config setting. Annotation box shows it and D_eff/D_m.

Panel (D) — Limit Check: GB off recovers L2+L6
    Question: When microstructure is disabled, does L2+L4+L6 recover
    the L2+L6 result exactly?
    Runs calculate_defective_metal_flux_L6 with GB enhancement off and compares
    to solve_steady_state_flux_direct. Parity plot. Green box if < 1%.
"""

props = dict(boxstyle='round', facecolor='wheat', alpha=0.9)

fig, axes = plt.subplots(2, 2, figsize=STYLE['figsize'])
fig.suptitle(
    f'L2+L4+L6: Perfect Oxide + Defective Metal + Surface Kinetics\n'
    f'{OXIDE_KEY} + {METAL_KEY}  |  T = {T_op-273.15:.3f}°C  |  '
    f'{gb_label}  |  grain = {grain_size*1e6:.3f} μm',
    fontsize=STYLE['fontsize_suptitle']-1, fontweight='bold', y=0.98
)

# ── Regime overlay helper — identical to all previous levels ──────────────────
def plot_regime_overlay(ax, P_arr, J_arr, frac_s, frac_o, frac_m, valid_mask):
    rl = np.where(frac_s > 0.5, 'surface',
         np.where(frac_o > 0.5, 'oxide',
         np.where(frac_m > 0.5, 'metal', 'mixed')))

    region_styles = [
        ('surface', CURVE_STYLES['surface_region']['color'], 'Surface-limited'),
        ('oxide',   CURVE_STYLES['oxide_region']['color'],   'Oxide-limited'),
        ('metal',   CURVE_STYLES['metal_region']['color'],   'Metal-limited'),
        ('mixed',   CURVE_STYLES['mixed_region']['color'],   'Mixed'),
    ]

    for regime, color, label in region_styles:
        mask = (rl == regime) & valid_mask
        if not np.any(mask):
            continue
        idxs = np.where(mask)[0]
        for _, g in groupby(enumerate(idxs), lambda x: x[0] - x[1]):
            group = list(map(itemgetter(1), g))
            if len(group) < 3:
                continue
            P_seg = P_arr[group]
            J_seg = J_arr[group]
            ax.loglog(P_seg, J_seg, color=color, lw=4, alpha=0.7)
            slope_seg, *_ = stats.linregress(
                np.log10(P_seg), np.log10(np.abs(J_seg))
            )
            mid = len(group) // 2
            ax.text(
                P_seg[mid], J_seg[mid],
                f'{label}\nSlope={slope_seg:.2f}',
                color=color,
                fontsize=STYLE['fontsize_annotation']-1,
                fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
            )

# ─────────────────────────────────────────────────────────────────────────────
# (A) FLUX vs PRESSURE — GB off vs GB on
# ─────────────────────────────────────────────────────────────────────────────
ax1 = axes[0, 0]
P_A = np.logspace(
    np.log10(CONDITIONS['P_range'][0]),
    np.log10(CONDITIONS['P_range'][1]),
    CONDITIONS['n_points']
)

mode_cases = [
    (False, 'black',                        '-',  'GB off  (L2+L6 baseline)'),
    (True,  CURVE_STYLES['L4_gb']['color'], '--', 'GB on'),
]

for m, color, ls, label in mode_cases:
    J_m  = np.full(len(P_A), np.nan)
    fs_m = np.full(len(P_A), np.nan)
    fo_m = np.full(len(P_A), np.nan)
    fm_m = np.full(len(P_A), np.nan)

    for i, P_up in enumerate(P_A):
        try:
            r = calculate_defective_metal_flux_L6(
                P_up, P_down,
                L_m,
                T_op,
                k_diss, K_eq,
                D_ox, K_ox, L_ox,
                D_m,
                K_s_m,
                microstructure_params=microstructure,
                enable_gb_enhancement=m,
            )
            J_m[i]  = r['flux']
            fs_m[i] = r['resistances']['fraction_surface']
            fo_m[i] = r['resistances']['fraction_oxide']
            fm_m[i] = r['resistances']['fraction_metal']
        except Exception:
            pass

    v_m = ~np.isnan(J_m) & (J_m > 0)
    if not np.any(v_m):
        print(f"  ⚠  GB {'on' if m else 'off'}: no valid points")
        continue

    lw    = STYLE['linewidth'] if not m else STYLE['linewidth'] - 0.5
    alpha_val = 1.0            if not m else 0.8

    ax1.loglog(
        P_A[v_m], J_m[v_m],
        color=color, ls=ls, lw=lw, alpha=alpha_val,
        label=label
    )
    plot_regime_overlay(ax1, P_A, J_m, fs_m, fo_m, fm_m, v_m)

# Net slope annotation
v_net = ~np.isnan(J_arr) & (J_arr > 0)
if np.sum(v_net) > 2:
    slope_net, *_ = stats.linregress(
        np.log10(P_RANGE_ARR[v_net]),
        np.log10(np.abs(J_arr[v_net]))
    )
    ax1.text(
        0.98, 0.02, f'Net slope ({gb_label}) = {slope_net:.2f}',
        transform=ax1.transAxes, ha='right', va='bottom',
        fontsize=STYLE['fontsize_annotation'], fontweight='bold',
        bbox=dict(boxstyle='square', fc='wheat', ec='gray', alpha=1)
    )
else:
    slope_net = np.nan

ax1.set_xlabel('Upstream Pressure $P_{up}$ (Pa)',      fontsize=STYLE['fontsize_axis'])
ax1.set_ylabel('Flux $J_{ss}$ (mol/m²/s)',              fontsize=STYLE['fontsize_axis'])
ax1.set_title('(A) Flux vs Pressure: GB off vs on', fontsize=STYLE['fontsize_title'])
ax1.legend(fontsize=STYLE['fontsize_legend']-1, loc='upper left')
ax1.grid(True, which='both', alpha=STYLE['grid_alpha'])
ax1.tick_params(labelsize=STYLE['fontsize_tick'])

# ─────────────────────────────────────────────────────────────────────────────
# (B) D_eff/D_m vs TEMPERATURE — GB off vs GB on
# ─────────────────────────────────────────────────────────────────────────────
ax2 = axes[0, 1]

temperatures_K = np.linspace(T_RANGE[0], T_RANGE[1], N_T)

mode_D_styles = [
    (False, 'black',                        '-',  'GB off  (D_eff = D_m)'),
    (True,  CURVE_STYLES['L4_gb']['color'], '--', 'GB on'),
]

for m, color, ls, label in mode_D_styles:
    ratio_arr = np.full(N_T, np.nan)

    for j, T_K in enumerate(temperatures_K):
        D_latt  = get_D_m_at_T(T_K)
        D_eff_j = get_D_eff_at_T(T_K, m, microstructure)
        if D_latt > 0:
            ratio_arr[j] = D_eff_j / D_latt

    v_r = ~np.isnan(ratio_arr)
    if not np.any(v_r):
        continue

    ax2.semilogy(
        temperatures_K[v_r] - 273.15, ratio_arr[v_r],
        color=color, ls=ls, lw=STYLE['linewidth'],
        label=label
    )

    # Annotate crossover where ratio crosses 1
    if m:
        cross = np.where(np.diff(np.sign(ratio_arr[v_r] - 1.0)))[0]
        if len(cross) > 0:
            T_cross = temperatures_K[v_r][cross[0]] - 273.15
            ax2.axvline(T_cross, color=color, ls=':', lw=1.5, alpha=0.6)
            ax2.text(
                T_cross + 5, 2.0,
                f'{T_cross:.3f}°C',
                color=color, fontsize=STYLE['fontsize_annotation']-1
            )

# Unity reference line
ax2.axhline(
    1.0, color='gray', ls='--', lw=1.5, alpha=0.7,
    label='$D_{eff}/D_m = 1$  (lattice)'
)

# T_op marker
ax2.axvline(
    T_op - 273.15, color='black', ls=':', lw=1.5, alpha=0.5
)
ax2.text(
    T_op - 273.15 + 5, 0.5,
    f'T_op\n{T_op-273.15:.3f}°C',
    fontsize=STYLE['fontsize_annotation']-1, color='black'
)

# With coarse grains D_eff/D_m stays near 1 and the log axis spans a sliver of a
# decade, leaving the T_op label (y=0.5) far off-axis and the saved figure
# enormous. Keep 0.3-3 in view.
_lo, _hi = ax2.get_ylim()
ax2.set_ylim(min(_lo, 0.3), max(_hi, 3.0))

ax2.set_xlabel('Temperature (°C)',      fontsize=STYLE['fontsize_axis'])
ax2.set_ylabel('$D_{eff}/D_m$',         fontsize=STYLE['fontsize_axis'])
ax2.set_title('(B) $D_{eff}/D_m$ vs Temperature: GB off vs on',
              fontsize=STYLE['fontsize_title'])
ax2.legend(fontsize=STYLE['fontsize_legend']-1, loc='upper right')
ax2.grid(True, which='both', alpha=STYLE['grid_alpha'])
ax2.tick_params(labelsize=STYLE['fontsize_tick'])

# ─────────────────────────────────────────────────────────────────────────────
# (C) RESISTANCE FRACTIONS vs PRESSURE
# ─────────────────────────────────────────────────────────────────────────────
ax3 = axes[1, 0]

ax3.semilogx(
    P_RANGE_ARR[valid], frac_surface_arr[valid] * 100,
    color=CURVE_STYLES['surface_region']['color'],
    lw=STYLE['linewidth'], label='Surface (dissociation)'
)
ax3.semilogx(
    P_RANGE_ARR[valid], frac_oxide_arr[valid] * 100,
    color=CURVE_STYLES['oxide_region']['color'],
    lw=STYLE['linewidth'], label='Oxide (diffusion)'
)
ax3.semilogx(
    P_RANGE_ARR[valid], frac_metal_arr[valid] * 100,
    color=CURVE_STYLES['metal_region']['color'],
    lw=STYLE['linewidth'], label=f'Metal ({gb_label})'
)
ax3.axhline(
    50,
    color=CURVE_STYLES['threshold']['color'],
    ls=CURVE_STYLES['threshold']['ls'],
    lw=CURVE_STYLES['threshold']['lw'],
    alpha=CURVE_STYLES['threshold']['alpha'],
    label='50% threshold'
)

if P_cross_surf_oxide is not None:
    ax3.axvline(P_cross_surf_oxide, color='gray', ls=':', lw=1.5, alpha=0.7)
    ax3.text(
        P_cross_surf_oxide * 1.5, 53,
        f'S→O\n{P_cross_surf_oxide:.1e} Pa',
        fontsize=STYLE['fontsize_annotation']-2, color='gray'
    )
if P_cross_oxide_metal is not None:
    ax3.axvline(P_cross_oxide_metal, color='gray', ls=':', lw=1.5, alpha=0.7)
    ax3.text(
        P_cross_oxide_metal * 1.5, 53,
        f'O→M\n{P_cross_oxide_metal:.1e} Pa',
        fontsize=STYLE['fontsize_annotation']-2, color='gray'
    )

if P_cross_surf_oxide is None and P_cross_oxide_metal is None:
    dominant = ('Surface' if n_surface >= n_oxide and n_surface >= n_metal
                else 'Oxide' if n_oxide >= n_metal else 'Metal')
    ax3.text(
        0.05, 0.50,
        f'No crossover in range\n{dominant}-limited throughout',
        transform=ax3.transAxes,
        fontsize=STYLE['fontsize_annotation'],
        va='center',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9)
    )

ax3.text(
    0.05, 0.95,
    f'{gb_label}\nD_eff/D_m = {D_eff/D_m:.4f}',
    transform=ax3.transAxes,
    fontsize=STYLE['fontsize_annotation']-1, va='top',
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8)
)

ax3.set_xlabel('Upstream Pressure $P_{up}$ (Pa)', fontsize=STYLE['fontsize_axis'])
ax3.set_ylabel('Resistance Fraction (%)',          fontsize=STYLE['fontsize_axis'])
ax3.set_title('(C) Rate-Limiting Step Analysis',   fontsize=STYLE['fontsize_title'])
ax3.set_ylim(0, 105)
ax3.legend(fontsize=STYLE['fontsize_legend'], loc='center right')
ax3.grid(True, alpha=STYLE['grid_alpha'])
ax3.tick_params(labelsize=STYLE['fontsize_tick'])

# ─────────────────────────────────────────────────────────────────────────────
# (D) LIMIT CHECK — GB off recovers L2+L6
# ─────────────────────────────────────────────────────────────────────────────
ax4 = axes[1, 1]

P_D = np.logspace(
    np.log10(CONDITIONS['P_range'][0]),
    np.log10(CONDITIONS['P_range'][1]),
    CONDITIONS['n_points']
)
J_none = np.full(len(P_D), np.nan)
J_L2L6 = np.full(len(P_D), np.nan)

for i, P_up in enumerate(P_D):
    try:
        r_none = calculate_defective_metal_flux_L6(
            P_up, P_down,
            L_m,
            T_op,
            k_diss, K_eq,
            D_ox, K_ox, L_ox,
            D_m,
            K_s_m,
            microstructure_params=microstructure,
            enable_gb_enhancement=False,
        )
        J_none[i] = r_none['flux']
    except Exception:
        pass

    try:
        r_L2L6 = solve_steady_state_flux_direct(
            P_up, P_down, L_m,
            k_diss, K_eq,
            D_ox, K_ox, L_ox,
            D_m, K_s_m
        )
        J_L2L6[i] = r_L2L6['J_ss']
    except Exception:
        pass

valid_parity = (~np.isnan(J_none) & ~np.isnan(J_L2L6) &
                (J_none > 0) & (J_L2L6 > 0))

if np.any(valid_parity):
    ax4.loglog(
        J_L2L6[valid_parity], J_none[valid_parity],
        'o', color=CURVE_STYLES['L2_L6']['color'],
        markersize=6, alpha=0.8,
        label='L2+L4+L6  (GB off)'
    )

    J_min = min(J_L2L6[valid_parity].min(), J_none[valid_parity].min()) * 0.5
    J_max = max(J_L2L6[valid_parity].max(), J_none[valid_parity].max()) * 2.0
    ax4.loglog(
        [J_min, J_max], [J_min, J_max],
        color=CURVE_STYLES['parity']['color'],
        ls=CURVE_STYLES['parity']['ls'],
        lw=CURVE_STYLES['parity']['lw'],
        alpha=CURVE_STYLES['parity']['alpha'],
        label='Perfect agreement'
    )

    rel_errors = (np.abs(J_none[valid_parity] - J_L2L6[valid_parity])
                  / J_L2L6[valid_parity] * 100)
    max_err   = rel_errors.max()
    mean_err  = rel_errors.mean()
    box_color = 'lightgreen' if max_err < 1.0 else 'lightyellow'

    ax4.text(
        0.05, 0.95,
        f'GB off\n'
        f'Max error  = {max_err:.2e}%\n'
        f'Mean error = {mean_err:.2e}%\n'
        f'{"✓ VALIDATED" if max_err < 1.0 else "⚠ CHECK"}',
        transform=ax4.transAxes, fontsize=STYLE['fontsize_annotation'],
        va='top',
        bbox=dict(boxstyle='round', facecolor=box_color, alpha=0.9)
    )
else:
    max_err  = np.nan
    mean_err = np.nan
    ax4.text(
        0.5, 0.5, 'No valid parity points\nCheck solver',
        transform=ax4.transAxes, ha='center', va='center',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9)
    )

ax4.set_xlabel('L2+L6 Flux $J_{L2+L6}$ (mol/m²/s)',         fontsize=STYLE['fontsize_axis'])
ax4.set_ylabel('L2+L4+L6 Flux $J$ (GB off) (mol/m²/s)',  fontsize=STYLE['fontsize_axis'])
ax4.set_title('(D) Limit Check: GB off → L2+L6',          fontsize=STYLE['fontsize_title'])
ax4.set_aspect('equal', adjustable='box')
ax4.legend(fontsize=STYLE['fontsize_legend'], loc='lower right')
ax4.grid(True, which='both', alpha=STYLE['grid_alpha'])
ax4.tick_params(labelsize=STYLE['fontsize_tick'])

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
save_figure(fig, "L2L4L6_perfect_oxide_defective_metal_surface_kinetics", FIGS)

print("\n" + "=" * 60)
print("L2+L4+L6 FIGURE 1 — VALIDATION SUMMARY")
print("=" * 60)
if not np.isnan(slope_net):
    print(f"  Net log-log slope ({gb_label}): {slope_net:.4f}")
print(f"  D_eff/D_m at T_op:")
for m, D in D_eff_modes.items():
    print(f"    GB {'on ' if m else 'off'}  {D/D_m:.4f}")
print(f"  Limit check error (GB off): {max_err:.2e}%  "
      f"({'✓' if not np.isnan(max_err) and max_err < 1.0 else '⚠'})")
if P_cross_surf_oxide is not None:
    print(f"  S→O crossover:  {P_cross_surf_oxide:.2e} Pa")
if P_cross_oxide_metal is not None:
    print(f"  O→M crossover:  {P_cross_oxide_metal:.2e} Pa")

/Users/rwest/CoMoChEng Dropbox/Richard West/Northeastern/Research/Research-MHI-2026-hydrogen/Continuum_H_permeation/calculations/defective_metal.py:222: UserWarning: Temperature 573.0 K is outside the validated range [1023.0, 1223.0] K of metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985's diffusion data. The GB enhancement is extrapolated.
  warnings.warn(
/Users/rwest/CoMoChEng Dropbox/Richard West/Northeastern/Research/Research-MHI-2026-hydrogen/Continuum_H_permeation/calculations/defective_metal.py:222: UserWarning: Temperature 590.9 K is outside the validated range [1023.0, 1223.0] K of metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985's diffusion data. The GB enhancement is extrapolated.
  warnings.warn(
/Users/rwest/CoMoChEng Dropbox/Richard West/Northeastern/Research/Research-MHI-2026-hydrogen/Continuum_H_permeation/calculations/defective_metal.py:222: UserWarning: Temperature 608.9 K is outside the validated range [1023.0, 1223.0] K of metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt19

/var/folders/d4/mt_nqf_x42vgxs9t9mw1l59m0000gn/T/ipykernel_20394/1550916536.py:406: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



L2+L4+L6 FIGURE 1 — VALIDATION SUMMARY
  Net log-log slope (GB on): 0.7564
  D_eff/D_m at T_op:
    GB off  1.0000
    GB on   1.0001
  Limit check error (GB off): 0.00e+00%  (✓)
  S→O crossover:  1.51e+00 Pa


In [32]:
# =============================================================================
# CELL 6 — DATAFRAME OUTPUT
# =============================================================================
df = pd.DataFrame(rows)

df_display = df.copy()
for col in df_display.columns:
    if col in ('Rate-Limiting', 'GB enhancement', 'Error'):
        continue
    if df_display[col].dtype == float:
        if 'fraction' in col.lower():
            df_display[col] = df_display[col].round(2)
        elif 'theta' in col.lower():
            df_display[col] = df_display[col].round(6)
        elif 'D_eff/D_m' in col:
            df_display[col] = df_display[col].round(6)
        else:
            df_display[col] = df_display[col].apply(
                lambda x: f'{x:.4e}' if pd.notna(x) else 'NaN'
            )

n_surface_df = (df['Rate-Limiting'] == 'SURFACE').sum()
n_oxide_df   = (df['Rate-Limiting'] == 'OXIDE').sum()
n_metal_df   = (df['Rate-Limiting'] == 'METAL').sum()
n_mixed_df   = (df['Rate-Limiting'] == 'MIXED').sum()
n_error_df   = (df['Rate-Limiting'] == 'ERROR').sum()

print("=" * 60)
print("L2+L4+L6 RESULTS DATAFRAME")
print("=" * 60)
print(f"  Oxide:           {OXIDE_KEY}")
print(f"  Metal:           {METAL_KEY}")
print(f"  T:               {T_op-273.15:.3f}°C  ({T_op} K)")
print(f"  L_ox:            {L_ox*1e6:.3f} μm")
print(f"  L_m:             {L_m*1e3:.3f} mm")
print(f"  GB enhancement:  {gb_label}")
print(f"  grain_size:      {grain_size*1e6:.3f} μm")
print(f"  k_diss:          {k_diss:.3e}  mol/m²/s/Pa")
print(f"  K_eq:            {K_eq:.3e}  Pa⁻¹")
print(f"  α:               {alpha:.3e}  mol/m²/s/Pa^0.5  (oxide)")
print(f"  β_latt:          {beta_L:.3e}  mol/m²/s/Pa^0.5  (lattice)")
print(f"  β_eff:           {beta_eff:.3e}  mol/m²/s/Pa^0.5  ({gb_label})")
print(f"  D_eff/D_m:       {D_eff/D_m:.6f}")

print(f"\n  Total rows:        {len(df)}")
print(f"\n  Resistance regime:")
print(f"    Surface-limited: {n_surface_df}")
print(f"    Oxide-limited:   {n_oxide_df}")
print(f"    Metal-limited:   {n_metal_df}")
print(f"    Mixed:           {n_mixed_df}")
if n_error_df > 0:
    print(f"    Errors:          {n_error_df}  ⚠")

# D_eff statistics across pressure sweep
d_eff_col = df.get('D_eff (m²/s)', pd.Series(dtype=float))
if 'D_eff (m²/s)' in df.columns:
    d_vals = df['D_eff (m²/s)'].dropna()
    if len(d_vals) > 0:
        print(f"\n  D_eff across pressure sweep:")
        print(f"    Min:   {d_vals.min():.4e} m²/s")
        print(f"    Max:   {d_vals.max():.4e} m²/s")
        print(f"    Mean:  {d_vals.mean():.4e} m²/s")
        print(f"    D_m:   {D_m:.4e} m²/s")

if 'D_eff/D_m' in df.columns:
    ratio_vals = df['D_eff/D_m'].dropna()
    if len(ratio_vals) > 0:
        print(f"\n  D_eff/D_m range:  "
              f"{ratio_vals.min():.6f} → {ratio_vals.max():.6f}")

print("=" * 60)

display(df_display)

L2+L4+L6 RESULTS DATAFRAME
  Oxide:           Cr2O3_sample4
  Metal:           metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985
  T:               599.850°C  (873 K)
  L_ox:            1.000 μm
  L_m:             1.000 mm
  GB enhancement:  GB on
  grain_size:      100.000 μm
  k_diss:          2.370e-09  mol/m²/s/Pa
  K_eq:            2.799e-05  Pa⁻¹
  α:               4.002e-09  mol/m²/s/Pa^0.5  (oxide)
  β_latt:          9.534e-09  mol/m²/s/Pa^0.5  (lattice)
  β_eff:           9.535e-09  mol/m²/s/Pa^0.5  (GB on)
  D_eff/D_m:       1.000065

  Total rows:        40

  Resistance regime:
    Surface-limited: 20
    Oxide-limited:   19
    Metal-limited:   0
    Mixed:           1

  D_eff across pressure sweep:
    Min:   2.4868e-10 m²/s
    Max:   2.4868e-10 m²/s
    Mean:  2.4868e-10 m²/s
    D_m:   2.4866e-10 m²/s

  D_eff/D_m range:  1.000065 → 1.000065


,P_up (Pa),P_int (Pa),J_ss (mol/m²/s),θ_surface,D_eff (m²/s),D_eff/D_m,α_oxide,β_eff,R_surface,R_oxide,R_metal,R_total,fraction_surface (%),fraction_oxide (%),fraction_metal (%),Rate-Limiting,GB enhancement
0,1.0000e-07,6.1792e-16,2.3702e-16,4.4487e-10,2.4868e-10,1.000065,4.0017e-09,9.5350e-09,4.2191e+15,2.4990e+08,1.0488e+08,4.2191e+15,100.00,0.00,0.00,SURFACE,GB on
1,2.2855e-07,3.2276e-15,5.4170e-16,1.0167e-09,2.4868e-10,1.000065,4.0017e-09,9.5350e-09,1.8460e+15,2.4990e+08,1.0488e+08,1.8460e+15,100.00,0.00,0.00,SURFACE,GB on
2,5.2233e-07,1.6859e-14,1.2380e-15,2.3237e-09,2.4868e-10,1.000065,4.0017e-09,9.5350e-09,8.0773e+14,2.4990e+08,1.0488e+08,8.0773e+14,100.00,0.00,0.00,SURFACE,GB on
3,1.1938e-06,8.8059e-14,2.8295e-15,5.3107e-09,2.4868e-10,1.000065,4.0017e-09,9.5350e-09,3.5342e+14,2.4990e+08,1.0488e+08,3.5342e+14,100.00,0.00,0.00,SURFACE,GB on
4,2.7283e-06,4.5996e-13,6.4667e-15,1.2137e-08,2.4868e-10,1.000065,4.0017e-09,9.5350e-09,1.5464e+14,2.4990e+08,1.0488e+08,1.5464e+14,100.00,0.00,0.00,SURFACE,GB on
5,6.2355e-06,2.4025e-12,1.4779e-14,2.7740e-08,2.4868e-10,1.000065,4.0017e-09,9.5350e-09,6.7662e+13,2.4990e+08,1.0488e+08,6.7662e+13,100.00,0.00,0.00,SURFACE,GB on
6,1.4251e-05,1.2549e-11,3.3777e-14,6.3397e-08,2.4868e-10,1.000065,4.0017e-09,9.5350e-09,2.9605e+13,2.4990e+08,1.0488e+08,2.9606e+13,100.00,0.00,0.00,SURFACE,GB on
7,3.2570e-05,6.5547e-11,7.7196e-14,1.4489e-07,2.4868e-10,1.000065,4.0017e-09,9.5350e-09,1.2954e+13,2.4990e+08,1.0488e+08,1.2954e+13,100.00,0.00,0.00,SURFACE,GB on
8,7.4438e-05,3.4235e-10,1.7642e-13,3.3113e-07,2.4868e-10,1.000065,4.0017e-09,9.5350e-09,5.6679e+12,2.4990e+08,1.0488e+08,5.6683e+12,99.99,0.00,0.00,SURFACE,GB on
9,1.7013e-04,1.7880e-09,4.0318e-13,7.5674e-07,2.4868e-10,1.000065,4.0017e-09,9.5350e-09,2.4800e+12,2.4990e+08,1.0488e+08,2.4803e+12,99.99,0.01,0.00,SURFACE,GB on


## L3+L4+L6: Defective Oxide + Defective Metal + Surface Kinetics

In [33]:
# =============================================================================
# CELL 1 — SETUP
# Run once. All imports and helper functions.
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import brentq
import pandas as pd
from itertools import groupby
from operator import itemgetter
from IPython.display import display

from calculations.surface_kinetics import (
    calculate_full_model_flux_L346_v2,
    calculate_mixed_defect_flux_L6,
    calculate_defective_metal_flux_L6,
    solve_steady_state_flux_direct,
)
from calculations.defective_metal import metal_diffusivity
from calculations.config.model_config import (
    R, F,
    METALS, OXIDES, CONDITIONS, MICROSTRUCTURE, MODEL_OPTIONS, SUGGESTED_RANGES_LEVEL5,
    PLOT_STYLE as STYLE,
    CURVE_STYLES,
    apply_style,
    build_simulation_config,
)

# ── Simulation config ─────────────────────────────────────────────────────────
SIM       = build_simulation_config()

METAL_KEY = SIM['metal_name']
OXIDE_KEY = SIM['oxide_name']
METAL     = METALS[METAL_KEY]
OXIDE     = OXIDES[OXIDE_KEY]
SK_OX     = OXIDE['surface_kinetics']
SK_M      = METAL['surface_kinetics']

# ── Pressure sweep ────────────────────────────────────────────────────────────
P_RANGE_ARR = np.logspace(
    np.log10(CONDITIONS['P_range'][0]),
    np.log10(CONDITIONS['P_range'][1]),
    CONDITIONS['n_points']
)
P_DOWN      = SIM['P_downstream']
T_RANGE     = CONDITIONS['T_range']
N_T         = CONDITIONS['n_points']

# ── Arrhenius helpers — oxide surface ────────────────────────────────────────
def get_k_diss_at_T(T_K):
    return SK_OX['k_diss_ref'] * np.exp(
        (-SK_OX['E_diss'] / R) * (1/T_K - 1/SK_OX['T_ref'])
    )

def get_K_eq_at_T(T_K):
    return SK_OX['K_eq_ref'] * np.exp(
        (-SK_OX['H_eq'] / R) * (1/T_K - 1/SK_OX['T_ref'])
    )

def get_D_ox_at_T(T_K):
    return OXIDE['D_ox_ref'] * np.exp(
        (-OXIDE['E_D_ox'] / R) * (1/T_K - 1/OXIDE['T_ref'])
    )

def get_K_ox_at_T(T_K):
    return OXIDE['K_ox_ref'] * np.exp(
        (-OXIDE['H_sol_ox'] / R) * (1/T_K - 1/OXIDE['T_ref'])
    )

# ── Arrhenius helpers — metal ────────────────────────────────────────────────
def get_D_m_at_T(T_K):
    return METAL['D_ref'] * np.exp(
        (-METAL['E_D'] / R) * (1/T_K - 1/METAL['T_ref'])
    )

def get_K_s_m_at_T(T_K):
    return METAL['K_s_ref'] * np.exp(
        (-METAL['H_s'] / R) * (1/T_K - 1/METAL['T_ref'])
    )

def get_k_diss_metal_at_T(T_K):
    return SK_M['k_diss_metal_ref'] * np.exp(
        (-SK_M['E_diss_metal'] / R) * (1/T_K - 1/SK_M['T_ref'])
    )

def get_K_eq_metal_at_T(T_K):
    return SK_M['K_eq_metal_ref'] * np.exp(
        (-SK_M['H_eq_metal'] / R) * (1/T_K - 1/SK_M['T_ref'])
    )

# ── D_eff helper ──────────────────────────────────────────────────────────────
def get_D_eff_at_T(T_K, gb_on, microstructure):
    """Metal diffusivity at T_K, with or without grain-boundary enhancement."""
    return metal_diffusivity(get_D_m_at_T(T_K), T_K, microstructure,
                             enable_gb_enhancement=gb_on)

# ── Reference activation energies ────────────────────────────────────────────
E_REF_SURFACE = SK_OX['E_diss']
E_REF_OXIDE   = OXIDE['E_D_ox'] + OXIDE['H_sol_ox']
E_REF_METAL   = METAL['E_D']    + METAL['H_s']

print("=" * 60)
print("SETUP COMPLETE")
print("=" * 60)
print(f"  Metal:         {METAL_KEY}")
print(f"  Oxide:         {OXIDE_KEY}")
print(f"  E_ref surface: {E_REF_SURFACE/1000:.3f} kJ/mol")
print(f"  E_ref oxide:   {E_REF_OXIDE/1000:.3f} kJ/mol")
print(f"  E_ref metal:   {E_REF_METAL/1000:.3f} kJ/mol")
print(f"  P range:       {P_RANGE_ARR[0]:.0e} – {P_RANGE_ARR[-1]:.0e} Pa")
print(f"  T range:       {T_RANGE[0]} – {T_RANGE[1]} K")

SETUP COMPLETE
  Metal:         metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985
  Oxide:         Cr2O3_sample4
  E_ref surface: 57.950 kJ/mol
  E_ref oxide:   234.000 kJ/mol
  E_ref metal:   79.400 kJ/mol
  P range:       1e-07 – 1e+07 Pa
  T range:       573 – 1273 K


In [34]:
# =============================================================================
# CELL 2 — PARAMETERS
# All physics values at operating temperature, pulled from config.
# =============================================================================

# ── Operating conditions ──────────────────────────────────────────────────────
T_op   = SIM['T_operating']
L_ox   = SIM['L_oxide']
L_m    = SIM['L_metal']
P_down = SIM['P_downstream']

# ── Surface kinetics at T_op ──────────────────────────────────────────────────
k_diss   = get_k_diss_at_T(T_op)
K_eq     = get_K_eq_at_T(T_op)
k_recomb = k_diss / K_eq

# ── Oxide transport at T_op ───────────────────────────────────────────────────
D_ox = get_D_ox_at_T(T_op)
K_ox = get_K_ox_at_T(T_op)

# ── Metal lattice transport at T_op ───────────────────────────────────────────
D_m   = get_D_m_at_T(T_op)
K_s_m = get_K_s_m_at_T(T_op)

# ── Metal surface kinetics — for pinhole paths ────────────────────────────────
k_diss_metal = get_k_diss_metal_at_T(T_op)
K_eq_metal   = get_K_eq_metal_at_T(T_op)

# ── Microstructure from config ────────────────────────────────────────────────
microstructure = SIM['microstructure']
gb_on          = MODEL_OPTIONS['enable_gb_enhancement']
gb_label       = 'GB on' if gb_on else 'GB off'
grain_size     = microstructure['grain_size']

# ── D_eff at T_op — GB enhancement off and on ────────────────────────────────
D_eff_modes = {}
for m in (False, True):
    D_eff_modes[m] = get_D_eff_at_T(T_op, m, microstructure)
D_eff = D_eff_modes[gb_on]

# ── Defect configuration from SIM ────────────────────────────────────────────
defect_config        = SIM['defect_config']
use_sieverts_pinhole = SIM['oxide_defects'].get('use_sieverts_pinhole', False)
k_diss_metal_arg     = None if use_sieverts_pinhole else k_diss_metal
K_eq_metal_arg       = None if use_sieverts_pinhole else K_eq_metal

# ── Derived ───────────────────────────────────────────────────────────────────
alpha_intact = D_ox * K_ox / L_ox
beta_L       = D_m   * K_s_m / L_m
beta_eff     = D_eff * K_s_m / L_m

# ── Validation parameters ─────────────────────────────────────────────────────
f_limit_check = CONDITIONS['defect_limit_check']

# ── Print verification ────────────────────────────────────────────────────────
print("=" * 60)
print(f"L3+L4+L6 PARAMETERS AT T = {T_op-273.15:.3f}°C  ({T_op} K)")
print("=" * 60)

print(f"\n  Geometry")
print(f"    L_ox   = {L_ox:.3e}  m")
print(f"    L_m    = {L_m:.3e}  m")
print(f"    P_down = {P_down:.1e}  Pa")

print(f"\n  Oxide surface kinetics  [config: oxide surface_kinetics]")
print(f"    k_diss   = {k_diss:.3e}  mol/m²/s/Pa")
print(f"    K_eq     = {K_eq:.3e}  Pa⁻¹")

print(f"\n  Oxide transport  [config: Arrhenius]")
print(f"    D_ox   = {D_ox:.3e}  m²/s")
print(f"    K_ox   = {K_ox:.3e}  mol/m³/Pa^0.5")
print(f"    α      = {alpha_intact:.3e}  mol/m²/s/Pa^0.5")

print(f"\n  Defect configuration  [config: OXIDE_DEFECTS]")
total_f = sum(v['area_fraction'] for v in defect_config.values())
for path, cfg in defect_config.items():
    extras = ''
    if 'thickness_factor' in cfg:
        extras = f"  γ={cfg['thickness_factor']}"
    if 'diffusivity_factor' in cfg:
        extras = f"  δ={cfg['diffusivity_factor']}"
    print(f"    {path:<16} f = {cfg['area_fraction']*100:.3f}%{extras}")
print(f"    {'intact':<16} f = {(1-total_f)*100:.3f}%")
print(f"    Total defect fraction = {total_f*100:.3f}%")

print(f"\n  Metal lattice transport  [config: Arrhenius]")
print(f"    D_m    = {D_m:.3e}  m²/s")
print(f"    K_s_m  = {K_s_m:.3e}  mol/m³/Pa^0.5")

print(f"\n  Microstructure  [config: MICROSTRUCTURE]")
print(f"    GB enhancement = {gb_label}  [config: MODEL_OPTIONS]")
print(f"    grain_size = {grain_size*1e6:.3f} μm")

print(f"\n  D_eff at T_op — GB off / on")
for m, D in D_eff_modes.items():
    marker = ' ← active' if m == gb_on else ''
    print(f"    GB {'on ' if m else 'off'}  D_eff/D_m = {D/D_m:.4f}{marker}")

print(f"\n  Derived permeances")
print(f"    α_intact = {alpha_intact:.3e}  mol/m²/s/Pa^0.5")
print(f"    β_latt   = {beta_L:.3e}  mol/m²/s/Pa^0.5")
print(f"    β_eff    = {beta_eff:.3e}  mol/m²/s/Pa^0.5  ({gb_label})")

print(f"\n  Pinhole mode: "
      f"{'Sieverts limit' if use_sieverts_pinhole else 'Metal surface kinetics'}")

if total_f > 0.5:
    print(f"\n  ⚠  WARNING: total defect fraction = {total_f*100:.3f}%")
if total_f == 0:
    print(f"\n  ⚠  WARNING: all defect fractions zero — L3+L4+L6 = L2+L4+L6")
if not gb_on:
    print(f"\n  ⚠  GB enhancement off — L3+L4+L6 = L3+L6")

print("\n" + "=" * 60)

L3+L4+L6 PARAMETERS AT T = 599.850°C  (873 K)

  Geometry
    L_ox   = 1.000e-06  m
    L_m    = 1.000e-03  m
    P_down = 0.0e+00  Pa

  Oxide surface kinetics  [config: oxide surface_kinetics]
    k_diss   = 2.370e-09  mol/m²/s/Pa
    K_eq     = 2.799e-05  Pa⁻¹

  Oxide transport  [config: Arrhenius]
    D_ox   = 1.395e-17  m²/s
    K_ox   = 2.869e+02  mol/m³/Pa^0.5
    α      = 4.002e-09  mol/m²/s/Pa^0.5

  Defect configuration  [config: OXIDE_DEFECTS]
    pinhole          f = 1.000%
    crack            f = 0.500%  γ=0.1
    grain_boundary   f = 0.500%  δ=10
    intact           f = 98.000%
    Total defect fraction = 2.000%

  Metal lattice transport  [config: Arrhenius]
    D_m    = 2.487e-10  m²/s
    K_s_m  = 3.834e-02  mol/m³/Pa^0.5

  Microstructure  [config: MICROSTRUCTURE]
    GB enhancement = GB on  [config: MODEL_OPTIONS]
    grain_size = 100.000 μm

  D_eff at T_op — GB off / on
    GB off  D_eff/D_m = 1.0000
    GB on   D_eff/D_m = 1.0001 ← active

  Derived permeances


/Users/rwest/CoMoChEng Dropbox/Richard West/Northeastern/Research/Research-MHI-2026-hydrogen/Continuum_H_permeation/calculations/defective_metal.py:222: UserWarning: Temperature 873.0 K is outside the validated range [1023.0, 1223.0] K of metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985's diffusion data. The GB enhancement is extrapolated.
  warnings.warn(


In [35]:
# =============================================================================
# CELL 3 — COMPUTE
# Single-pass loop over full pressure range.
# =============================================================================

rows             = []
J_arr            = np.full(len(P_RANGE_ARR), np.nan)
J_intact_arr     = np.full(len(P_RANGE_ARR), np.nan)
J_pinhole_arr    = np.full(len(P_RANGE_ARR), np.nan)
J_crack_arr      = np.full(len(P_RANGE_ARR), np.nan)
J_gb_arr         = np.full(len(P_RANGE_ARR), np.nan)
frac_intact_arr  = np.full(len(P_RANGE_ARR), np.nan)
frac_pinhole_arr = np.full(len(P_RANGE_ARR), np.nan)
frac_crack_arr   = np.full(len(P_RANGE_ARR), np.nan)
frac_gb_arr      = np.full(len(P_RANGE_ARR), np.nan)
enhancement_arr  = np.full(len(P_RANGE_ARR), np.nan)
D_eff_avg_arr    = np.full(len(P_RANGE_ARR), np.nan)
frac_surface_arr = np.full(len(P_RANGE_ARR), np.nan)
frac_oxide_arr   = np.full(len(P_RANGE_ARR), np.nan)
frac_metal_arr   = np.full(len(P_RANGE_ARR), np.nan)
dominant_path_arr= np.empty(len(P_RANGE_ARR), dtype=object)
rate_lim_arr     = np.empty(len(P_RANGE_ARR), dtype=object)

for i, P_up in enumerate(P_RANGE_ARR):
    try:
        r = calculate_full_model_flux_L346_v2(
            P_up, P_down, L_m, T_op,
            k_diss, K_eq,
            D_ox, K_ox, L_ox,
            D_m, K_s_m,
            microstructure,
            defect_config,
            k_diss_metal=k_diss_metal_arg,
            K_eq_metal=K_eq_metal_arg,
            enable_gb_enhancement=gb_on,
        )

        fb  = r['flux_breakdown']
        dom = r['dominant_path']
        res = r.get('flux_weighted_resistances')

        J_arr[i]            = r['J_total']
        J_intact_arr[i]     = fb.get('intact',         {}).get('flux', np.nan)
        J_pinhole_arr[i]    = fb.get('pinhole',         {}).get('flux', np.nan)
        J_crack_arr[i]      = fb.get('crack',           {}).get('flux', np.nan)
        J_gb_arr[i]         = fb.get('grain_boundary',  {}).get('flux', np.nan)
        frac_intact_arr[i]  = fb.get('intact',         {}).get('fraction_of_total', 0.0)
        frac_pinhole_arr[i] = fb.get('pinhole',         {}).get('fraction_of_total', 0.0)
        frac_crack_arr[i]   = fb.get('crack',           {}).get('fraction_of_total', 0.0)
        frac_gb_arr[i]      = fb.get('grain_boundary',  {}).get('fraction_of_total', 0.0)
        enhancement_arr[i]  = r['enhancement_vs_intact']
        D_eff_avg_arr[i]    = r['D_eff_avg']
        dominant_path_arr[i]= dom
        rate_lim_arr[i]     = r['system_rate_limiting']

        if res is not None:
            frac_surface_arr[i] = res['fraction_surface']
            frac_oxide_arr[i]   = res['fraction_oxide']
            frac_metal_arr[i]   = res['fraction_metal']

        rows.append({
            'P_up (Pa)':            P_up,
            'J_total (mol/m²/s)':   r['J_total'],
            'J_intact':             J_intact_arr[i],
            'J_pinhole':            J_pinhole_arr[i],
            'J_crack':              J_crack_arr[i],
            'J_grain_boundary':     J_gb_arr[i],
            'frac_intact (%)':      frac_intact_arr[i]  * 100,
            'frac_pinhole (%)':     frac_pinhole_arr[i] * 100,
            'frac_crack (%)':       frac_crack_arr[i]   * 100,
            'frac_GB (%)':          frac_gb_arr[i]      * 100,
            'enhancement_factor':   r['enhancement_vs_intact'],
            'D_eff_avg (m²/s)':     r['D_eff_avg'],
            'D_eff_avg/D_m':        r['D_eff_avg'] / D_m,
            'dominant_path':        dom,
            'fraction_surface (%)': frac_surface_arr[i] * 100
                                    if not np.isnan(frac_surface_arr[i]) else np.nan,
            'fraction_oxide (%)':   frac_oxide_arr[i]   * 100
                                    if not np.isnan(frac_oxide_arr[i])   else np.nan,
            'fraction_metal (%)':   frac_metal_arr[i]   * 100
                                    if not np.isnan(frac_metal_arr[i])   else np.nan,
            'system_rate_limiting': r['system_rate_limiting'].upper(),
            'GB enhancement':       gb_label,
        })

    except Exception as e:
        dominant_path_arr[i] = 'error'
        rate_lim_arr[i]      = 'error'
        rows.append({
            'P_up (Pa)':           P_up,
            'system_rate_limiting': 'ERROR',
            'GB enhancement':       gb_label,
            'Error':                str(e),
        })

# ── Derived arrays ────────────────────────────────────────────────────────────
valid = ~np.isnan(J_arr)

# Only classify where flux_weighted_resistances was not None
res_valid = valid & ~np.isnan(frac_surface_arr)

rl_arr = np.where(
    res_valid & (frac_surface_arr > 0.5), 'surface',
    np.where(
        res_valid & (frac_oxide_arr > 0.5), 'oxide',
        np.where(
            res_valid & (frac_metal_arr > 0.5), 'metal', 'mixed'
        )
    )
)

# ── Regime summary ────────────────────────────────────────────────────────────
n_surface = np.sum((frac_surface_arr > 0.5) & res_valid)
n_oxide   = np.sum((frac_oxide_arr   > 0.5) & res_valid)
n_metal   = np.sum((frac_metal_arr   > 0.5) & res_valid)
n_mixed   = np.sum(
    (frac_surface_arr <= 0.5) & (frac_oxide_arr <= 0.5) &
    (frac_metal_arr   <= 0.5) & res_valid
)

surf_dom_mask  = (frac_surface_arr > 0.90) & res_valid
oxide_dom_mask = (frac_oxide_arr   > 0.90) & res_valid
metal_dom_mask = (frac_metal_arr   > 0.90) & res_valid
P_surf_dom     = P_RANGE_ARR[surf_dom_mask]
P_oxide_dom    = P_RANGE_ARR[oxide_dom_mask]
P_metal_dom    = P_RANGE_ARR[metal_dom_mask]

P_for_surface_Ea = np.median(P_surf_dom)  if len(P_surf_dom)  > 3 else None
P_for_oxide_Ea   = np.median(P_oxide_dom) if len(P_oxide_dom) > 3 else None
P_for_metal_Ea   = np.median(P_metal_dom) if len(P_metal_dom) > 3 else None

# ── Crossover pressures ───────────────────────────────────────────────────────
def find_crossover(frac_a, frac_b, valid_mask):
    vc   = valid_mask & ~np.isnan(frac_a) & ~np.isnan(frac_b)
    if np.sum(vc) < 2:
        return None
    diff = frac_a[vc] - frac_b[vc]
    Pv   = P_RANGE_ARR[vc]
    sc   = np.where(np.diff(np.sign(diff)))[0]
    if len(sc) == 0:
        return None
    idx = sc[0]
    return Pv[idx] if diff[idx] > 0 else None

P_cross_surf_oxide  = find_crossover(frac_surface_arr, frac_oxide_arr,  res_valid)
P_cross_surf_metal  = find_crossover(frac_surface_arr, frac_metal_arr,  res_valid)
P_cross_oxide_metal = find_crossover(frac_oxide_arr,   frac_metal_arr,  res_valid)

# ── Dominant path summary ─────────────────────────────────────────────────────
dom_counts = {}
for p in ['intact', 'pinhole', 'crack', 'grain_boundary', 'mixed', 'error']:
    dom_counts[p] = np.sum(dominant_path_arr == p)

# ── Print regime summary ──────────────────────────────────────────────────────
print("=" * 60)
print(f"L3+L4+L6 REGIME SUMMARY")
print(f"T = {T_op-273.15:.3f}°C  |  {gb_label}  |  "
      f"{np.sum(valid)} valid points")
print("=" * 60)

print(f"\n  Resistance regime (dominant path, fraction > 50%)")
print(f"    Surface-limited: {n_surface:3d}  points")
print(f"    Oxide-limited:   {n_oxide:3d}  points")
print(f"    Metal-limited:   {n_metal:3d}  points")
print(f"    Mixed (no res):  {n_mixed:3d}  points")

print(f"\n  Dominant oxide path")
for path, count in dom_counts.items():
    if count > 0:
        print(f"    {path:<18}  {count:3d} points")

print(f"\n  Dominant regions (fraction > 90%) — used for Arrhenius")
for label, dom, P_ea in [
    ('Surface', P_surf_dom,  P_for_surface_Ea),
    ('Oxide',   P_oxide_dom, P_for_oxide_Ea),
    ('Metal',   P_metal_dom, P_for_metal_Ea),
]:
    if len(dom) > 3:
        print(f"    {label}-dominant: {len(dom):3d} pts  "
              f"P = {dom.min():.1e} – {dom.max():.1e} Pa  "
              f"→ {P_ea:.2e} Pa")
    else:
        print(f"    {label}-dominant: {len(dom):3d} pts  "
              f"(insufficient — {label.lower()} Ea not extractable)")

print(f"\n  Crossover pressures (diagnostic only)")
for desc, P_co in [
    ('Surface → Oxide', P_cross_surf_oxide),
    ('Surface → Metal', P_cross_surf_metal),
    ('Oxide   → Metal', P_cross_oxide_metal),
]:
    val = f'{P_co:.2e} Pa' if P_co is not None else 'not found'
    print(f"    {desc}:  {val}")

enh_valid = enhancement_arr[valid & ~np.isnan(enhancement_arr)]
if len(enh_valid) > 0:
    print(f"\n  Enhancement (J_total / J_intact):")
    print(f"    Min = {enh_valid.min():.3f}  Max = {enh_valid.max():.3f}")

if np.sum(valid) > 0:
    print(f"\n  J range: {np.nanmin(J_arr):.2e} → {np.nanmax(J_arr):.2e} mol/m²/s")
else:
    print(f"\n  ⚠  No valid points — check solver signature and config.")
print("=" * 60)

L3+L4+L6 REGIME SUMMARY
T = 599.850°C  |  GB on  |  40 valid points

  Resistance regime (dominant path, fraction > 50%)
    Surface-limited:  20  points
    Oxide-limited:    18  points
    Metal-limited:     0  points
    Mixed (no res):    2  points

  Dominant oxide path
    intact               26 points
    mixed                14 points

  Dominant regions (fraction > 90%) — used for Arrhenius
    Surface-dominant:  15 pts  P = 1.0e-07 – 1.1e-02 Pa  → 3.26e-05 Pa
    Oxide-dominant:   0 pts  (insufficient — oxide Ea not extractable)
    Metal-dominant:   0 pts  (insufficient — metal Ea not extractable)

  Crossover pressures (diagnostic only)
    Surface → Oxide:  1.51e+00 Pa
    Surface → Metal:  1.51e+00 Pa
    Oxide   → Metal:  not found

  Enhancement (J_total / J_intact):
    Min = 1.040  Max = 2.935

  J range: 6.96e-16 → 8.79e-06 mol/m²/s


In [36]:
# =============================================================================
# CELL 4 — FIGURE 1: Core Validation (2×2)
# =============================================================================
"""
Figure 1 validates the L3+L4+L6 model from four angles at operating
temperature. The key question is whether oxide defects and metal
microstructure act additively, multiplicatively, or whether one dominates.

Panel (A) — Flux vs Pressure: Full Hierarchy Sweep
    Question: How does each level of physical complexity shift the flux
    curve, and what is the combined effect of defective oxide + defective
    metal + surface kinetics?
    Five curves in order of complexity:
      - L2+L6      (f=0, GB off)  — black solid — pure series resistance
      - L3+L6      (f=config, GB off) — oxide defects only
      - L2+L4+L6   (f=0, GB=config)   — metal microstructure only
      - L3+L4+L6   (f=config, GB=config) — both active (config run)
    Each curve has colored regime overlays and slope annotation boxes
    using the identical pattern from all previous levels. The separation
    between L3+L6 and L2+L4+L6 curves shows which effect dominates at
    each pressure.

Panel (B) — Enhancement Decomposition vs Pressure
    Question: Are the effects of oxide defects and metal microstructure
    additive, multiplicative, or does one dominate?
    Plots three enhancement ratios vs pressure on a semilogy scale:
      - E_oxide  = J_L3+L6(f=config, GB off) / J_L2+L6
      - E_metal  = J_L2+L4+L6(f=0, GB=config) / J_L2+L6
      - E_both   = J_L3+L4+L6 / J_L2+L6
    A horizontal line at E=1 marks no enhancement. If E_both ≈ E_oxide ×
    E_metal the effects multiply. If E_both ≈ max(E_oxide, E_metal) one
    dominates. The annotation box states which behaviour is observed.

Panel (C) — Resistance Fractions vs Pressure
    Consistent style with all previous levels. Uses flux_weighted_resistances
    from the dominant path. Note annotation states which path is dominant
    and flags when flux_weighted_resistances is None (mixed dominant path).

Panel (D) — Limit Check: two limits on one parity plot
    Question: Does L3+L4+L6 recover L2+L4+L6 when f→0, and recover
    L3+L6 when GB enhancement is off?
    Two parity comparisons on the same axes:
      - Circles: J_L3+L4+L6(f→0) vs J_L2+L4+L6  (oxide defects off)
      - Squares: J_L3+L4+L6(GB off) vs J_L3+L6 (metal microstructure off)
    Both should fall on the diagonal. Green annotation if both < 1%.
    Validates hierarchical consistency: L3+L4+L6 collapses to the
    correct sub-model when each physical effect is individually disabled.
"""

props = dict(boxstyle='round', facecolor='wheat', alpha=0.9)

fig, axes = plt.subplots(2, 2, figsize=STYLE['figsize'])
fig.suptitle(
    f'L3+L4+L6: Full Model — Core Validation\n'
    f'{OXIDE_KEY} + {METAL_KEY}  |  T = {T_op-273.15:.3f}°C  |  '
    f'{gb_label}  |  defect={sum(v["area_fraction"] for v in defect_config.values())*100:.2f}%',
    fontsize=STYLE['fontsize_suptitle']-1, fontweight='bold', y=0.98
)

# ── Regime overlay helper ─────────────────────────────────────────────────────
def plot_regime_overlay(ax, P_arr, J_arr, frac_s, frac_o, frac_m, valid_mask):
    rl = np.where(frac_s > 0.5, 'surface',
         np.where(frac_o > 0.5, 'oxide',
         np.where(frac_m > 0.5, 'metal', 'mixed')))

    region_styles = [
        ('surface', CURVE_STYLES['surface_region']['color'], 'Surface-limited'),
        ('oxide',   CURVE_STYLES['oxide_region']['color'],   'Oxide-limited'),
        ('metal',   CURVE_STYLES['metal_region']['color'],   'Metal-limited'),
        ('mixed',   CURVE_STYLES['mixed_region']['color'],   'Mixed'),
    ]

    for regime, color, lbl in region_styles:
        mask = (rl == regime) & valid_mask
        if not np.any(mask):
            continue
        idxs = np.where(mask)[0]
        for _, g in groupby(enumerate(idxs), lambda x: x[0] - x[1]):
            group = list(map(itemgetter(1), g))
            if len(group) < 3:
                continue
            P_seg = P_arr[group]
            J_seg = J_arr[group]
            ax.loglog(P_seg, J_seg, color=color, lw=4, alpha=0.7)
            slope_seg, *_ = stats.linregress(
                np.log10(P_seg), np.log10(np.abs(J_seg))
            )
            mid = len(group) // 2
            ax.text(
                P_seg[mid], J_seg[mid],
                f'{lbl}\nSlope={slope_seg:.2f}',
                color=color, fontsize=STYLE['fontsize_annotation']-1,
                fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
            )

# ─────────────────────────────────────────────────────────────────────────────
# (A) FLUX vs PRESSURE — full hierarchy sweep
# ─────────────────────────────────────────────────────────────────────────────
ax1 = axes[0, 0]
P_A = np.logspace(
    np.log10(CONDITIONS['P_range'][0]),
    np.log10(CONDITIONS['P_range'][1]),
    CONDITIONS['n_points']
)

# Build zero-defect config for f=0 cases
dc_zero = {
    path: {**cfg, 'area_fraction': f_limit_check}
    for path, cfg in defect_config.items()
}

hierarchy_cases = [
    # (label, solver_fn, kwargs, color, ls)
    (
        'L2+L6  (f=0, GB off)',
        'L2L6', None, 'black', '-',
    ),
    (
        'L3+L6  (f=config, GB off)',
        'L3L6', None,
        CURVE_STYLES['L3_L6']['color'],
        CURVE_STYLES['L3_L6']['ls'],
    ),
    (
        f'L2+L4+L6  (f=0, {gb_label})',
        'L2L4L6', None,
        CURVE_STYLES['L4_gb']['color'],
        '--',
    ),
    (
        f'L3+L4+L6  (both active)',
        'L3L4L6', None,
        CURVE_STYLES['L3_L4_L6']['color'] if 'L3_L4_L6' in CURVE_STYLES
        else 'crimson',
        '-',
    ),
]

# Storage for enhancement computation in panel B
J_curves = {}

for label, solver_key, _, color, ls in hierarchy_cases:
    J_h  = np.full(len(P_A), np.nan)
    fs_h = np.full(len(P_A), np.nan)
    fo_h = np.full(len(P_A), np.nan)
    fm_h = np.full(len(P_A), np.nan)

    for i, P_up in enumerate(P_A):
        try:
            if solver_key == 'L2L6':
                r = solve_steady_state_flux_direct(
                    P_up, P_down, L_m,
                    k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m
                )
                J_h[i]  = r['J_ss']
                fs_h[i] = r['resistances']['fraction_surface']
                fo_h[i] = r['resistances']['fraction_oxide']
                fm_h[i] = r['resistances']['fraction_metal']

            elif solver_key == 'L3L6':
                r = calculate_mixed_defect_flux_L6(
                    P_up, P_down, L_m,
                    k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m,
                    defect_config,
                    k_diss_metal=k_diss_metal_arg,
                    K_eq_metal=K_eq_metal_arg,
                )
                J_h[i] = r['J_total']
                dom = r.get('dominant_path', 'intact')
                if dom == 'intact':
                    res = r['intact_path']['resistances']
                elif dom in r.get('defect_paths', {}):
                    res = r['defect_paths'][dom]['path_result']['resistances']
                else:
                    res = r['intact_path']['resistances']
                fs_h[i] = res.get('fraction_surface', np.nan)
                fo_h[i] = res.get('fraction_oxide',   np.nan)
                fm_h[i] = res.get('fraction_metal',   np.nan)

            elif solver_key == 'L2L4L6':
                r = calculate_defective_metal_flux_L6(
                    P_up, P_down, L_m, T_op,
                    k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m,
                    microstructure_params=microstructure,
                    enable_gb_enhancement=gb_on,
                )
                J_h[i]  = r['flux']
                fs_h[i] = r['resistances']['fraction_surface']
                fo_h[i] = r['resistances']['fraction_oxide']
                fm_h[i] = r['resistances']['fraction_metal']

            elif solver_key == 'L3L4L6':
                r = calculate_full_model_flux_L346_v2(
                    P_up, P_down, L_m, T_op,
                    k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m,
                    microstructure, defect_config,
                    k_diss_metal=k_diss_metal_arg,
                    K_eq_metal=K_eq_metal_arg,
                    enable_gb_enhancement=gb_on,
                )
                J_h[i] = r['J_total']
                res = r.get('flux_weighted_resistances')
                if res is not None:
                    fs_h[i] = res['fraction_surface']
                    fo_h[i] = res['fraction_oxide']
                    fm_h[i] = res['fraction_metal']

        except Exception:
            pass

    J_curves[solver_key] = J_h.copy()

    v_h = ~np.isnan(J_h) & (J_h > 0)
    if not np.any(v_h):
        print(f"  ⚠  {label}: no valid points")
        continue

    lw = STYLE['linewidth'] if solver_key == 'L2L6' else STYLE['linewidth'] - 0.5
    ax1.loglog(
        P_A[v_h], J_h[v_h],
        color=color, ls=ls, lw=lw,
        alpha=1.0 if solver_key in ('L2L6', 'L3L4L6') else 0.75,
        label=label
    )
    plot_regime_overlay(ax1, P_A, J_h, fs_h, fo_h, fm_h, v_h)

# Net slope
v_net = ~np.isnan(J_arr) & (J_arr > 0)
if np.sum(v_net) > 2:
    slope_net, *_ = stats.linregress(
        np.log10(P_RANGE_ARR[v_net]), np.log10(np.abs(J_arr[v_net]))
    )
    ax1.text(
        0.98, 0.02, f'Net slope (L3+L4+L6) = {slope_net:.2f}',
        transform=ax1.transAxes, ha='right', va='bottom',
        fontsize=STYLE['fontsize_annotation'], fontweight='bold',
        bbox=dict(boxstyle='square', fc='wheat', ec='gray', alpha=1)
    )

ax1.set_xlabel('Upstream Pressure $P_{up}$ (Pa)',         fontsize=STYLE['fontsize_axis'])
ax1.set_ylabel('Flux $J$ (mol/m²/s)',                      fontsize=STYLE['fontsize_axis'])
ax1.set_title('(A) Flux vs Pressure: Full Hierarchy Sweep', fontsize=STYLE['fontsize_title'])
ax1.legend(fontsize=STYLE['fontsize_legend']-2, loc='upper left')
ax1.grid(True, which='both', alpha=STYLE['grid_alpha'])
ax1.tick_params(labelsize=STYLE['fontsize_tick'])

# ─────────────────────────────────────────────────────────────────────────────
# (B) ENHANCEMENT DECOMPOSITION vs PRESSURE
# ─────────────────────────────────────────────────────────────────────────────
ax2 = axes[0, 1]

J_L2L6   = J_curves.get('L2L6',   np.full(len(P_A), np.nan))
J_L3L6   = J_curves.get('L3L6',   np.full(len(P_A), np.nan))
J_L2L4L6 = J_curves.get('L2L4L6', np.full(len(P_A), np.nan))
J_L3L4L6 = J_curves.get('L3L4L6', np.full(len(P_A), np.nan))

v_enh = (J_L2L6 > 0) & ~np.isnan(J_L2L6)

E_oxide = np.full(len(P_A), np.nan)
E_metal = np.full(len(P_A), np.nan)
E_both  = np.full(len(P_A), np.nan)
E_prod  = np.full(len(P_A), np.nan)

E_oxide[v_enh] = J_L3L6[v_enh]   / J_L2L6[v_enh]
E_metal[v_enh] = J_L2L4L6[v_enh] / J_L2L6[v_enh]
E_both[v_enh]  = J_L3L4L6[v_enh] / J_L2L6[v_enh]
E_prod[v_enh]  = E_oxide[v_enh] * E_metal[v_enh]

for arr, color, ls, label in [
    (E_oxide, CURVE_STYLES['L3_L6']['color'],   '--', 'E_oxide  (L3+L6 / L2+L6)'),
    (E_metal, CURVE_STYLES['L4_gb']['color'], '-.', 'E_metal  (L2+L4+L6 / L2+L6)'),
    (E_both,  'crimson',                         '-',  'E_both   (L3+L4+L6 / L2+L6)'),
    (E_prod,  'gray',                            ':',  'E_oxide × E_metal  (if multiplicative)'),
]:
    v_a = ~np.isnan(arr) & (arr > 0)
    if not np.any(v_a):
        continue
    ax2.semilogx(P_A[v_a], arr[v_a], color=color, ls=ls,
                 lw=STYLE['linewidth'], label=label)

ax2.axhline(1.0, color='gray', ls='--', lw=1.5, alpha=0.5, label='E = 1 (no enhancement)')

# Annotation: additive vs multiplicative
v_mid = v_enh & ~np.isnan(E_both) & ~np.isnan(E_prod)
if np.any(v_mid):
    ratio = np.nanmedian(E_both[v_mid] / E_prod[v_mid])
    if ratio > 0.8:
        behaviour = 'Multiplicative\n(E_both ≈ E_ox × E_met)'
    elif np.nanmedian(E_both[v_mid]) > np.nanmedian(np.maximum(E_oxide[v_mid], E_metal[v_mid])):
        behaviour = 'Super-multiplicative'
    else:
        behaviour = 'Sub-multiplicative\n(one effect dominates)'
    ax2.text(
        0.05, 0.95, behaviour,
        transform=ax2.transAxes, fontsize=STYLE['fontsize_annotation'],
        va='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9)
    )

ax2.set_xlabel('Upstream Pressure $P_{up}$ (Pa)', fontsize=STYLE['fontsize_axis'])
ax2.set_ylabel('Enhancement (J / J$_{L2+L6}$)',   fontsize=STYLE['fontsize_axis'])
ax2.set_title('(B) Enhancement Decomposition',     fontsize=STYLE['fontsize_title'])
ax2.legend(fontsize=STYLE['fontsize_legend']-2, loc='upper right')
ax2.grid(True, which='both', alpha=STYLE['grid_alpha'])
ax2.tick_params(labelsize=STYLE['fontsize_tick'])

# ─────────────────────────────────────────────────────────────────────────────
# (C) RESISTANCE FRACTIONS vs PRESSURE
# ─────────────────────────────────────────────────────────────────────────────
ax3 = axes[1, 0]

ax3.semilogx(
    P_RANGE_ARR[res_valid], frac_surface_arr[res_valid] * 100,
    color=CURVE_STYLES['surface_region']['color'],
    lw=STYLE['linewidth'], label='Surface (dissociation)'
)
ax3.semilogx(
    P_RANGE_ARR[res_valid], frac_oxide_arr[res_valid] * 100,
    color=CURVE_STYLES['oxide_region']['color'],
    lw=STYLE['linewidth'], label='Oxide (diffusion)'
)
ax3.semilogx(
    P_RANGE_ARR[res_valid], frac_metal_arr[res_valid] * 100,
    color=CURVE_STYLES['metal_region']['color'],
    lw=STYLE['linewidth'], label=f'Metal ({gb_label})'
)
ax3.axhline(
    50,
    color=CURVE_STYLES['threshold']['color'],
    ls=CURVE_STYLES['threshold']['ls'],
    lw=CURVE_STYLES['threshold']['lw'],
    alpha=CURVE_STYLES['threshold']['alpha'],
    label='50% threshold'
)

n_mixed_dom = np.sum(dominant_path_arr == 'mixed')
frac_missing = np.sum(valid) - np.sum(res_valid)

if P_cross_surf_oxide is not None:
    ax3.axvline(P_cross_surf_oxide, color='gray', ls=':', lw=1.5, alpha=0.7)
    ax3.text(
        P_cross_surf_oxide * 1.5, 53,
        f'S→O\n{P_cross_surf_oxide:.1e} Pa',
        fontsize=STYLE['fontsize_annotation']-2, color='gray'
    )
if P_cross_oxide_metal is not None:
    ax3.axvline(P_cross_oxide_metal, color='gray', ls=':', lw=1.5, alpha=0.7)
    ax3.text(
        P_cross_oxide_metal * 1.5, 53,
        f'O→M\n{P_cross_oxide_metal:.1e} Pa',
        fontsize=STYLE['fontsize_annotation']-2, color='gray'
    )

annot_c = f'Fractions from dominant path\n{gb_label}'
if frac_missing > 0:
    annot_c += f'\n{frac_missing} pts: dominant=mixed\n(flux_weighted_resistances=None)'
ax3.text(
    0.05, 0.95, annot_c,
    transform=ax3.transAxes,
    fontsize=STYLE['fontsize_annotation']-1, va='top',
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8)
)

ax3.set_xlabel('Upstream Pressure $P_{up}$ (Pa)', fontsize=STYLE['fontsize_axis'])
ax3.set_ylabel('Resistance Fraction (%)',          fontsize=STYLE['fontsize_axis'])
ax3.set_title('(C) Rate-Limiting Step Analysis',   fontsize=STYLE['fontsize_title'])
ax3.set_ylim(0, 105)
ax3.legend(fontsize=STYLE['fontsize_legend'], loc='center right')
ax3.grid(True, alpha=STYLE['grid_alpha'])
ax3.tick_params(labelsize=STYLE['fontsize_tick'])

# ─────────────────────────────────────────────────────────────────────────────
# (D) LIMIT CHECK — two limits on one parity plot
# ─────────────────────────────────────────────────────────────────────────────
ax4 = axes[1, 1]

P_D = np.logspace(
    np.log10(CONDITIONS['P_range'][0]),
    np.log10(CONDITIONS['P_range'][1]),
    CONDITIONS['n_points']
)

# Limit 1: f→0 should recover L2+L4+L6
J_f0        = np.full(len(P_D), np.nan)   # L3+L4+L6 with f→0
J_L2L4L6_D  = np.full(len(P_D), np.nan)  # L2+L4+L6 reference

# Limit 2: GB off should recover L3+L6
J_mnone     = np.full(len(P_D), np.nan)  # L3+L4+L6 with GB off
J_L3L6_D    = np.full(len(P_D), np.nan)  # L3+L6 reference

for i, P_up in enumerate(P_D):
    try:
        r = calculate_full_model_flux_L346_v2(
            P_up, P_down, L_m, T_op,
            k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m,
            microstructure, dc_zero,
            k_diss_metal=k_diss_metal_arg,
            K_eq_metal=K_eq_metal_arg,
            enable_gb_enhancement=gb_on,
        )
        J_f0[i] = r['J_total']
    except Exception:
        pass

    try:
        r = calculate_defective_metal_flux_L6(
            P_up, P_down, L_m, T_op,
            k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m,
            microstructure_params=microstructure,
            enable_gb_enhancement=gb_on,
        )
        J_L2L4L6_D[i] = r['flux']
    except Exception:
        pass

    try:
        r = calculate_full_model_flux_L346_v2(
            P_up, P_down, L_m, T_op,
            k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m,
            microstructure, defect_config,
            k_diss_metal=k_diss_metal_arg,
            K_eq_metal=K_eq_metal_arg,
            enable_gb_enhancement=False,
        )
        J_mnone[i] = r['J_total']
    except Exception:
        pass

    try:
        r = calculate_mixed_defect_flux_L6(
            P_up, P_down, L_m,
            k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m,
            defect_config,
            k_diss_metal=k_diss_metal_arg,
            K_eq_metal=K_eq_metal_arg,
        )
        J_L3L6_D[i] = r['J_total']
    except Exception:
        pass

# Parity plot — limit 1: circles
vp1 = (~np.isnan(J_f0) & ~np.isnan(J_L2L4L6_D) &
       (J_f0 > 0) & (J_L2L4L6_D > 0))
# Parity plot — limit 2: squares
vp2 = (~np.isnan(J_mnone) & ~np.isnan(J_L3L6_D) &
       (J_mnone > 0) & (J_L3L6_D > 0))

all_J = []
if np.any(vp1):
    all_J += [J_L2L4L6_D[vp1].min(), J_f0[vp1].min(),
              J_L2L4L6_D[vp1].max(), J_f0[vp1].max()]
if np.any(vp2):
    all_J += [J_L3L6_D[vp2].min(), J_mnone[vp2].min(),
              J_L3L6_D[vp2].max(), J_mnone[vp2].max()]

if all_J:
    J_min_p = min(all_J) * 0.5
    J_max_p = max(all_J) * 2.0
    ax4.loglog(
        [J_min_p, J_max_p], [J_min_p, J_max_p],
        color=CURVE_STYLES['parity']['color'],
        ls=CURVE_STYLES['parity']['ls'],
        lw=CURVE_STYLES['parity']['lw'],
        alpha=CURVE_STYLES['parity']['alpha'],
        label='Perfect agreement'
    )

max_err1 = max_err2 = np.nan

if np.any(vp1):
    ax4.loglog(
        J_L2L4L6_D[vp1], J_f0[vp1],
        'o', color=CURVE_STYLES['L2_L6']['color'],
        markersize=6, alpha=0.8,
        label=f'f→0 → L2+L4+L6'
    )
    err1 = np.abs(J_f0[vp1] - J_L2L4L6_D[vp1]) / J_L2L4L6_D[vp1] * 100
    max_err1 = err1.max()

if np.any(vp2):
    ax4.loglog(
        J_L3L6_D[vp2], J_mnone[vp2],
        's', color=CURVE_STYLES['L3_L6']['color'],
        markersize=6, alpha=0.8,
        label=f'GB off → L3+L6'
    )
    err2 = np.abs(J_mnone[vp2] - J_L3L6_D[vp2]) / J_L3L6_D[vp2] * 100
    max_err2 = err2.max()

ok1 = not np.isnan(max_err1) and max_err1 < 1.0
ok2 = not np.isnan(max_err2) and max_err2 < 1.0
box_color = 'lightgreen' if (ok1 and ok2) else 'lightyellow'

annot_d = (
    f'Limit 1: f→0\n'
    f'  Max err = {max_err1:.2e}%  {"✓" if ok1 else "⚠"}\n\n'
    f'Limit 2: GB off\n'
    f'  Max err = {max_err2:.2e}%  {"✓" if ok2 else "⚠"}'
)
ax4.text(
    0.05, 0.95, annot_d,
    transform=ax4.transAxes, fontsize=STYLE['fontsize_annotation'],
    va='top',
    bbox=dict(boxstyle='round', facecolor=box_color, alpha=0.9)
)

ax4.set_xlabel('Reference Flux (mol/m²/s)',   fontsize=STYLE['fontsize_axis'])
ax4.set_ylabel('L3+L4+L6 Flux (mol/m²/s)',   fontsize=STYLE['fontsize_axis'])
ax4.set_title('(D) Limit Check: Two Limits',  fontsize=STYLE['fontsize_title'])
ax4.set_aspect('equal', adjustable='box')
ax4.legend(fontsize=STYLE['fontsize_legend']-1, loc='lower right')
ax4.grid(True, which='both', alpha=STYLE['grid_alpha'])
ax4.tick_params(labelsize=STYLE['fontsize_tick'])

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
save_figure(fig, "L3L4L6_core_validation", FIGS)

print("\n" + "=" * 60)
print("L3+L4+L6 FIGURE 1 — VALIDATION SUMMARY")
print("=" * 60)
if np.sum(v_net) > 2:
    print(f"  Net log-log slope (L3+L4+L6): {slope_net:.4f}")
print(f"  D_eff_avg/D_m at T_op:  "
      f"{np.nanmean(D_eff_avg_arr[valid]):.4f}")
print(f"  Enhancement range:  "
      f"{np.nanmin(enhancement_arr[valid]):.3f} – "
      f"{np.nanmax(enhancement_arr[valid]):.3f}")
print(f"\n  Limit check 1 (f→0 → L2+L4+L6): max err = {max_err1:.2e}%  "
      f"{'✓' if ok1 else '⚠'}")
print(f"  Limit check 2 (GB off → L3+L6): max err = {max_err2:.2e}%  "
      f"{'✓' if ok2 else '⚠'}")

/var/folders/d4/mt_nqf_x42vgxs9t9mw1l59m0000gn/T/ipykernel_20394/626661576.py:517: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



L3+L4+L6 FIGURE 1 — VALIDATION SUMMARY
  Net log-log slope (L3+L4+L6): 0.7172
  D_eff_avg/D_m at T_op:  0.0000
  Enhancement range:  1.040 – 2.935

  Limit check 1 (f→0 → L2+L4+L6): max err = 6.56e-14%  ✓
  Limit check 2 (GB off → L3+L6): max err = 2.14e-14%  ✓


In [38]:
# =============================================================================
# CELL 6 — DATAFRAME OUTPUT
# =============================================================================
df = pd.DataFrame(rows)

df_display = df.copy()
for col in df_display.columns:
    if col in ('system_rate_limiting', 'dominant_path', 'GB enhancement', 'Error'):
        continue
    if df_display[col].dtype == float:
        if 'fraction' in col.lower() and 'frac_' in col.lower():
            df_display[col] = df_display[col].round(2)
        elif 'fraction' in col.lower():
            df_display[col] = df_display[col].round(2)
        elif 'D_eff' in col and '/D_m' in col:
            df_display[col] = df_display[col].round(6)
        elif 'enhancement' in col.lower():
            df_display[col] = df_display[col].round(4)
        else:
            df_display[col] = df_display[col].apply(
                lambda x: f'{x:.4e}' if pd.notna(x) else 'NaN'
            )

n_surface_df = (df['system_rate_limiting'] == 'SURFACE').sum()
n_oxide_df   = (df['system_rate_limiting'] == 'OXIDE').sum()
n_metal_df   = (df['system_rate_limiting'] == 'METAL').sum()
n_mixed_df   = (df['system_rate_limiting'] == 'MIXED').sum()
n_error_df   = (df['system_rate_limiting'] == 'ERROR').sum()

n_intact_df  = (df['dominant_path'] == 'intact').sum()
n_pin_df     = (df['dominant_path'] == 'pinhole').sum()
n_crack_df   = (df['dominant_path'] == 'crack').sum()
n_gb_df      = (df['dominant_path'] == 'grain_boundary').sum()
n_mixdom_df  = (df['dominant_path'] == 'mixed').sum()

print("=" * 60)
print("L3+L4+L6 RESULTS DATAFRAME")
print("=" * 60)
print(f"  Oxide:             {OXIDE_KEY}")
print(f"  Metal:             {METAL_KEY}")
print(f"  T:                 {T_op-273.15:.3f}°C  ({T_op} K)")
print(f"  L_ox:              {L_ox*1e6:.3f} μm")
print(f"  L_m:               {L_m*1e3:.3f} mm")
print(f"  GB enhancement:    {gb_label}")
print(f"  grain_size:        {grain_size*1e6:.3f} μm")
print(f"  k_diss:            {k_diss:.3e}  mol/m²/s/Pa")
print(f"  α_intact:          {alpha_intact:.3e}  mol/m²/s/Pa^0.5")
print(f"  β_eff:             {beta_eff:.3e}  mol/m²/s/Pa^0.5  ({gb_label})")
print(f"  D_eff/D_m:         {D_eff/D_m:.6f}")

print(f"\n  Defect configuration")
for path, cfg in defect_config.items():
    print(f"    {path:<16} f = {cfg['area_fraction']*100:.3f}%")

print(f"\n  Total rows:          {len(df)}")
print(f"\n  Resistance regime (dominant path):")
print(f"    Surface-limited:   {n_surface_df}")
print(f"    Oxide-limited:     {n_oxide_df}")
print(f"    Metal-limited:     {n_metal_df}")
print(f"    Mixed (res=None):  {n_mixed_df}")
if n_error_df > 0:
    print(f"    Errors:            {n_error_df}  ⚠")

print(f"\n  Dominant oxide path:")
print(f"    Intact:            {n_intact_df}")
print(f"    Pinhole:           {n_pin_df}")
print(f"    Crack:             {n_crack_df}")
print(f"    Grain boundary:    {n_gb_df}")
print(f"    Mixed (>1 path):   {n_mixdom_df}")

if 'enhancement_factor' in df.columns:
    enh_vals = df['enhancement_factor'].dropna()
    if len(enh_vals) > 0:
        print(f"\n  Enhancement (J_total / J_intact):")
        print(f"    Min  = {enh_vals.min():.4f}")
        print(f"    Max  = {enh_vals.max():.4f}")
        print(f"    Mean = {enh_vals.mean():.4f}")

if 'D_eff_avg/D_m' in df.columns:
    deff_vals = df['D_eff_avg/D_m'].dropna()
    if len(deff_vals) > 0:
        print(f"\n  D_eff_avg/D_m range:  "
              f"{deff_vals.min():.6f} → {deff_vals.max():.6f}")

print("=" * 60)

display(df_display)

L3+L4+L6 RESULTS DATAFRAME
  Oxide:             Cr2O3_sample4
  Metal:             metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985
  T:                 599.850°C  (873 K)
  L_ox:              1.000 μm
  L_m:               1.000 mm
  GB enhancement:    GB on
  grain_size:        100.000 μm
  k_diss:            2.370e-09  mol/m²/s/Pa
  α_intact:          4.002e-09  mol/m²/s/Pa^0.5
  β_eff:             9.535e-09  mol/m²/s/Pa^0.5  (GB on)
  D_eff/D_m:         1.000065

  Defect configuration
    pinhole          f = 1.000%
    crack            f = 0.500%
    grain_boundary   f = 0.500%

  Total rows:          40

  Resistance regime (dominant path):
    Surface-limited:   6
    Oxide-limited:     19
    Metal-limited:     0
    Mixed (res=None):  15

  Dominant oxide path:
    Intact:            26
    Pinhole:           0
    Crack:             0
    Grain boundary:    0
    Mixed (>1 path):   14

  Enhancement (J_total / J_intact):
    Min  = 1.0401
    Max  = 2.9349
    Mean = 1.5898

 

,P_up (Pa),J_total (mol/m²/s),J_intact,J_pinhole,J_crack,J_grain_boundary,frac_intact (%),frac_pinhole (%),frac_crack (%),frac_GB (%),enhancement_factor,D_eff_avg (m²/s),D_eff_avg/D_m,dominant_path,fraction_surface (%),fraction_oxide (%),fraction_metal (%),system_rate_limiting,GB enhancement
0,1.0000e-07,6.9563e-16,2.3702e-16,4.6098e-14,2.3702e-16,2.3702e-16,3.3391e+01,6.6268e+01,1.7036e-01,1.7036e-01,2.9349,2.4868e-10,1.000065,mixed,100.00,0.00,0.00,MIXED,GB on
1,2.2855e-07,1.5895e-15,5.4170e-16,1.0532e-13,5.4170e-16,5.4170e-16,3.3398e+01,6.6261e+01,1.7040e-01,1.7040e-01,2.9343,2.4868e-10,1.000065,mixed,100.00,0.00,0.00,MIXED,GB on
2,5.2233e-07,3.6311e-15,1.2380e-15,2.4055e-13,1.2380e-15,1.2380e-15,3.3413e+01,6.6246e+01,1.7047e-01,1.7047e-01,2.9330,2.4868e-10,1.000065,mixed,100.00,0.00,0.00,MIXED,GB on
3,1.1938e-06,8.2902e-15,2.8295e-15,5.4890e-13,2.8295e-15,2.8295e-15,3.3448e+01,6.6211e+01,1.7065e-01,1.7065e-01,2.9300,2.4868e-10,1.000065,mixed,100.00,0.00,0.00,MIXED,GB on
4,2.7283e-06,1.8903e-14,6.4667e-15,1.2501e-12,6.4667e-15,6.4667e-15,3.3526e+01,6.6132e+01,1.7105e-01,1.7105e-01,2.9231,2.4868e-10,1.000065,mixed,99.99,0.00,0.01,MIXED,GB on
5,6.2355e-06,4.2975e-14,1.4779e-14,2.8343e-12,1.4779e-14,1.4779e-14,3.3703e+01,6.5953e+01,1.7195e-01,1.7195e-01,2.9078,2.4868e-10,1.000065,mixed,99.98,0.00,0.02,MIXED,GB on
6,1.4251e-05,9.7092e-14,3.3777e-14,6.3652e-12,3.3778e-14,3.3778e-14,3.4093e+01,6.5559e+01,1.7395e-01,1.7395e-01,2.8745,2.4868e-10,1.000065,mixed,99.95,0.00,0.05,MIXED,GB on
7,3.2570e-05,2.1662e-13,7.7196e-14,1.4020e-11,7.7197e-14,7.7197e-14,3.4924e+01,6.4720e+01,1.7819e-01,1.7819e-01,2.8061,2.4868e-10,1.000065,mixed,99.90,0.00,0.10,MIXED,GB on
8,7.4438e-05,4.7276e-13,1.7642e-13,2.9810e-11,1.7643e-13,1.7643e-13,3.6571e+01,6.3056e+01,1.8660e-01,1.8660e-01,2.6797,2.4868e-10,1.000065,mixed,99.77,0.00,0.23,MIXED,GB on
9,1.7013e-04,1.0002e-12,4.0318e-13,6.0101e-11,4.0322e-13,4.0322e-13,3.9505e+01,6.0092e+01,2.0158e-01,2.0158e-01,2.4807,2.4868e-10,1.000065,mixed,99.50,0.00,0.49,MIXED,GB on
